<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/06_LLM_Recommendation_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# NOTEBOOK 06.01 — INITIALIZATION AND RESEARCH SCOPE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06")
print("CANDIDATE STRATEGY EVALUATION")
print("=" * 100)

print("\nResearch objective:")
print(
    "Empirically evaluate candidate missing-value imputation "
    "strategies under controlled missingness scenarios and "
    "generate evidence for AIR-LLM adaptive strategy selection."
)

print("\nResearch scope:")
print("  • Multiple benchmark datasets")
print("  • Feature-level evaluation")
print("  • MCAR and MAR missingness")
print("  • Missingness rates configured in Notebook 00")
print("  • Repeated deterministic experiments")
print("  • Ground-truth recovery from observed values")
print("  • Numerical reconstruction metrics")
print("  • Categorical reconstruction metrics")
print("  • Distributional fidelity")
print("  • Dependency preservation")
print("  • Downstream predictive utility")
print("  • Runtime measurement")
print("  • LLM recommendation evaluation")
print("  • Statistical significance analysis")
print("  • Adaptive utility-based selection")

print("\nNotebook sequence:")
for cell, purpose in [
    ("06.01", "Notebook 06 initialization and research scope"),
    ("06.02", "Load Notebook 00 master configuration"),
    ("06.03", "Validate datasets, features, targets, and experiment configuration"),
    ("06.04", "Load processed experimental datasets"),
    ("06.05", "Build evaluation dataset registry"),
    ("06.06", "Detect/validate feature types"),
    ("06.07", "Build predictor-selection function"),
    ("06.08", "Build missingness-mask generation"),
    ("06.09", "Build predictor encoding pipeline"),
    ("06.10", "Define statistical candidate imputers"),
    ("06.11", "Define KNN candidate"),
    ("06.12", "Define iterative/MICE candidates"),
    ("06.13", "Define Random Forest candidate"),
    ("06.14", "Define Gradient Boosting candidate"),
    ("06.15", "Define MissForest candidate"),
    ("06.16", "Define SoftImpute / matrix-factorization candidates"),
    ("06.17", "Candidate strategy compatibility validation"),
    ("06.18", "Build LLM-recommended candidate evaluation plan"),
    ("06.19", "Validate evaluation-plan integrity and expected experiment count"),
    ("06.20", "Small-run preparation / candidate execution preflight"),
    ("06.21", "Corrected Candidate Evaluation Engine + small validation + full execution"),
    ("06.22", "Evaluation Status Summary"),
    ("06.23", "Candidate Performance Analysis"),
    ("06.24", "LLM Recommendation Evaluation"),
    ("06.25", "Statistical Significance"),
    ("06.26", "Adaptive Utility-Based Selection"),
]:
    print(f"  {cell}  {purpose}")

print("\nResearch principle:")
print(
    "Observed values are masked to create known ground truth; "
    "imputation is performed without access to the masked truth."
)

print("\n" + "=" * 100)
print("NOTEBOOK 06 INITIALIZATION : READY")
print("=" * 100)

AIR-LLM — NOTEBOOK 06
CANDIDATE STRATEGY EVALUATION

Research objective:
Empirically evaluate candidate missing-value imputation strategies under controlled missingness scenarios and generate evidence for AIR-LLM adaptive strategy selection.

Research scope:
  • Multiple benchmark datasets
  • Feature-level evaluation
  • MCAR and MAR missingness
  • Missingness rates configured in Notebook 00
  • Repeated deterministic experiments
  • Ground-truth recovery from observed values
  • Numerical reconstruction metrics
  • Categorical reconstruction metrics
  • Distributional fidelity
  • Dependency preservation
  • Downstream predictive utility
  • Runtime measurement
  • LLM recommendation evaluation
  • Statistical significance analysis
  • Adaptive utility-based selection

Notebook sequence:
  06.01  Notebook 06 initialization and research scope
  06.02  Load Notebook 00 master configuration
  06.03  Validate datasets, features, targets, and experiment configuration
  06.04  Load proces

In [2]:
# ============================================================
# NOTEBOOK 06.02 — MOUNT GOOGLE DRIVE + LOAD MASTER CONFIG
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.02")
print("MOUNT GOOGLE DRIVE + LOAD NOTEBOOK 00 MASTER CONFIGURATION")
print("=" * 100)


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import json
import yaml
import random
import warnings
import time
import inspect
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


print("\n[06.02.01] Imports completed.")


# ============================================================
# 2. MOUNT GOOGLE DRIVE
# ============================================================

try:

    from google.colab import drive

    drive.mount(
        "/content/drive",
        force_remount=False
    )

    print(
        "\n[06.02.02] Google Drive mounted successfully."
    )

except ImportError:

    raise RuntimeError(
        "This notebook is intended to run in Google Colab, "
        "but google.colab could not be imported."
    )


# ============================================================
# 3. VALIDATE DRIVE
# ============================================================

DRIVE_ROOT = Path(
    "/content/drive"
)

MYDRIVE_ROOT = (
    DRIVE_ROOT /
    "MyDrive"
)


if not DRIVE_ROOT.exists():

    raise FileNotFoundError(
        "Google Drive mount point was not found:\n"
        f"{DRIVE_ROOT}"
    )


if not MYDRIVE_ROOT.exists():

    raise FileNotFoundError(
        "Google MyDrive directory was not found:\n"
        f"{MYDRIVE_ROOT}"
    )


print(
    "\n[06.02.03] Google Drive filesystem validation: PASS"
)

print(
    f"Drive root : {DRIVE_ROOT}"
)

print(
    f"MyDrive    : {MYDRIVE_ROOT}"
)


# ============================================================
# 4. CANONICAL AIR-LLM PROJECT ROOT
# ============================================================

PROJECT_ROOT = (
    MYDRIVE_ROOT /
    "AIR_LLM_Research"
)


CONFIG_PATH = (
    PROJECT_ROOT /
    "config" /
    "config.yaml"
)


print(
    "\n[06.02.04] AIR-LLM project paths initialized."
)

print(
    f"PROJECT_ROOT : {PROJECT_ROOT}"
)

print(
    f"CONFIG_PATH  : {CONFIG_PATH}"
)


# ============================================================
# 5. VALIDATE PROJECT ROOT
# ============================================================

if not PROJECT_ROOT.exists():

    raise FileNotFoundError(
        "\nAIR-LLM project root not found:\n"
        f"{PROJECT_ROOT}\n\n"
        "Expected canonical project root:\n"
        "/content/drive/MyDrive/AIR_LLM_Research"
    )


if not PROJECT_ROOT.is_dir():

    raise NotADirectoryError(
        f"AIR-LLM project root is not a directory:\n"
        f"{PROJECT_ROOT}"
    )


print(
    "\n[06.02.05] AIR-LLM project root validation: PASS"
)


# ============================================================
# 6. VALIDATE CONFIGURATION DIRECTORY
# ============================================================

CONFIG_DIR = (
    PROJECT_ROOT /
    "config"
)


if not CONFIG_DIR.exists():

    raise FileNotFoundError(
        f"Configuration directory not found:\n"
        f"{CONFIG_DIR}"
    )


if not CONFIG_DIR.is_dir():

    raise NotADirectoryError(
        f"Configuration path is not a directory:\n"
        f"{CONFIG_DIR}"
    )


print(
    "[06.02.06] Configuration directory validation: PASS"
)


# ============================================================
# 7. VALIDATE MASTER CONFIGURATION FILE
# ============================================================

if not CONFIG_PATH.exists():

    raise FileNotFoundError(
        "\nNotebook 00 configuration not found:\n"
        f"{CONFIG_PATH}\n\n"
        "Run Notebook 00 first and ensure "
        "config/config.yaml was exported."
    )


if not CONFIG_PATH.is_file():

    raise FileNotFoundError(
        f"Configuration path is not a file:\n"
        f"{CONFIG_PATH}"
    )


print(
    "[06.02.07] config.yaml validation: PASS"
)


# ============================================================
# 8. LOAD NOTEBOOK 00 MASTER CONFIGURATION
# ============================================================

try:

    with open(
        CONFIG_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        CONFIG = yaml.safe_load(f)

except yaml.YAMLError as exc:

    raise RuntimeError(
        "Failed to parse config.yaml.\n"
        f"Error: {exc}"
    )


if not isinstance(
    CONFIG,
    dict
):

    raise TypeError(
        "config.yaml must contain a dictionary."
    )


if len(CONFIG) == 0:

    raise RuntimeError(
        "config.yaml was loaded but contains no "
        "configuration sections."
    )


print(
    "\n[06.02.08] Master configuration loaded."
)


# ============================================================
# 9. EXTRACT COMMON CONFIGURATION VALUES
# ============================================================
# These variables are created only when their corresponding
# sections exist. Existing Notebook 00 configuration remains
# the authoritative source.

if isinstance(
    CONFIG.get("datasets"),
    dict
):

    DATASET_CONFIG = CONFIG[
        "datasets"
    ]

elif isinstance(
    CONFIG.get("dataset_registry"),
    dict
):

    DATASET_CONFIG = CONFIG[
        "dataset_registry"
    ]

else:

    DATASET_CONFIG = {}


# ------------------------------------------------------------
# Dataset IDs
# ------------------------------------------------------------

if DATASET_CONFIG:

    DATASET_IDS = list(
        DATASET_CONFIG.keys()
    )

else:

    DATASET_IDS = []


# ------------------------------------------------------------
# Experiment configuration
# ------------------------------------------------------------

if isinstance(
    CONFIG.get("experiment"),
    dict
):

    EXPERIMENT_CONFIG = CONFIG[
        "experiment"
    ]

else:

    EXPERIMENT_CONFIG = {}


# ============================================================
# 10. CONFIGURATION SUMMARY
# ============================================================

print(
    "\n" + "-" * 100
)

print(
    "MASTER CONFIGURATION SUMMARY"
)

print(
    "-" * 100
)

print(
    f"Project root                  : {PROJECT_ROOT}"
)

print(
    f"Configuration file            : {CONFIG_PATH}"
)

print(
    f"Top-level configuration keys : {len(CONFIG)}"
)

print(
    f"Dataset IDs detected          : {len(DATASET_IDS)}"
)

if DATASET_IDS:

    print(
        "Datasets                      : "
        +
        ", ".join(
            str(x)
            for x in DATASET_IDS
        )
    )


# ============================================================
# 11. FINAL VALIDATION
# ============================================================

required_state = {

    "PROJECT_ROOT":
        PROJECT_ROOT,

    "CONFIG_PATH":
        CONFIG_PATH,

    "CONFIG":
        CONFIG,

}


missing_state = [

    name

    for name, value
    in required_state.items()

    if value is None

]


if missing_state:

    raise RuntimeError(
        "Required Notebook 06.02 state is missing:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_state
        )
    )


print(
    "\n" + "=" * 100
)

print(
    "NOTEBOOK 06.02 — MASTER CONFIGURATION LOAD: PASSED"
)

print(
    "=" * 100
)

AIR-LLM — NOTEBOOK 06.02
MOUNT GOOGLE DRIVE + LOAD NOTEBOOK 00 MASTER CONFIGURATION

[06.02.01] Imports completed.
Mounted at /content/drive

[06.02.02] Google Drive mounted successfully.

[06.02.03] Google Drive filesystem validation: PASS
Drive root : /content/drive
MyDrive    : /content/drive/MyDrive

[06.02.04] AIR-LLM project paths initialized.
PROJECT_ROOT : /content/drive/MyDrive/AIR_LLM_Research
CONFIG_PATH  : /content/drive/MyDrive/AIR_LLM_Research/config/config.yaml

[06.02.05] AIR-LLM project root validation: PASS
[06.02.06] Configuration directory validation: PASS
[06.02.07] config.yaml validation: PASS

[06.02.08] Master configuration loaded.

----------------------------------------------------------------------------------------------------
MASTER CONFIGURATION SUMMARY
----------------------------------------------------------------------------------------------------
Project root                  : /content/drive/MyDrive/AIR_LLM_Research
Configuration file            : 

In [3]:
# ============================================================
# NOTEBOOK 06.03 — DATASET / FEATURE / TARGET / EXPERIMENT
#                 VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.03")
print("DATASET / FEATURE / TARGET / EXPERIMENT VALIDATION")
print("=" * 100)


# ============================================================
# 1. IMPORTS AND REQUIRED STATE
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


if "CONFIG" not in globals():
    raise RuntimeError(
        "CONFIG is not available. Run Notebook 06.02 first."
    )

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(
        "/content/drive/MyDrive/AIR_LLM_Research"
    )


# ============================================================
# 2. CANONICAL DATASET REGISTRY
# ============================================================

EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]


# ============================================================
# 3. TARGET REGISTRY
# ============================================================

KNOWN_TARGETS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}


TARGET_REGISTRY = {}


# Try Notebook 00 configuration first
config_target_sources = [
    CONFIG.get("TARGET_REGISTRY"),
    CONFIG.get("target_registry"),
    CONFIG.get("targets"),
]


features_config = CONFIG.get("features", {})

if isinstance(features_config, dict):

    config_target_sources.extend([
        features_config.get("TARGET_REGISTRY"),
        features_config.get("target_registry"),
        features_config.get("targets"),
    ])


for source in config_target_sources:

    if not isinstance(source, dict):
        continue

    for dataset_id, value in source.items():

        if isinstance(value, str):

            TARGET_REGISTRY[
                str(dataset_id)
            ] = value

        elif isinstance(value, dict):

            for key in [
                "target",
                "target_column",
                "target_feature",
                "column",
            ]:

                if value.get(key) is not None:

                    TARGET_REGISTRY[
                        str(dataset_id)
                    ] = str(value[key])

                    break


# Fill any missing canonical targets
for dataset_id in EXPECTED_DATASETS:

    if dataset_id not in TARGET_REGISTRY:

        TARGET_REGISTRY[
            dataset_id
        ] = KNOWN_TARGETS[dataset_id]


# Validate target registry
missing_targets = [
    dataset_id
    for dataset_id in EXPECTED_DATASETS
    if dataset_id not in TARGET_REGISTRY
]


if missing_targets:

    raise RuntimeError(
        "Missing target definitions:\n"
        + "\n".join(
            f"  - {x}"
            for x in missing_targets
        )
    )


# ============================================================
# 4. PROCESSED DATASET FILE DISCOVERY
# ============================================================

PROCESSED_DIR = (
    PROJECT_ROOT /
    "data" /
    "processed"
)


if not PROCESSED_DIR.exists():

    raise FileNotFoundError(
        f"Processed-data directory not found:\n"
        f"{PROCESSED_DIR}"
    )


def find_processed_dataset(dataset_id):

    candidates = [
        PROCESSED_DIR / f"{dataset_id}_processed.csv",
        PROCESSED_DIR / f"{dataset_id}.csv",
        PROCESSED_DIR / dataset_id / "processed.csv",
    ]

    for path in candidates:

        if path.exists():
            return path

    matches = list(
        PROCESSED_DIR.rglob(
            f"{dataset_id}*.csv"
        )
    )

    if matches:
        return matches[0]

    return None


DATASET_PATH_REGISTRY = {}

missing_files = []


for dataset_id in EXPECTED_DATASETS:

    path = find_processed_dataset(
        dataset_id
    )

    if path is None:

        missing_files.append(
            dataset_id
        )

    else:

        DATASET_PATH_REGISTRY[
            dataset_id
        ] = path


if missing_files:

    raise FileNotFoundError(
        "Processed dataset files not found:\n"
        + "\n".join(
            f"  - {x}"
            for x in missing_files
        )
    )


# ============================================================
# 5. DATASET SCHEMA / TARGET VALIDATION
# ============================================================

validation_rows = []


for dataset_id in EXPECTED_DATASETS:

    path = DATASET_PATH_REGISTRY[
        dataset_id
    ]

    target = TARGET_REGISTRY[
        dataset_id
    ]

    # Read header only — full data loading occurs in 06.04
    header = pd.read_csv(
        path,
        nrows=0
    )

    columns = list(
        header.columns
    )

    target_present = (
        target in columns
    )

    if not target_present:

        raise RuntimeError(
            f"Target '{target}' not found in "
            f"'{dataset_id}'.\n"
            f"File: {path}"
        )

    validation_rows.append({

        "dataset_id":
            dataset_id,

        "columns":
            len(columns),

        "target":
            target,

        "target_present":
            True,

        "path":
            str(path),

    })


DATASET_VALIDATION_DF = pd.DataFrame(
    validation_rows
)


# ============================================================
# 6. MISSINGNESS EXPERIMENT CONFIGURATION
# ============================================================

missingness_config = CONFIG.get(
    "missingness",
    {}
)

if not isinstance(
    missingness_config,
    dict
):

    missingness_config = {}


# ------------------------------------------------------------
# Missingness mechanisms
# ------------------------------------------------------------

missingness_mechanisms = (
    missingness_config.get("mechanisms")
    or
    missingness_config.get("missingness_mechanisms")
    or
    CONFIG.get("missingness_mechanisms")
)


if missingness_mechanisms is None:

    # Use the mechanisms defined for the AIR-LLM experiment
    missingness_mechanisms = [
        "MCAR",
        "MAR",
    ]


missingness_mechanisms = [
    str(x).upper()
    for x in missingness_mechanisms
]


valid_mechanisms = {
    "MCAR",
    "MAR",
    "MNAR",
}


invalid_mechanisms = (
    set(missingness_mechanisms)
    -
    valid_mechanisms
)


if invalid_mechanisms:

    raise RuntimeError(
        "Invalid missingness mechanisms:\n"
        + "\n".join(
            f"  - {x}"
            for x in sorted(
                invalid_mechanisms
            )
        )
    )


# ------------------------------------------------------------
# Missingness rates
# ------------------------------------------------------------

missingness_rates = (
    missingness_config.get("rates")
    or
    missingness_config.get("missingness_rates")
    or
    CONFIG.get("missingness_rates")
)


if missingness_rates is None:

    missingness_rates = [
        0.10,
        0.20,
        0.30,
        0.40,
        0.50,
    ]


missingness_rates = [
    float(x)
    for x in missingness_rates
]


if not all(
    0 < x < 1
    for x in missingness_rates
):

    raise RuntimeError(
        "Missingness rates must be between 0 and 1:\n"
        f"{missingness_rates}"
    )


# ------------------------------------------------------------
# Repetitions
# ------------------------------------------------------------

repetitions = (
    missingness_config.get("repetitions")
    or
    CONFIG.get("repetitions")
    or
    5
)

repetitions = int(
    repetitions
)


if repetitions < 1:

    raise RuntimeError(
        "Repetitions must be >= 1."
    )


# ------------------------------------------------------------
# Master seed
# ------------------------------------------------------------

MASTER_SEED = int(
    CONFIG.get("master_seed")
    or
    CONFIG.get("seed")
    or
    42
)


# ============================================================
# 7. EXPORT VALIDATED EXPERIMENT STATE
# ============================================================

EXPERIMENT_CONFIGURATION = {

    "datasets":
        EXPECTED_DATASETS,

    "targets":
        TARGET_REGISTRY,

    "missingness_mechanisms":
        missingness_mechanisms,

    "missingness_rates":
        missingness_rates,

    "repetitions":
        repetitions,

    "master_seed":
        MASTER_SEED,

}


# ============================================================
# 8. DISPLAY VALIDATION
# ============================================================

print(
    "\n[06.03.01] Dataset / target configuration: PASS"
)

print(
    "[06.03.02] Processed dataset discovery: PASS"
)

print(
    "[06.03.03] Dataset schema / target validation: PASS"
)


display(
    DATASET_VALIDATION_DF
)


print(
    "\n" + "-" * 100
)

print("EXPERIMENT CONFIGURATION")

print("-" * 100)

print(
    f"Datasets                : "
    f"{EXPECTED_DATASETS}"
)

print(
    f"Targets                 : "
    f"{TARGET_REGISTRY}"
)

print(
    f"Missingness mechanisms  : "
    f"{missingness_mechanisms}"
)

print(
    f"Missingness rates       : "
    f"{missingness_rates}"
)

print(
    f"Repetitions             : "
    f"{repetitions}"
)

print(
    f"Master seed             : "
    f"{MASTER_SEED}"
)


# ============================================================
# 9. FINAL VALIDATION
# ============================================================

assert len(
    DATASET_PATH_REGISTRY
) == len(
    EXPECTED_DATASETS
)

assert len(
    TARGET_REGISTRY
) == len(
    EXPECTED_DATASETS
)

assert all(
    0 < rate < 1
    for rate in missingness_rates
)

assert repetitions >= 1


print(
    "\n" + "=" * 100
)

print(
    "DATASET / FEATURE / TARGET / EXPERIMENT VALIDATION : PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.03
DATASET / FEATURE / TARGET / EXPERIMENT VALIDATION

[06.03.01] Dataset / target configuration: PASS
[06.03.02] Processed dataset discovery: PASS
[06.03.03] Dataset schema / target validation: PASS


,dataset_id,columns,target,target_present,path
0,adult_income,15,income,True,/content/drive/MyDrive/AIR_LLM_Research/data/p...
1,bank_marketing,17,y,True,/content/drive/MyDrive/AIR_LLM_Research/data/p...
2,diabetes_130us,48,readmitted,True,/content/drive/MyDrive/AIR_LLM_Research/data/p...



----------------------------------------------------------------------------------------------------
EXPERIMENT CONFIGURATION
----------------------------------------------------------------------------------------------------
Datasets                : ['adult_income', 'bank_marketing', 'diabetes_130us']
Targets                 : {'adult_income': 'income', 'bank_marketing': 'y', 'diabetes_130us': 'readmitted'}
Missingness mechanisms  : ['MCAR', 'MAR', 'MNAR']
Missingness rates       : [0.1, 0.2, 0.3, 0.4, 0.5]
Repetitions             : 5
Master seed             : 42

DATASET / FEATURE / TARGET / EXPERIMENT VALIDATION : PASSED


In [4]:
# ============================================================
# NOTEBOOK 06.04 — LOAD PROCESSED EXPERIMENTAL DATASETS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.04")
print("LOAD PROCESSED EXPERIMENTAL DATASETS")
print("=" * 100)


PROCESSED_DIR = (
    PROJECT_ROOT /
    "data" /
    "processed"
)


if not PROCESSED_DIR.exists():
    raise FileNotFoundError(
        f"Processed-data directory not found:\n{PROCESSED_DIR}"
    )


def find_processed_dataset(dataset_id):

    candidates = [
        PROCESSED_DIR / f"{dataset_id}_processed.csv",
        PROCESSED_DIR / f"{dataset_id}.csv",
        PROCESSED_DIR / dataset_id / "processed.csv",
    ]

    for path in candidates:
        if path.exists():
            return path

    recursive_matches = list(
        PROCESSED_DIR.rglob(
            f"{dataset_id}*.csv"
        )
    )

    if recursive_matches:
        return recursive_matches[0]

    return None


EVALUATION_DATA = {}
DATASET_PATH_REGISTRY = {}

for dataset_id in DATASET_IDS:

    path = find_processed_dataset(
        dataset_id
    )

    if path is None:
        raise FileNotFoundError(
            f"No processed dataset found for '{dataset_id}'."
        )

    df = pd.read_csv(path)

    if df.empty:
        raise RuntimeError(
            f"Processed dataset '{dataset_id}' is empty."
        )

    EVALUATION_DATA[dataset_id] = df
    DATASET_PATH_REGISTRY[dataset_id] = path

    print(
        f"{dataset_id:<20} "
        f"rows={len(df):>8,} "
        f"columns={df.shape[1]:>4} "
        f"path={path}"
    )


print(
    f"\nDatasets loaded: {len(EVALUATION_DATA)}"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.04
LOAD PROCESSED EXPERIMENTAL DATASETS
adult_income         rows=  32,561 columns=  15 path=/content/drive/MyDrive/AIR_LLM_Research/data/processed/adult_income_processed.csv
bank_marketing       rows=  45,211 columns=  17 path=/content/drive/MyDrive/AIR_LLM_Research/data/processed/bank_marketing_processed.csv
diabetes_130us       rows= 101,766 columns=  48 path=/content/drive/MyDrive/AIR_LLM_Research/data/processed/diabetes_130us_processed.csv

Datasets loaded: 3


In [5]:
# ============================================================
# NOTEBOOK 06.05 — DATASET INTEGRITY / DATA-TYPE VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.05")
print("DATASET INTEGRITY / DATA-TYPE VALIDATION")
print("=" * 100)


# ============================================================
# 1. REQUIRED STATE
# ============================================================

required_objects = [
    "CONFIG",
    "PROJECT_ROOT",
    "EVALUATION_DATA",
    "DATASET_PATH_REGISTRY",
    "EXPECTED_DATASETS",
    "TARGET_REGISTRY",
]


missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:

    raise RuntimeError(
        "Required Notebook 06 state is missing:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_objects
        )
        +
        "\n\n"
        "Run Notebook 06.02 → 06.03 → 06.04 first."
    )


if not isinstance(
    EVALUATION_DATA,
    dict
):

    raise RuntimeError(
        "EVALUATION_DATA must be a dictionary."
    )


if not EVALUATION_DATA:

    raise RuntimeError(
        "EVALUATION_DATA is empty."
    )


# ============================================================
# 2. DATASET PRESENCE VALIDATION
# ============================================================

missing_datasets = [
    dataset_id
    for dataset_id in EXPECTED_DATASETS
    if dataset_id not in EVALUATION_DATA
]


if missing_datasets:

    raise RuntimeError(
        "Required datasets are missing from EVALUATION_DATA:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_datasets
        )
    )


print(
    "\n[06.05.01] Dataset presence validation: PASS"
)


# ============================================================
# 3. DATASET INTEGRITY VALIDATION
# ============================================================

DATASET_INTEGRITY_ROWS = []


for dataset_id in EXPECTED_DATASETS:

    df = EVALUATION_DATA[
        dataset_id
    ]

    # --------------------------------------------------------
    # Basic DataFrame validation
    # --------------------------------------------------------

    if not isinstance(
        df,
        pd.DataFrame
    ):

        raise RuntimeError(
            f"{dataset_id} is not a pandas DataFrame."
        )


    if df.empty:

        raise RuntimeError(
            f"{dataset_id} is empty."
        )


    # --------------------------------------------------------
    # Column validation
    # --------------------------------------------------------

    if df.shape[1] == 0:

        raise RuntimeError(
            f"{dataset_id} contains no columns."
        )


    if df.columns.duplicated().any():

        duplicated_columns = (
            df.columns[
                df.columns.duplicated()
            ]
            .tolist()
        )

        raise RuntimeError(
            f"{dataset_id} contains duplicate columns:\n"
            +
            "\n".join(
                f"  - {x}"
                for x
                in duplicated_columns
            )
        )


    # --------------------------------------------------------
    # Target validation
    # --------------------------------------------------------

    target = TARGET_REGISTRY[
        dataset_id
    ]


    if target not in df.columns:

        raise RuntimeError(
            f"Target '{target}' not found in "
            f"{dataset_id}."
        )


    # --------------------------------------------------------
    # Missing-value summary
    # --------------------------------------------------------

    missing_cells = int(
        df.isna()
        .sum()
        .sum()
    )


    missing_columns = int(
        df.isna()
        .any()
        .sum()
    )


    # --------------------------------------------------------
    # Duplicate-row count
    # --------------------------------------------------------

    duplicate_rows = int(
        df.duplicated()
        .sum()
    )


    DATASET_INTEGRITY_ROWS.append({

        "dataset_id":
            dataset_id,

        "rows":
            int(
                len(df)
            ),

        "columns":
            int(
                df.shape[1]
            ),

        "target":
            target,

        "target_present":
            True,

        "missing_cells":
            missing_cells,

        "missing_columns":
            missing_columns,

        "duplicate_rows":
            duplicate_rows,

    })


DATASET_INTEGRITY_DF = pd.DataFrame(
    DATASET_INTEGRITY_ROWS
)


# ============================================================
# 4. DATA-TYPE VALIDATION
# ============================================================

DATA_TYPE_ROWS = []


for dataset_id in EXPECTED_DATASETS:

    df = EVALUATION_DATA[
        dataset_id
    ]

    target = TARGET_REGISTRY[
        dataset_id
    ]


    numerical_columns = (
        df.select_dtypes(
            include=np.number
        )
        .columns
        .tolist()
    )


    categorical_columns = (
        df.select_dtypes(
            exclude=np.number
        )
        .columns
        .tolist()
    )


    # --------------------------------------------------------
    # Target type
    # --------------------------------------------------------

    target_dtype = str(
        df[target].dtype
    )


    target_is_numeric = (
        pd.api.types.is_numeric_dtype(
            df[target]
        )
    )


    DATA_TYPE_ROWS.append({

        "dataset_id":
            dataset_id,

        "numerical_features":
            len(
                numerical_columns
            ),

        "categorical_features":
            len(
                categorical_columns
            ),

        "target":
            target,

        "target_dtype":
            target_dtype,

        "target_is_numeric":
            target_is_numeric,

        "total_features":
            len(
                df.columns
            ),

    })


DATA_TYPE_VALIDATION_DF = pd.DataFrame(
    DATA_TYPE_ROWS
)


print(
    "[06.05.02] Dataset integrity validation: PASS"
)

print(
    "[06.05.03] Data-type validation: PASS"
)


# ============================================================
# 5. FEATURE-TYPE REGISTRY
# ============================================================

FEATURE_TYPE_ROWS = []


for dataset_id in EXPECTED_DATASETS:

    df = EVALUATION_DATA[
        dataset_id
    ]

    target = TARGET_REGISTRY[
        dataset_id
    ]


    for feature in df.columns:

        # Target is excluded from imputation-feature analysis
        if feature == target:
            continue


        if pd.api.types.is_numeric_dtype(
            df[feature]
        ):

            feature_type = "numerical"

        else:

            feature_type = "categorical"


        FEATURE_TYPE_ROWS.append({

            "dataset_id":
                dataset_id,

            "feature":
                feature,

            "feature_type":
                feature_type,

            "dtype":
                str(
                    df[feature].dtype
                ),

            "missing_count":
                int(
                    df[feature]
                    .isna()
                    .sum()
                ),

            "missing_rate":
                float(
                    df[feature]
                    .isna()
                    .mean()
                ),

            "unique_values":
                int(
                    df[feature]
                    .nunique(
                        dropna=True
                    )
                ),

        })


FEATURE_TYPE_DF = pd.DataFrame(
    FEATURE_TYPE_ROWS
)


# Compatibility aliases for later notebooks
FEATURE_TYPE_VALIDATION_DF = (
    FEATURE_TYPE_DF.copy()
)


FEATURE_PROFILE_DF = (
    FEATURE_TYPE_DF.copy()
)


# ============================================================
# 6. DATA QUALITY VALIDATION
# ============================================================

invalid_feature_types = (
    FEATURE_TYPE_DF[
        ~FEATURE_TYPE_DF[
            "feature_type"
        ].isin(
            [
                "numerical",
                "categorical",
            ]
        )
    ]
)


if not invalid_feature_types.empty:

    raise RuntimeError(
        "Unsupported feature types detected:\n"
        +
        invalid_feature_types[
            [
                "dataset_id",
                "feature",
                "feature_type",
            ]
        ]
        .to_string(
            index=False
        )
    )


# Check for columns containing only missing values
all_missing_features = []


for dataset_id in EXPECTED_DATASETS:

    df = EVALUATION_DATA[
        dataset_id
    ]

    target = TARGET_REGISTRY[
        dataset_id
    ]


    for feature in df.columns:

        if feature == target:
            continue


        if df[feature].isna().all():

            all_missing_features.append(
                f"{dataset_id} -> {feature}"
            )


if all_missing_features:

    raise RuntimeError(
        "Features containing 100% missing values detected:\n"
        +
        "\n".join(
            f"  - {x}"
            for x
            in all_missing_features
        )
    )


print(
    "[06.05.04] Feature data-quality validation: PASS"
)


# ============================================================
# 7. FINAL DATASET VALIDATION TABLE
# ============================================================

display(
    DATASET_INTEGRITY_DF
)


print(
    "\nDATA-TYPE SUMMARY"
)

display(
    DATA_TYPE_VALIDATION_DF
)


print(
    "\nFEATURE-TYPE SUMMARY"
)

display(
    FEATURE_TYPE_DF.head(20)
)


# ============================================================
# 8. FINAL VALIDATION
# ============================================================

assert (
    len(EVALUATION_DATA)
    ==
    len(EXPECTED_DATASETS)
)


assert not DATASET_INTEGRITY_DF.empty


assert not DATA_TYPE_VALIDATION_DF.empty


assert not FEATURE_TYPE_DF.empty


assert set(
    FEATURE_TYPE_DF[
        "feature_type"
    ].unique()
).issubset(
    {
        "numerical",
        "categorical",
    }
)


# ============================================================
# 9. FINAL REPORT
# ============================================================

print(
    "\n" + "=" * 100
)

print(
    "DATASET INTEGRITY SUMMARY"
)

print(
    "-" * 100
)

for dataset_id in EXPECTED_DATASETS:

    row = DATASET_INTEGRITY_DF[
        DATASET_INTEGRITY_DF[
            "dataset_id"
        ]
        ==
        dataset_id
    ].iloc[0]


    print(
        f"{dataset_id:<20} "
        f"rows={row['rows']:>8,}  "
        f"columns={row['columns']:>3}  "
        f"missing_cells={row['missing_cells']:>8,}  "
        f"duplicates={row['duplicate_rows']:>6,}"
    )


print(
    "\nFeature-type records : "
    f"{len(FEATURE_TYPE_DF):,}"
)


print(
    "Numerical features    : "
    f"{sum(FEATURE_TYPE_DF['feature_type'] == 'numerical'):,}"
)


print(
    "Categorical features  : "
    f"{sum(FEATURE_TYPE_DF['feature_type'] == 'categorical'):,}"
)


print(
    "\n" + "=" * 100
)

print(
    "DATASET INTEGRITY / DATA-TYPE VALIDATION : PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.05
DATASET INTEGRITY / DATA-TYPE VALIDATION

[06.05.01] Dataset presence validation: PASS
[06.05.02] Dataset integrity validation: PASS
[06.05.03] Data-type validation: PASS
[06.05.04] Feature data-quality validation: PASS


,dataset_id,rows,columns,target,target_present,missing_cells,missing_columns,duplicate_rows
0,adult_income,32561,15,income,True,4262,3,24
1,bank_marketing,45211,17,y,True,0,0,0
2,diabetes_130us,101766,48,readmitted,True,374017,9,0



DATA-TYPE SUMMARY


,dataset_id,numerical_features,categorical_features,target,target_dtype,target_is_numeric,total_features
0,adult_income,6,9,income,object,False,15
1,bank_marketing,7,10,y,object,False,17
2,diabetes_130us,11,37,readmitted,object,False,48



FEATURE-TYPE SUMMARY


,dataset_id,feature,feature_type,dtype,missing_count,missing_rate,unique_values
0,adult_income,age,numerical,int64,0,0.000000,73
1,adult_income,workclass,categorical,object,1836,0.056386,8
2,adult_income,fnlwgt,numerical,int64,0,0.000000,21648
3,adult_income,education,categorical,object,0,0.000000,16
4,adult_income,education_num,numerical,int64,0,0.000000,16
5,adult_income,marital_status,categorical,object,0,0.000000,7
6,adult_income,occupation,categorical,object,1843,0.056601,14
7,adult_income,relationship,categorical,object,0,0.000000,6
8,adult_income,race,categorical,object,0,0.000000,5
9,adult_income,sex,categorical,object,0,0.000000,2



DATASET INTEGRITY SUMMARY
----------------------------------------------------------------------------------------------------
adult_income         rows=  32,561  columns= 15  missing_cells=   4,262  duplicates=    24
bank_marketing       rows=  45,211  columns= 17  missing_cells=       0  duplicates=     0
diabetes_130us       rows= 101,766  columns= 48  missing_cells= 374,017  duplicates=     0

Feature-type records : 77
Numerical features    : 24
Categorical features  : 53

DATASET INTEGRITY / DATA-TYPE VALIDATION : PASSED


In [6]:
# ============================================================
# NOTEBOOK 06.06 — FEATURE CHARACTERIZATION / VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.06")
print("FEATURE CHARACTERIZATION / VALIDATION")
print("=" * 100)


# ============================================================
# 1. REQUIRED STATE
# ============================================================

required_objects = [
    "EVALUATION_DATA",
    "TARGET_REGISTRY",
    "EXPECTED_DATASETS",
]


missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Required state is missing:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_objects
        )
        +
        "\n\n"
        "Run Notebook 06.02 → 06.03 → 06.04 → 06.05 first."
    )


# ============================================================
# 2. FEATURE TYPE DETECTION
# ============================================================

def detect_feature_type(series):

    # Boolean variables are categorical
    if pd.api.types.is_bool_dtype(series):
        return "categorical"

    # Numeric variables
    if pd.api.types.is_numeric_dtype(series):
        return "numerical"

    # Object, string, category, datetime-like and
    # other non-numeric variables are treated as categorical
    return "categorical"


# ============================================================
# 3. BUILD FEATURE CHARACTERIZATION REGISTRY
# ============================================================

FEATURE_TYPE_REGISTRY = {}

feature_rows = []


for dataset_id in EXPECTED_DATASETS:

    if dataset_id not in EVALUATION_DATA:
        raise RuntimeError(
            f"Dataset '{dataset_id}' is missing from EVALUATION_DATA."
        )


    df = EVALUATION_DATA[
        dataset_id
    ]


    target = TARGET_REGISTRY.get(
        dataset_id
    )


    if target is None:
        raise RuntimeError(
            f"No target defined for dataset '{dataset_id}'."
        )


    if target not in df.columns:
        raise RuntimeError(
            f"Target '{target}' not found in dataset '{dataset_id}'."
        )


    FEATURE_TYPE_REGISTRY[
        dataset_id
    ] = {}


    for feature in df.columns:

        series = df[feature]

        feature_type = detect_feature_type(
            series
        )

        FEATURE_TYPE_REGISTRY[
            dataset_id
        ][feature] = feature_type


        # ----------------------------------------------------
        # Basic statistics
        # ----------------------------------------------------

        n_rows = len(df)

        missing_count = int(
            series.isna().sum()
        )

        missing_rate = (
            missing_count / n_rows
            if n_rows > 0
            else 0.0
        )

        observed_count = int(
            series.notna().sum()
        )

        unique_values = int(
            series.nunique(
                dropna=True
            )
        )


        # ----------------------------------------------------
        # Cardinality
        # ----------------------------------------------------

        cardinality_ratio = (
            unique_values / observed_count
            if observed_count > 0
            else 0.0
        )


        # ----------------------------------------------------
        # Constant / all-missing detection
        # ----------------------------------------------------

        all_missing = (
            observed_count == 0
        )

        constant_feature = (
            observed_count > 0
            and unique_values <= 1
        )


        # ----------------------------------------------------
        # Target status
        # ----------------------------------------------------

        is_target = (
            feature == target
        )


        # ----------------------------------------------------
        # Dtype
        # ----------------------------------------------------

        dtype = str(
            series.dtype
        )


        # ----------------------------------------------------
        # Numerical statistics
        # ----------------------------------------------------

        if feature_type == "numerical":

            numeric_series = pd.to_numeric(
                series,
                errors="coerce"
            )

            finite_values = numeric_series[
                np.isfinite(
                    numeric_series
                )
            ]

            if len(finite_values) > 0:

                minimum = float(
                    finite_values.min()
                )

                maximum = float(
                    finite_values.max()
                )

                mean_value = float(
                    finite_values.mean()
                )

                std_value = float(
                    finite_values.std(
                        ddof=0
                    )
                )

            else:

                minimum = np.nan
                maximum = np.nan
                mean_value = np.nan
                std_value = np.nan

        else:

            minimum = np.nan
            maximum = np.nan
            mean_value = np.nan
            std_value = np.nan


        # ----------------------------------------------------
        # Characterization record
        # ----------------------------------------------------

        feature_rows.append({

            "dataset_id":
                dataset_id,

            "feature":
                feature,

            "feature_type":
                feature_type,

            "dtype":
                dtype,

            "is_target":
                is_target,

            "rows":
                int(n_rows),

            "observed_count":
                observed_count,

            "missing_count":
                missing_count,

            "missing_rate":
                float(missing_rate),

            "unique_values":
                unique_values,

            "cardinality_ratio":
                float(cardinality_ratio),

            "constant_feature":
                bool(constant_feature),

            "all_missing":
                bool(all_missing),

            "minimum":
                minimum,

            "maximum":
                maximum,

            "mean":
                mean_value,

            "std":
                std_value,

        })


# ============================================================
# 4. CREATE AUTHORITATIVE FEATURE DATAFRAME
# ============================================================

FEATURE_CHARACTERIZATION_DF = pd.DataFrame(
    feature_rows
)


if FEATURE_CHARACTERIZATION_DF.empty:

    raise RuntimeError(
        "Feature characterization dataframe is empty."
    )


# ============================================================
# 5. AUTHORITATIVE FEATURE-TYPE DATAFRAME
# ============================================================

FEATURE_TYPE_DF = (
    FEATURE_CHARACTERIZATION_DF[
        [
            "dataset_id",
            "feature",
            "feature_type",
            "dtype",
            "is_target",
            "unique_values",
            "missing_count",
            "missing_rate",
        ]
    ]
    .copy()
)


# Compatibility aliases for downstream notebooks
FEATURE_TYPE_VALIDATION_DF = (
    FEATURE_TYPE_DF.copy()
)

FEATURE_PROFILE_DF = (
    FEATURE_CHARACTERIZATION_DF.copy()
)


# ============================================================
# 6. VALIDATE FEATURE TYPES
# ============================================================

valid_feature_types = {
    "numerical",
    "categorical",
}


invalid_feature_types = (
    set(
        FEATURE_TYPE_DF[
            "feature_type"
        ].unique()
    )
    -
    valid_feature_types
)


if invalid_feature_types:

    raise RuntimeError(
        "Unsupported feature types detected:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in sorted(
                invalid_feature_types
            )
        )
    )


# ============================================================
# 7. VALIDATE ALL-MISSING FEATURES
# ============================================================

all_missing_features = (
    FEATURE_CHARACTERIZATION_DF[
        FEATURE_CHARACTERIZATION_DF[
            "all_missing"
        ]
    ]
)


if not all_missing_features.empty:

    print(
        "\nWARNING: All-missing features detected:"
    )

    display(
        all_missing_features[
            [
                "dataset_id",
                "feature",
                "feature_type",
            ]
        ]
    )


# ============================================================
# 8. CONSTANT FEATURE REPORT
# ============================================================

constant_features = (
    FEATURE_CHARACTERIZATION_DF[
        FEATURE_CHARACTERIZATION_DF[
            "constant_feature"
        ]
    ]
)


if not constant_features.empty:

    print(
        "\nConstant features detected:"
    )

    display(
        constant_features[
            [
                "dataset_id",
                "feature",
                "feature_type",
                "unique_values",
            ]
        ]
    )


# ============================================================
# 9. TARGET VALIDATION
# ============================================================

target_validation_rows = []


for dataset_id in EXPECTED_DATASETS:

    target = TARGET_REGISTRY[
        dataset_id
    ]

    target_rows = (
        FEATURE_CHARACTERIZATION_DF[
            (
                FEATURE_CHARACTERIZATION_DF[
                    "dataset_id"
                ]
                ==
                dataset_id
            )
            &
            (
                FEATURE_CHARACTERIZATION_DF[
                    "feature"
                ]
                ==
                target
            )
        ]
    )


    if len(target_rows) != 1:

        raise RuntimeError(
            f"Target validation failed for "
            f"{dataset_id}: {target}"
        )


    target_row = target_rows.iloc[0]


    target_validation_rows.append({

        "dataset_id":
            dataset_id,

        "target":
            target,

        "feature_type":
            target_row[
                "feature_type"
            ],

        "dtype":
            target_row[
                "dtype"
            ],

        "unique_values":
            target_row[
                "unique_values"
            ],

        "missing_count":
            target_row[
                "missing_count"
            ],

        "valid":
            True,

    })


TARGET_VALIDATION_DF = pd.DataFrame(
    target_validation_rows
)


# ============================================================
# 10. DATASET-LEVEL FEATURE SUMMARY
# ============================================================

DATASET_FEATURE_SUMMARY_ROWS = []


for dataset_id in EXPECTED_DATASETS:

    dataset_features = (
        FEATURE_CHARACTERIZATION_DF[
            (
                FEATURE_CHARACTERIZATION_DF[
                    "dataset_id"
                ]
                ==
                dataset_id
            )
        ]
    )


    non_target_features = (
        dataset_features[
            ~dataset_features[
                "is_target"
            ]
        ]
    )


    numerical_count = int(
        (
            non_target_features[
                "feature_type"
            ]
            ==
            "numerical"
        ).sum()
    )


    categorical_count = int(
        (
            non_target_features[
                "feature_type"
            ]
            ==
            "categorical"
        ).sum()
    )


    DATASET_FEATURE_SUMMARY_ROWS.append({

        "dataset_id":
            dataset_id,

        "total_columns":
            int(
                len(dataset_features)
            ),

        "target":
            TARGET_REGISTRY[
                dataset_id
            ],

        "features_for_imputation":
            int(
                len(non_target_features)
            ),

        "numerical_features":
            numerical_count,

        "categorical_features":
            categorical_count,

        "features_with_missing":
            int(
                (
                    non_target_features[
                        "missing_count"
                    ]
                    >
                    0
                ).sum()
            ),

        "constant_features":
            int(
                non_target_features[
                    "constant_feature"
                ].sum()
            ),

    })


DATASET_FEATURE_SUMMARY_DF = pd.DataFrame(
    DATASET_FEATURE_SUMMARY_ROWS
)


# ============================================================
# 11. DISPLAY RESULTS
# ============================================================

print(
    "\n[06.06.01] Feature characterization completed."
)

print(
    f"Total feature records : "
    f"{len(FEATURE_CHARACTERIZATION_DF):,}"
)


print(
    "\nDATASET FEATURE SUMMARY"
)

display(
    DATASET_FEATURE_SUMMARY_DF
)


print(
    "\nTARGET VALIDATION"
)

display(
    TARGET_VALIDATION_DF
)


print(
    "\nFEATURE CHARACTERIZATION"
)

display(
    FEATURE_CHARACTERIZATION_DF
)


# ============================================================
# 12. FINAL VALIDATION
# ============================================================

if len(
    FEATURE_CHARACTERIZATION_DF
) != sum(
    len(
        EVALUATION_DATA[
            dataset_id
        ].columns
    )
    for dataset_id
    in EXPECTED_DATASETS
):

    raise RuntimeError(
        "Feature characterization record count does not "
        "match the total number of dataset columns."
    )


if not set(
    FEATURE_TYPE_DF[
        "feature_type"
    ].unique()
).issubset(
    valid_feature_types
):

    raise RuntimeError(
        "Final feature-type validation failed."
    )


if not all(
    TARGET_VALIDATION_DF[
        "valid"
    ]
):

    raise RuntimeError(
        "One or more target validations failed."
    )


# ============================================================
# 13. FINAL REPORT
# ============================================================

print(
    "\n" + "=" * 100
)

print(
    "FEATURE CHARACTERIZATION SUMMARY"
)

print(
    "-" * 100
)

for dataset_id in EXPECTED_DATASETS:

    row = DATASET_FEATURE_SUMMARY_DF[
        DATASET_FEATURE_SUMMARY_DF[
            "dataset_id"
        ]
        ==
        dataset_id
    ].iloc[0]


    print(
        f"{dataset_id:<20} "
        f"imputation_features="
        f"{row['features_for_imputation']:>3}  "
        f"numerical="
        f"{row['numerical_features']:>3}  "
        f"categorical="
        f"{row['categorical_features']:>3}  "
        f"missing_features="
        f"{row['features_with_missing']:>3}"
    )


print(
    "\nFeature-type registry datasets : "
    f"{len(FEATURE_TYPE_REGISTRY)}"
)

print(
    "Feature characterization records: "
    f"{len(FEATURE_CHARACTERIZATION_DF):,}"
)

print(
    "Numerical feature records       : "
    f"{(
        FEATURE_CHARACTERIZATION_DF['feature_type']
        == 'numerical'
    ).sum():,}"
)

print(
    "Categorical feature records     : "
    f"{(
        FEATURE_CHARACTERIZATION_DF['feature_type']
        == 'categorical'
    ).sum():,}"
)


print(
    "\n" + "=" * 100
)

print(
    "FEATURE CHARACTERIZATION / VALIDATION : PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.06
FEATURE CHARACTERIZATION / VALIDATION

Constant features detected:


,dataset_id,feature,feature_type,unique_values
69,diabetes_130us,examide,categorical,1
70,diabetes_130us,citoglipton,categorical,1



[06.06.01] Feature characterization completed.
Total feature records : 80

DATASET FEATURE SUMMARY


,dataset_id,total_columns,target,features_for_imputation,numerical_features,categorical_features,features_with_missing,constant_features
0,adult_income,15,income,14,6,8,3,0
1,bank_marketing,17,y,16,7,9,0,0
2,diabetes_130us,48,readmitted,47,11,36,9,2



TARGET VALIDATION


,dataset_id,target,feature_type,dtype,unique_values,missing_count,valid
0,adult_income,income,categorical,object,2,0,True
1,bank_marketing,y,categorical,object,2,0,True
2,diabetes_130us,readmitted,categorical,object,3,0,True



FEATURE CHARACTERIZATION


,dataset_id,feature,feature_type,dtype,is_target,rows,observed_count,missing_count,missing_rate,unique_values,cardinality_ratio,constant_feature,all_missing,minimum,maximum,mean,std
0,adult_income,age,numerical,int64,False,32561,32561,0,0.000000,73,0.002242,False,False,17.0,90.0,38.581647,13.640223
1,adult_income,workclass,categorical,object,False,32561,30725,1836,0.056386,8,0.000260,False,False,NaN,NaN,NaN,NaN
2,adult_income,fnlwgt,numerical,int64,False,32561,32561,0,0.000000,21648,0.664844,False,False,12285.0,1484705.0,189778.366512,105548.356881
3,adult_income,education,categorical,object,False,32561,32561,0,0.000000,16,0.000491,False,False,NaN,NaN,NaN,NaN
4,adult_income,education_num,numerical,int64,False,32561,32561,0,0.000000,16,0.000491,False,False,1.0,16.0,10.080679,2.572681
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,diabetes_130us,metformin-rosiglitazone,categorical,object,False,101766,101766,0,0.000000,2,0.000020,False,False,NaN,NaN,NaN,NaN
76,diabetes_130us,metformin-pioglitazone,categorical,object,False,101766,101766,0,0.000000,2,0.000020,False,False,NaN,NaN,NaN,NaN
77,diabetes_130us,change,categorical,object,False,101766,101766,0,0.000000,2,0.000020,False,False,NaN,NaN,NaN,NaN
78,diabetes_130us,diabetesMed,categorical,object,False,101766,101766,0,0.000000,2,0.000020,False,False,NaN,NaN,NaN,NaN



FEATURE CHARACTERIZATION SUMMARY
----------------------------------------------------------------------------------------------------
adult_income         imputation_features= 14  numerical=  6  categorical=  8  missing_features=  3
bank_marketing       imputation_features= 16  numerical=  7  categorical=  9  missing_features=  0
diabetes_130us       imputation_features= 47  numerical= 11  categorical= 36  missing_features=  9

Feature-type registry datasets : 3
Feature characterization records: 80
Numerical feature records       : 24
Categorical feature records     : 56

FEATURE CHARACTERIZATION / VALIDATION : PASSED


In [7]:
# ============================================================
# NOTEBOOK 06.07 — PREDICTOR SELECTION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.07")
print("LEAKAGE-SAFE / DATA-AWARE PREDICTOR SELECTION")
print("=" * 100)


# ============================================================
# 1. REQUIRED STATE
# ============================================================

required_objects = [
    "EVALUATION_DATA",
    "TARGET_REGISTRY",
    "FEATURE_TYPE_REGISTRY",
]


missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:

    raise RuntimeError(
        "Required predictor-selection state is missing:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_objects
        )
        +
        "\n\n"
        "Run Notebook 06.02 → 06.06 first."
    )


# ============================================================
# 2. SAFE FEATURE ASSOCIATION SCORE
# ============================================================

def _association_score(
    series,
    target_series,
    feature_type,
    target_type
):
    """
    Estimate univariate association between a candidate
    predictor and the target.

    This score is used ONLY for predictor prioritization.
    It is not an evaluation metric for imputation quality.
    """

    valid_mask = (
        series.notna()
        &
        target_series.notna()
    )


    if valid_mask.sum() < 10:
        return 0.0


    x = series.loc[
        valid_mask
    ]

    y = target_series.loc[
        valid_mask
    ]


    # --------------------------------------------------------
    # Numerical predictor
    # --------------------------------------------------------

    if feature_type == "numerical":

        x_numeric = pd.to_numeric(
            x,
            errors="coerce"
        )

        y_numeric = pd.to_numeric(
            y,
            errors="coerce"
        )


        valid_numeric = (
            x_numeric.notna()
            &
            y_numeric.notna()
        )


        if valid_numeric.sum() < 10:
            return 0.0


        x_numeric = x_numeric.loc[
            valid_numeric
        ]

        y_numeric = y_numeric.loc[
            valid_numeric
        ]


        # If target is numerical, use absolute
        # Spearman correlation.
        if target_type == "numerical":

            correlation = (
                x_numeric
                .corr(
                    y_numeric,
                    method="spearman"
                )
            )

            if pd.isna(correlation):
                return 0.0

            return float(
                abs(correlation)
            )


        # Numerical predictor against categorical target.
        # Rank correlation with encoded target provides a
        # deterministic screening signal.
        y_codes = (
            pd.Series(
                pd.factorize(
                    y_numeric
                )[0],
                index=y_numeric.index
            )
        )


        correlation = (
            x_numeric
            .corr(
                y_codes,
                method="spearman"
            )
        )


        if pd.isna(correlation):
            return 0.0


        return float(
            abs(correlation)
        )


    # --------------------------------------------------------
    # Categorical predictor
    # --------------------------------------------------------

    x_codes = pd.Series(
        pd.factorize(x)[0],
        index=x.index
    )


    if target_type == "numerical":

        y_numeric = pd.to_numeric(
            y,
            errors="coerce"
        )


        valid_numeric = (
            y_numeric.notna()
        )


        if valid_numeric.sum() < 10:
            return 0.0


        # Eta-like association using between-category
        # variance relative to total variance.
        y_valid = y_numeric.loc[
            valid_numeric
        ]

        x_valid = x_codes.loc[
            valid_numeric
        ]


        overall_mean = float(
            y_valid.mean()
        )


        total_variance = float(
            ((y_valid - overall_mean) ** 2).sum()
        )


        if total_variance <= 0:
            return 0.0


        between_variance = 0.0


        for category in x_valid.unique():

            group = y_valid[
                x_valid == category
            ]


            if len(group) == 0:
                continue


            between_variance += (
                len(group)
                *
                (
                    float(group.mean())
                    -
                    overall_mean
                ) ** 2
            )


        score = (
            between_variance
            /
            total_variance
        )


        return float(
            np.clip(
                score,
                0.0,
                1.0
            )
        )


    # --------------------------------------------------------
    # Categorical predictor vs categorical target
    # --------------------------------------------------------

    # Cramér's V
    contingency = pd.crosstab(
        x,
        y
    )


    if (
        contingency.shape[0] < 2
        or
        contingency.shape[1] < 2
    ):
        return 0.0


    observed = (
        contingency
        .to_numpy(
            dtype=float
        )
    )


    total = observed.sum()


    if total <= 0:
        return 0.0


    row_totals = observed.sum(
        axis=1,
        keepdims=True
    )

    column_totals = observed.sum(
        axis=0,
        keepdims=True
    )


    expected = (
        row_totals
        *
        column_totals
        /
        total
    )


    valid_expected = (
        expected > 0
    )


    chi_square = np.sum(
        (
            (
                observed
                -
                expected
            ) ** 2
            /
            np.where(
                valid_expected,
                expected,
                1.0
            )
        )
    )


    n = observed.sum()

    phi2 = (
        chi_square / n
    )


    rows, columns = observed.shape

    denominator = min(
        columns - 1,
        rows - 1
    )


    if denominator <= 0:
        return 0.0


    return float(
        np.sqrt(
            phi2 / denominator
        )
    )


# ============================================================
# 3. MAIN PREDICTOR SELECTION FUNCTION
# ============================================================

def select_predictors(
    df,
    target_feature,
    candidate_feature=None,
    max_predictors=20,
    min_observed_fraction=0.50
):
    """
    Select context predictors for imputation.

    Design principles:
      1. Exclude the target.
      2. Exclude the feature currently being imputed.
      3. Remove all-missing and near-empty predictors.
      4. Prefer predictors with stronger association
         with the target.
      5. Prefer predictors with lower missingness when
         association is comparable.
      6. Preserve deterministic ordering.
    """

    if not isinstance(
        df,
        pd.DataFrame
    ):
        raise TypeError(
            "df must be a pandas DataFrame."
        )


    if target_feature not in df.columns:

        raise KeyError(
            f"Target '{target_feature}' not found."
        )


    if max_predictors < 1:

        raise ValueError(
            "max_predictors must be >= 1."
        )


    # --------------------------------------------------------
    # Candidate exclusion
    # --------------------------------------------------------

    excluded = {
        target_feature
    }


    if candidate_feature is not None:

        if candidate_feature not in df.columns:

            raise KeyError(
                f"Candidate feature "
                f"'{candidate_feature}' not found."
            )

        excluded.add(
            candidate_feature
        )


    candidates = [
        column
        for column in df.columns
        if column not in excluded
    ]


    if not candidates:
        return []


    # --------------------------------------------------------
    # Target type
    # --------------------------------------------------------

    target_type = (
        FEATURE_TYPE_REGISTRY
        .get(
            target_feature
        )
    )


    # Dataset-aware registry
    if target_feature not in FEATURE_TYPE_REGISTRY:

        dataset_match = None

        for dataset_id, registry in (
            FEATURE_TYPE_REGISTRY.items()
        ):

            if target_feature in registry:

                dataset_match = registry[
                    target_feature
                ]

                break


        target_type = dataset_match


    if target_type is None:

        target_type = (
            "numerical"
            if pd.api.types.is_numeric_dtype(
                df[target_feature]
            )
            else "categorical"
        )


    # --------------------------------------------------------
    # Score candidate predictors
    # --------------------------------------------------------

    predictor_rows = []


    for feature in candidates:

        series = df[
            feature
        ]


        observed_fraction = float(
            series.notna().mean()
        )


        missing_rate = float(
            series.isna().mean()
        )


        unique_values = int(
            series.nunique(
                dropna=True
            )
        )


        # Exclude predictors that provide too little
        # observed information.
        if (
            observed_fraction
            <
            min_observed_fraction
        ):
            continue


        # Exclude constant predictors.
        if unique_values <= 1:
            continue


        feature_type = None


        # Try to retrieve type from the registry.
        for dataset_id, registry in (
            FEATURE_TYPE_REGISTRY.items()
        ):

            if feature in registry:

                feature_type = registry[
                    feature
                ]

                break


        if feature_type is None:

            feature_type = (
                "numerical"
                if pd.api.types.is_numeric_dtype(
                    series
                )
                else "categorical"
            )


        association = _association_score(
            series=series,
            target_series=df[
                target_feature
            ],
            feature_type=feature_type,
            target_type=target_type,
        )


        predictor_rows.append({

            "feature":
                feature,

            "feature_type":
                feature_type,

            "association_score":
                association,

            "missing_rate":
                missing_rate,

            "observed_fraction":
                observed_fraction,

            "unique_values":
                unique_values,

        })


    if not predictor_rows:
        return []


    predictor_df = pd.DataFrame(
        predictor_rows
    )


    # --------------------------------------------------------
    # Deterministic ranking
    # --------------------------------------------------------

    predictor_df = (
        predictor_df
        .sort_values(
            by=[
                "association_score",
                "observed_fraction",
                "unique_values",
                "feature",
            ],
            ascending=[
                False,
                False,
                False,
                True,
            ],
            kind="mergesort",
        )
    )


    predictor_df = (
        predictor_df
        .head(
            max_predictors
        )
    )


    return predictor_df[
        "feature"
    ].tolist()


# ============================================================
# 4. PREDICTOR-SELECTION DIAGNOSTIC
# ============================================================

print(
    "\n[06.07.01] Predictor-selection function: READY"
)

print(
    "[06.07.02] Leakage exclusion: ENABLED"
)

print(
    "[06.07.03] Missingness filtering: ENABLED"
)

print(
    "[06.07.04] Association-based ranking: ENABLED"
)

print(
    "[06.07.05] Deterministic ranking: ENABLED"
)


# ============================================================
# 5. FUNCTION SANITY CHECK
# ============================================================

for dataset_id in EXPECTED_DATASETS:

    df = EVALUATION_DATA[
        dataset_id
    ]

    target = TARGET_REGISTRY[
        dataset_id
    ]


    predictors = select_predictors(
        df=df,
        target_feature=target,
        max_predictors=10,
    )


    if not isinstance(
        predictors,
        list
    ):

        raise RuntimeError(
            f"Predictor selection failed for "
            f"{dataset_id}."
        )


    if target in predictors:

        raise RuntimeError(
            f"Target leakage detected for "
            f"{dataset_id}."
        )


    print(
        f"\n{dataset_id}"
    )

    print(
        f"  Target    : {target}"
    )

    print(
        f"  Predictors: {len(predictors)}"
    )

    print(
        f"  Top       : {predictors[:5]}"
    )


# ============================================================
# 6. FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 100
)

print(
    "PREDICTOR SELECTION : PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.07
LEAKAGE-SAFE / DATA-AWARE PREDICTOR SELECTION

[06.07.01] Predictor-selection function: READY
[06.07.02] Leakage exclusion: ENABLED
[06.07.03] Missingness filtering: ENABLED
[06.07.04] Association-based ranking: ENABLED
[06.07.05] Deterministic ranking: ENABLED

adult_income
  Target    : income
  Predictors: 10
  Top       : ['relationship', 'marital_status', 'education', 'occupation', 'sex']

bank_marketing
  Target    : y
  Predictors: 10
  Top       : ['poutcome', 'month', 'contact', 'housing', 'job']

diabetes_130us
  Target    : readmitted
  Predictors: 10
  Top       : ['diag_1', 'diag_2', 'diag_3', 'medical_specialty', 'payer_code']

PREDICTOR SELECTION : PASSED


In [8]:
# ============================================================
# NOTEBOOK 06.08 — MISSINGNESS MASK GENERATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.08")
print("ROBUST MISSINGNESS-MASK GENERATION")
print("=" * 100)


# ============================================================
# 1. REQUIRED STATE
# ============================================================

required_objects = [
    "np",
    "pd",
    "EVALUATION_DATA",
    "TARGET_REGISTRY",
    "FEATURE_TYPE_REGISTRY",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required state missing for Notebook 06.08:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing_objects
        )
        +
        "\n\nRun Notebook 06.02 through 06.07 first."
    )


# ============================================================
# 2. PREPARE PREDICTORS FOR MAR
# ============================================================

def _prepare_mar_predictors(
    predictors
):
    """
    Convert observed predictors into numeric variables
    suitable for deterministic MAR propensity construction.

    Numerical:
        median imputation + standardization

    Categorical:
        frequency encoding
    """

    if not isinstance(
        predictors,
        pd.DataFrame
    ):
        raise TypeError(
            "predictors must be a pandas DataFrame."
        )

    if predictors.empty:
        return pd.DataFrame(
            index=predictors.index
        )

    encoded = pd.DataFrame(
        index=predictors.index
    )

    for column in predictors.columns:

        series = predictors[column]

        # ----------------------------------------------------
        # Numerical predictor
        # ----------------------------------------------------

        if pd.api.types.is_numeric_dtype(
            series
        ):

            values = pd.to_numeric(
                series,
                errors="coerce"
            )

            median_value = values.median()

            if pd.isna(median_value):
                median_value = 0.0

            values = values.fillna(
                median_value
            )

            std_value = values.std()

            if (
                pd.isna(std_value)
                or
                std_value == 0
            ):

                encoded[column] = 0.0

            else:

                encoded[column] = (
                    values - values.mean()
                ) / std_value

        # ----------------------------------------------------
        # Categorical predictor
        # ----------------------------------------------------

        else:

            values = (
                series
                .astype("string")
                .fillna("__MISSING__")
            )

            frequencies = (
                values
                .value_counts(
                    normalize=True
                )
            )

            encoded[column] = (
                values
                .map(
                    frequencies
                )
                .fillna(0.0)
                .astype(float)
            )

    return encoded


# ============================================================
# 3. BUILD MAR PROPENSITY
# ============================================================

def _build_mar_propensity(
    predictors,
    random_state=42
):
    """
    Construct a deterministic propensity score from
    observed predictor information.
    """

    encoded = _prepare_mar_predictors(
        predictors
    )

    if encoded.empty:
        return None

    values = encoded.to_numpy(
        dtype=float
    )

    values = np.nan_to_num(
        values,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    if values.shape[1] == 0:
        return None

    raw_score = values.mean(
        axis=1
    )

    score = (
        pd.Series(
            raw_score,
            index=predictors.index
        )
        .rank(
            method="average",
            pct=True
        )
        .to_numpy()
    )

    rng = np.random.default_rng(
        int(random_state)
    )

    score = (
        score
        +
        rng.uniform(
            0.0,
            1e-10,
            size=len(score)
        )
    )

    return score


# ============================================================
# 4. MAIN MISSINGNESS MASK GENERATOR
# ============================================================

def generate_missingness_mask(
    series,
    rate,
    mechanism="MCAR",
    seed=42,
    predictors=None
):
    """
    Generate a deterministic missingness mask.

    True  = value selected for artificial masking
    False = value remains observed

    Only originally observed values are eligible for masking.
    """

    # --------------------------------------------------------
    # Validate target series
    # --------------------------------------------------------

    if not isinstance(
        series,
        pd.Series
    ):
        raise TypeError(
            "series must be a pandas Series."
        )

    # --------------------------------------------------------
    # Validate mechanism
    # --------------------------------------------------------

    mechanism = str(
        mechanism
    ).strip().upper()

    if mechanism not in {
        "MCAR",
        "MAR",
    }:
        raise ValueError(
            "Supported mechanisms are MCAR and MAR."
        )

    # --------------------------------------------------------
    # Validate rate
    # --------------------------------------------------------

    rate = float(rate)

    if not 0 < rate < 1:
        raise ValueError(
            "Missingness rate must satisfy 0 < rate < 1."
        )

    rng = np.random.default_rng(
        int(seed)
    )

    # --------------------------------------------------------
    # Preserve originally missing values
    # --------------------------------------------------------

    observed = (
        series.notna()
        .to_numpy()
    )

    candidate_indices = np.flatnonzero(
        observed
    )

    mask = pd.Series(
        False,
        index=series.index,
        dtype=bool
    )

    if len(candidate_indices) == 0:
        return mask

    # --------------------------------------------------------
    # Number of values to mask
    # --------------------------------------------------------

    n_remove = int(
        round(
            len(candidate_indices)
            *
            rate
        )
    )

    n_remove = max(
        1,
        min(
            n_remove,
            len(candidate_indices)
        )
    )

    # ========================================================
    # MCAR
    # ========================================================

    if mechanism == "MCAR":

        selected = rng.choice(
            candidate_indices,
            size=n_remove,
            replace=False
        )

    # ========================================================
    # MAR
    # ========================================================

    else:

        if predictors is None:
            raise ValueError(
                "MAR requires observed predictor data."
            )

        if not isinstance(
            predictors,
            pd.DataFrame
        ):
            raise TypeError(
                "predictors must be a pandas DataFrame."
            )

        if not predictors.index.equals(
            series.index
        ):
            raise ValueError(
                "Predictor index must exactly match "
                "the target series index."
            )

        predictor_df = predictors.drop(
            columns=[
                series.name
            ],
            errors="ignore"
        )

        if predictor_df.empty:
            raise ValueError(
                "MAR requires at least one predictor."
            )

        propensity = _build_mar_propensity(
            predictor_df,
            random_state=seed
        )

        if propensity is None:
            raise ValueError(
                "Unable to construct MAR propensity."
            )

        candidate_scores = propensity[
            candidate_indices
        ]

        tie_break = rng.uniform(
            0.0,
            1e-10,
            size=len(candidate_scores)
        )

        ranking_score = (
            candidate_scores
            +
            tie_break
        )

        order = np.argsort(
            ranking_score
        )

        selected = candidate_indices[
            order[-n_remove:]
        ]

    # --------------------------------------------------------
    # Construct mask
    # --------------------------------------------------------

    mask.iloc[
        selected
    ] = True

    return mask


# ============================================================
# 5. MASK VALIDATION FUNCTION
# ============================================================

def validate_missingness_mask(
    series,
    mask,
    requested_rate,
    mechanism
):
    """
    Validate structural correctness and requested masking rate.
    """

    if not isinstance(
        mask,
        pd.Series
    ):
        raise RuntimeError(
            "Missingness mask must be a pandas Series."
        )

    if not mask.index.equals(
        series.index
    ):
        raise RuntimeError(
            "Mask index does not match target series."
        )

    # Never mask values that were already missing.
    illegal_masking = (
        mask
        &
        series.isna()
    )

    if illegal_masking.any():
        raise RuntimeError(
            "Mask attempts to overwrite originally "
            "missing values."
        )

    observed_count = int(
        series.notna().sum()
    )

    masked_count = int(
        mask.sum()
    )

    if observed_count == 0:

        return {
            "valid": True,
            "mechanism": str(mechanism),
            "requested_rate": float(
                requested_rate
            ),
            "actual_rate": 0.0,
            "masked_count": 0,
            "observed_count": 0,
        }

    actual_rate = (
        masked_count
        /
        observed_count
    )

    # Because masking operates on integer cells,
    # the realized rate can differ slightly from the
    # requested rate.
    tolerance = max(
        1.0 / observed_count,
        1e-12
    )

    valid = (
        masked_count > 0
        and
        abs(
            actual_rate
            -
            float(requested_rate)
        )
        <= tolerance
    )

    return {
        "valid": bool(valid),
        "mechanism": str(mechanism),
        "requested_rate": float(
            requested_rate
        ),
        "actual_rate": float(
            actual_rate
        ),
        "masked_count": masked_count,
        "observed_count": observed_count,
    }


# ============================================================
# 6. INITIALIZATION REPORT
# ============================================================

print(
    "\n[06.08.01] Missingness generator initialized."
)

print(
    "[06.08.02] MCAR generation: ENABLED"
)

print(
    "[06.08.03] MAR generation: ENABLED"
)

print(
    "[06.08.04] Numerical predictors: ENABLED"
)

print(
    "[06.08.05] Categorical predictors: ENABLED"
)

print(
    "[06.08.06] Original missing-value protection: ENABLED"
)

print(
    "[06.08.07] Deterministic seed control: ENABLED"
)


# ============================================================
# 7. SANITY-CHECK DATASET
# ============================================================

_test_dataset = EXPECTED_DATASETS[0]

_test_df = EVALUATION_DATA[
    _test_dataset
]

_test_target = TARGET_REGISTRY[
    _test_dataset
]

# Use age when available; otherwise use first
# non-target feature.
if "age" in _test_df.columns:

    _test_feature = "age"

else:

    _test_feature = next(
        column
        for column in _test_df.columns
        if column != _test_target
    )


# ============================================================
# 8. BUILD TEST PREDICTOR SET
# ============================================================

_test_predictor_names = select_predictors(
    _test_df,
    target_feature=_test_target,
    candidate_feature=_test_feature,
    max_predictors=10,
)

_test_predictors = _test_df[
    _test_predictor_names
].copy()


# ============================================================
# 9. MCAR TEST
# ============================================================

_test_mcar_mask = generate_missingness_mask(
    series=_test_df[
        _test_feature
    ],
    rate=0.10,
    mechanism="MCAR",
    seed=42,
    predictors=_test_predictors,
)

_test_mcar_validation = validate_missingness_mask(
    series=_test_df[
        _test_feature
    ],
    mask=_test_mcar_mask,
    requested_rate=0.10,
    mechanism="MCAR",
)


# ============================================================
# 10. MAR TEST
# ============================================================

_test_mar_mask = generate_missingness_mask(
    series=_test_df[
        _test_feature
    ],
    rate=0.10,
    mechanism="MAR",
    seed=42,
    predictors=_test_predictors,
)

_test_mar_validation = validate_missingness_mask(
    series=_test_df[
        _test_feature
    ],
    mask=_test_mar_mask,
    requested_rate=0.10,
    mechanism="MAR",
)


# ============================================================
# 11. VALIDATION REPORT
# ============================================================

MASK_VALIDATION_DF = pd.DataFrame([
    _test_mcar_validation,
    _test_mar_validation,
])


display(
    MASK_VALIDATION_DF
)


if not bool(
    MASK_VALIDATION_DF[
        "valid"
    ].all()
):
    raise RuntimeError(
        "Missingness-mask validation failed."
    )


# ============================================================
# 12. FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 100
)

print(
    "MISSINGNESS-MASK GENERATION : PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.08
ROBUST MISSINGNESS-MASK GENERATION

[06.08.01] Missingness generator initialized.
[06.08.02] MCAR generation: ENABLED
[06.08.03] MAR generation: ENABLED
[06.08.04] Numerical predictors: ENABLED
[06.08.05] Categorical predictors: ENABLED
[06.08.06] Original missing-value protection: ENABLED
[06.08.07] Deterministic seed control: ENABLED


,valid,mechanism,requested_rate,actual_rate,masked_count,observed_count
0,True,MCAR,0.1,0.099997,3256,32561
1,True,MAR,0.1,0.099997,3256,32561



MISSINGNESS-MASK GENERATION : PASSED


In [9]:
# ============================================================
# NOTEBOOK 06.09 — PREDICTOR ENCODING PIPELINE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.09")
print("BUILD LEAKAGE-SAFE PREDICTOR ENCODING PIPELINE")
print("=" * 100)


# ============================================================
# 1. REQUIRED IMPORTS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)


# ============================================================
# 2. PREDICTOR ENCODING FUNCTION
# ============================================================

def encode_predictors(
    X_train,
    X_test,
    sparse_output=True
):
    """
    Build a leakage-safe predictor preprocessing pipeline.

    Numerical predictors:
        median imputation
        standardization

    Categorical predictors:
        most-frequent imputation
        one-hot encoding
        unseen categories ignored

    The encoder is fitted ONLY on X_train and subsequently
    applied to X_test.

    Parameters
    ----------
    X_train : pandas.DataFrame
        Training predictors.

    X_test : pandas.DataFrame
        Validation/test predictors.

    sparse_output : bool
        Whether one-hot encoded output should remain sparse.

    Returns
    -------
    X_train_encoded
    X_test_encoded
    predictor_encoder
    """

    # --------------------------------------------------------
    # Validate input
    # --------------------------------------------------------

    if not isinstance(
        X_train,
        pd.DataFrame
    ):
        raise TypeError(
            "X_train must be a pandas DataFrame."
        )

    if not isinstance(
        X_test,
        pd.DataFrame
    ):
        raise TypeError(
            "X_test must be a pandas DataFrame."
        )

    if X_train.empty:
        raise ValueError(
            "X_train contains no predictor columns."
        )

    if X_test.empty:
        raise ValueError(
            "X_test contains no predictor columns."
        )

    # --------------------------------------------------------
    # Ensure identical predictor schema
    # --------------------------------------------------------

    if list(
        X_train.columns
    ) != list(
        X_test.columns
    ):

        raise ValueError(
            "X_train and X_test must contain the same "
            "predictor columns in the same order."
        )

    # --------------------------------------------------------
    # Copy inputs
    # --------------------------------------------------------

    X_train = X_train.copy()

    X_test = X_test.copy()

    # --------------------------------------------------------
    # Detect numerical predictors
    # --------------------------------------------------------

    numerical_columns = (
        X_train
        .select_dtypes(
            include=[np.number]
        )
        .columns
        .tolist()
    )

    # --------------------------------------------------------
    # Detect categorical predictors
    # --------------------------------------------------------

    categorical_columns = [
        column
        for column in X_train.columns
        if column not in numerical_columns
    ]

    transformers = []

    # ========================================================
    # 3. NUMERICAL PIPELINE
    # ========================================================

    if numerical_columns:

        numerical_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            ),
        ])

        transformers.append(
            (
                "numerical",
                numerical_pipeline,
                numerical_columns,
            )
        )

    # ========================================================
    # 4. CATEGORICAL PIPELINE
    # ========================================================

    if categorical_columns:

        categorical_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=sparse_output,
                    dtype=np.float32,
                )
            ),
        ])

        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                categorical_columns,
            )
        )

    # --------------------------------------------------------
    # Validate transformer construction
    # --------------------------------------------------------

    if not transformers:

        raise RuntimeError(
            "Unable to construct predictor preprocessing "
            "pipeline."
        )

    # ========================================================
    # 5. COLUMN TRANSFORMER
    # ========================================================

    predictor_encoder = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False,
    )

    # ========================================================
    # 6. FIT ONLY ON TRAINING DATA
    # ========================================================

    X_train_encoded = (
        predictor_encoder
        .fit_transform(
            X_train
        )
    )

    # ========================================================
    # 7. TRANSFORM TEST DATA
    # ========================================================

    X_test_encoded = (
        predictor_encoder
        .transform(
            X_test
        )
    )

    # ========================================================
    # 8. OUTPUT VALIDATION
    # ========================================================

    if X_train_encoded.shape[1] == 0:

        raise RuntimeError(
            "Predictor encoding produced zero features."
        )

    if (
        X_train_encoded.shape[1]
        !=
        X_test_encoded.shape[1]
    ):

        raise RuntimeError(
            "Encoded train/test feature dimensions differ."
        )

    return (
        X_train_encoded,
        X_test_encoded,
        predictor_encoder,
    )


# ============================================================
# 9. DIAGNOSTIC REPORT
# ============================================================

print(
    "\n[06.09.01] Numerical preprocessing : "
    "median imputation + standardization"
)

print(
    "[06.09.02] Categorical preprocessing : "
    "mode imputation + one-hot encoding"
)

print(
    "[06.09.03] Unknown categories : "
    "ignored safely"
)

print(
    "[06.09.04] Encoder fitting : "
    "TRAINING DATA ONLY"
)

print(
    "[06.09.05] Sparse encoding : "
    "AVAILABLE"
)

print(
    "[06.09.06] Leakage protection : "
    "ENABLED"
)


# ============================================================
# 10. FUNCTION SANITY CHECK
# ============================================================

_test_dataset = EXPECTED_DATASETS[0]

_test_df = EVALUATION_DATA[
    _test_dataset
]

_test_target = TARGET_REGISTRY[
    _test_dataset
]

_test_feature = (
    "age"
    if "age" in _test_df.columns
    else next(
        column
        for column in _test_df.columns
        if column != _test_target
    )
)

_test_predictor_names = select_predictors(
    _test_df,
    target_feature=_test_target,
    candidate_feature=_test_feature,
    max_predictors=5,
)

_test_X = _test_df[
    _test_predictor_names
].copy()

_test_split = max(
    1,
    int(
        len(_test_X) * 0.8
    )
)

_test_X_train = _test_X.iloc[
    :_test_split
].copy()

_test_X_test = _test_X.iloc[
    _test_split:
].copy()


(
    _test_train_encoded,
    _test_test_encoded,
    _test_encoder,
) = encode_predictors(
    _test_X_train,
    _test_X_test,
    sparse_output=True,
)


if (
    _test_train_encoded.shape[1]
    !=
    _test_test_encoded.shape[1]
):

    raise RuntimeError(
        "06.09 sanity check failed: "
        "train/test encoded dimensions differ."
    )


print(
    "\n[06.09.07] Sanity check dataset : "
    f"{_test_dataset}"
)

print(
    "[06.09.08] Predictor count : "
    f"{len(_test_predictor_names)}"
)

print(
    "[06.09.09] Encoded dimensions : "
    f"{_test_train_encoded.shape[1]}"
)

print(
    "\n" + "=" * 100
)

print(
    "PREDICTOR ENCODING PIPELINE : PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.09
BUILD LEAKAGE-SAFE PREDICTOR ENCODING PIPELINE

[06.09.01] Numerical preprocessing : median imputation + standardization
[06.09.02] Categorical preprocessing : mode imputation + one-hot encoding
[06.09.03] Unknown categories : ignored safely
[06.09.04] Encoder fitting : TRAINING DATA ONLY
[06.09.05] Sparse encoding : AVAILABLE
[06.09.06] Leakage protection : ENABLED

[06.09.07] Sanity check dataset : adult_income
[06.09.08] Predictor count : 5
[06.09.09] Encoded dimensions : 45

PREDICTOR ENCODING PIPELINE : PASSED


In [10]:

# ============================================================
# NOTEBOOK 06.10 — STATISTICAL CANDIDATE IMPUTERS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.10")
print("STATISTICAL CANDIDATE IMPUTERS")
print("=" * 100)


# ============================================================
# 1. REQUIRED IMPORTS
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 2. UTILITY: VALIDATE INPUT
# ============================================================

def _validate_imputation_inputs(
    train_series,
    test_series
):

    if not isinstance(
        train_series,
        pd.Series
    ):
        raise TypeError(
            "train_series must be a pandas Series."
        )

    if not isinstance(
        test_series,
        pd.Series
    ):
        raise TypeError(
            "test_series must be a pandas Series."
        )

    if train_series.empty:
        raise ValueError(
            "train_series is empty."
        )

    if test_series.empty:
        raise ValueError(
            "test_series is empty."
        )

    if train_series.isna().all():
        raise ValueError(
            "train_series contains no observed values."
        )


# ============================================================
# 3. MEAN IMPUTATION
# ============================================================

def impute_mean(
    train_series,
    test_series
):

    _validate_imputation_inputs(
        train_series,
        test_series
    )

    if not pd.api.types.is_numeric_dtype(
        train_series
    ):
        raise TypeError(
            "Mean imputation requires a numerical feature."
        )

    value = train_series.mean()

    if pd.isna(value):
        raise ValueError(
            "Unable to estimate mean from training data."
        )

    result = test_series.copy()

    result = result.fillna(
        value
    )

    return result


# ============================================================
# 4. MEDIAN IMPUTATION
# ============================================================

def impute_median(
    train_series,
    test_series
):

    _validate_imputation_inputs(
        train_series,
        test_series
    )

    if not pd.api.types.is_numeric_dtype(
        train_series
    ):
        raise TypeError(
            "Median imputation requires a numerical feature."
        )

    value = train_series.median()

    if pd.isna(value):
        raise ValueError(
            "Unable to estimate median from training data."
        )

    result = test_series.copy()

    result = result.fillna(
        value
    )

    return result


# ============================================================
# 5. MODE IMPUTATION
# ============================================================

def impute_mode(
    train_series,
    test_series
):

    _validate_imputation_inputs(
        train_series,
        test_series
    )

    mode_values = (
        train_series
        .dropna()
        .mode()
    )

    if mode_values.empty:
        raise ValueError(
            "Unable to estimate mode from training data."
        )

    value = mode_values.iloc[0]

    result = test_series.copy()

    result = result.fillna(
        value
    )

    return result


# ============================================================
# 6. CONSTANT IMPUTATION
# ============================================================

def impute_constant(
    train_series,
    test_series,
    constant_value=None
):

    _validate_imputation_inputs(
        train_series,
        test_series
    )

    if constant_value is None:

        if pd.api.types.is_numeric_dtype(
            train_series
        ):
            constant_value = 0.0

        else:
            constant_value = "MISSING"

    result = test_series.copy()

    result = result.fillna(
        constant_value
    )

    return result


# ============================================================
# 7. RANDOM-SAMPLE IMPUTATION
# ============================================================

def impute_random_sample(
    train_series,
    test_series,
    random_state=42
):

    _validate_imputation_inputs(
        train_series,
        test_series
    )

    observed = (
        train_series
        .dropna()
        .to_numpy()
    )

    if len(observed) == 0:
        raise ValueError(
            "Random-sample imputation requires "
            "observed training values."
        )

    result = test_series.copy()

    missing_mask = result.isna()

    missing_count = int(
        missing_mask.sum()
    )

    if missing_count == 0:
        return result

    rng = np.random.default_rng(
        int(random_state)
    )

    sampled_values = rng.choice(
        observed,
        size=missing_count,
        replace=True
    )

    result.loc[
        missing_mask
    ] = sampled_values

    return result


# ============================================================
# 8. STRATEGY REGISTRY
# ============================================================

STATISTICAL_IMPUTERS = {

    "mean":
        impute_mean,

    "median":
        impute_median,

    "mode":
        impute_mode,

    "constant":
        impute_constant,

    "random_sample":
        impute_random_sample,
}


# ============================================================
# 9. BASIC IMPLEMENTATION VALIDATION
# ============================================================

if not STATISTICAL_IMPUTERS:

    raise RuntimeError(
        "Statistical imputer registry is empty."
    )


required_statistical_strategies = {
    "mean",
    "median",
    "mode",
    "constant",
    "random_sample",
}


missing_statistical_strategies = (
    required_statistical_strategies
    -
    set(STATISTICAL_IMPUTERS.keys())
)


if missing_statistical_strategies:

    raise RuntimeError(
        "Missing statistical strategies:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in sorted(
                missing_statistical_strategies
            )
        )
    )


# ============================================================
# 10. FUNCTION SANITY TEST
# ============================================================

_test_train_numeric = pd.Series(
    [10.0, 20.0, 30.0, 40.0],
    name="numeric_test"
)

_test_masked_numeric = pd.Series(
    [10.0, np.nan, 30.0, np.nan],
    name="numeric_test"
)

_test_mean = impute_mean(
    _test_train_numeric,
    _test_masked_numeric
)

_test_median = impute_median(
    _test_train_numeric,
    _test_masked_numeric
)

_test_random = impute_random_sample(
    _test_train_numeric,
    _test_masked_numeric,
    random_state=42
)


if _test_mean.isna().any():
    raise RuntimeError(
        "Mean imputation sanity check failed."
    )

if _test_median.isna().any():
    raise RuntimeError(
        "Median imputation sanity check failed."
    )

if _test_random.isna().any():
    raise RuntimeError(
        "Random-sample imputation sanity check failed."
    )


_test_train_categorical = pd.Series(
    ["A", "B", "A", "C"],
    name="categorical_test"
)

_test_masked_categorical = pd.Series(
    ["A", np.nan, "A", np.nan],
    name="categorical_test"
)

_test_mode = impute_mode(
    _test_train_categorical,
    _test_masked_categorical
)

_test_constant = impute_constant(
    _test_train_categorical,
    _test_masked_categorical
)

if _test_mode.isna().any():
    raise RuntimeError(
        "Mode imputation sanity check failed."
    )

if _test_constant.isna().any():
    raise RuntimeError(
        "Constant imputation sanity check failed."
    )


# ============================================================
# 11. FINAL REPORT
# ============================================================

print(
    "\n[06.10.01] Mean imputation       : READY"
)

print(
    "[06.10.02] Median imputation     : READY"
)

print(
    "[06.10.03] Mode imputation       : READY"
)

print(
    "[06.10.04] Constant imputation   : READY"
)

print(
    "[06.10.05] Random-sample         : READY"
)

print(
    "[06.10.06] Training-only fitting : ENABLED"
)

print(
    "[06.10.07] Leakage protection    : ENABLED"
)

print(
    "\nRegistered statistical strategies:"
)

for strategy_id in sorted(
    STATISTICAL_IMPUTERS
):

    print(
        f"  ✓ {strategy_id}"
    )


print(
    "\nSanity tests: PASSED"
)

print(
    "\n" + "=" * 100
)

print(
    "STATISTICAL CANDIDATE IMPUTERS : PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.10
STATISTICAL CANDIDATE IMPUTERS

[06.10.01] Mean imputation       : READY
[06.10.02] Median imputation     : READY
[06.10.03] Mode imputation       : READY
[06.10.04] Constant imputation   : READY
[06.10.05] Random-sample         : READY
[06.10.06] Training-only fitting : ENABLED
[06.10.07] Leakage protection    : ENABLED

Registered statistical strategies:
  ✓ constant
  ✓ mean
  ✓ median
  ✓ mode
  ✓ random_sample

Sanity tests: PASSED

STATISTICAL CANDIDATE IMPUTERS : PASSED


In [11]:
# ============================================================
# NOTEBOOK 06.11 — KNN IMPUTATION CANDIDATE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.11")
print("LEAKAGE-SAFE KNN IMPUTATION CANDIDATE")
print("=" * 100)


# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler


# ============================================================
# 2. KNN IMPUTATION FUNCTION
# ============================================================

def fit_knn(
    X_train,
    X_test=None,
    n_neighbors=5,
    weights="uniform"
):
    """
    Leakage-safe KNN imputation.

    Parameters
    ----------
    X_train : pandas.DataFrame or numpy.ndarray
        Training feature matrix containing missing values.

    X_test : pandas.DataFrame or numpy.ndarray, optional
        Validation/test feature matrix containing missing values.

    n_neighbors : int
        Number of nearest neighbours.

    weights : {"uniform", "distance"}
        Weighting scheme used by KNNImputer.

    Returns
    -------
    X_train_imputed
        Imputed training matrix.

    X_test_imputed
        Imputed test matrix, or None.

    imputer
        Fitted KNNImputer.

    scaler
        Fitted StandardScaler.
    """

    # --------------------------------------------------------
    # Validate number of neighbours
    # --------------------------------------------------------

    if not isinstance(
        n_neighbors,
        int
    ):
        raise TypeError(
            "n_neighbors must be an integer."
        )

    if n_neighbors < 1:
        raise ValueError(
            "n_neighbors must be >= 1."
        )

    if weights not in {
        "uniform",
        "distance"
    }:

        raise ValueError(
            "weights must be 'uniform' or 'distance'."
        )

    # --------------------------------------------------------
    # Convert to DataFrame for consistent handling
    # --------------------------------------------------------

    if isinstance(
        X_train,
        pd.DataFrame
    ):

        train_columns = (
            X_train.columns.tolist()
        )

        X_train_work = X_train.copy()

    else:

        X_train_work = pd.DataFrame(
            X_train
        )

        train_columns = (
            X_train_work.columns.tolist()
        )

    if X_train_work.empty:

        raise ValueError(
            "X_train contains no features."
        )

    # --------------------------------------------------------
    # Validate numeric input
    # --------------------------------------------------------

    non_numeric_columns = (
        X_train_work
        .select_dtypes(
            exclude=[np.number]
        )
        .columns
        .tolist()
    )

    if non_numeric_columns:

        raise TypeError(
            "KNN candidate requires numerical input. "
            "Encode categorical predictors before calling "
            "fit_knn(). Non-numerical columns: "
            +
            ", ".join(
                map(
                    str,
                    non_numeric_columns
                )
            )
        )

    # --------------------------------------------------------
    # Validate finite values
    # --------------------------------------------------------

    X_train_work = X_train_work.replace(
        [np.inf, -np.inf],
        np.nan
    )

    if X_test is not None:

        if isinstance(
            X_test,
            pd.DataFrame
        ):

            if list(
                X_test.columns
            ) != train_columns:

                raise ValueError(
                    "X_train and X_test must contain "
                    "identical columns in identical order."
                )

            X_test_work = X_test.copy()

        else:

            X_test_work = pd.DataFrame(
                X_test,
                columns=train_columns
            )

        X_test_work = X_test_work.replace(
            [np.inf, -np.inf],
            np.nan
        )

    else:

        X_test_work = None

    # ========================================================
    # 3. STANDARDIZATION
    # ========================================================

    scaler = StandardScaler()

    X_train_scaled = (
        scaler.fit_transform(
            X_train_work
        )
    )

    if X_test_work is not None:

        X_test_scaled = (
            scaler.transform(
                X_test_work
            )
        )

    else:

        X_test_scaled = None

    # ========================================================
    # 4. KNN IMPUTATION
    # ========================================================

    effective_neighbors = min(
        n_neighbors,
        max(
            1,
            len(X_train_work) - 1
        )
    )

    imputer = KNNImputer(
        n_neighbors=effective_neighbors,
        weights=weights
    )

    X_train_imputed = (
        imputer.fit_transform(
            X_train_scaled
        )
    )

    if X_test_scaled is not None:

        X_test_imputed = (
            imputer.transform(
                X_test_scaled
            )
        )

    else:

        X_test_imputed = None

    # ========================================================
    # 5. OUTPUT VALIDATION
    # ========================================================

    if not np.isfinite(
        X_train_imputed
    ).all():

        raise RuntimeError(
            "KNN imputation produced non-finite "
            "training values."
        )

    if (
        X_test_imputed is not None
        and
        not np.isfinite(
            X_test_imputed
        ).all()
    ):

        raise RuntimeError(
            "KNN imputation produced non-finite "
            "test values."
        )

    return (
        X_train_imputed,
        X_test_imputed,
        imputer,
        scaler
    )


# ============================================================
# 6. KNN STRATEGY WRAPPER
# ============================================================

def impute_knn(
    train_matrix,
    test_matrix,
    n_neighbors=5,
    weights="uniform"
):
    """
    Convenience wrapper for AIR-LLM candidate evaluation.
    """

    (
        train_imputed,
        test_imputed,
        imputer,
        scaler
    ) = fit_knn(
        train_matrix,
        test_matrix,
        n_neighbors=n_neighbors,
        weights=weights
    )

    return {
        "train_imputed": train_imputed,
        "test_imputed": test_imputed,
        "imputer": imputer,
        "scaler": scaler,
        "n_neighbors": n_neighbors,
        "weights": weights,
    }


# ============================================================
# 7. KNN CONFIGURATION
# ============================================================

KNN_CONFIG = {

    "strategy_id":
        "knn",

    "method":
        "KNNImputer",

    "n_neighbors":
        5,

    "weights":
        "uniform",

    "scaling":
        "standard",

    "leakage_safe":
        True,
}


# ============================================================
# 8. SANITY TEST
# ============================================================

_test_train = pd.DataFrame({

    "x1": [
        1.0,
        2.0,
        3.0,
        4.0,
        5.0
    ],

    "x2": [
        10.0,
        20.0,
        np.nan,
        40.0,
        50.0
    ],

    "x3": [
        100.0,
        np.nan,
        300.0,
        400.0,
        500.0
    ],
})


_test_test = pd.DataFrame({

    "x1": [
        6.0,
        7.0
    ],

    "x2": [
        np.nan,
        70.0
    ],

    "x3": [
        600.0,
        np.nan
    ],
})


(
    _test_train_imputed,
    _test_test_imputed,
    _test_imputer,
    _test_scaler
) = fit_knn(
    _test_train,
    _test_test,
    n_neighbors=3
)


if not np.isfinite(
    _test_train_imputed
).all():

    raise RuntimeError(
        "KNN training sanity test failed."
    )


if not np.isfinite(
    _test_test_imputed
).all():

    raise RuntimeError(
        "KNN test sanity test failed."
    )


if (
    _test_train_imputed.shape
    !=
    _test_train.shape
):

    raise RuntimeError(
        "KNN training output shape changed."
    )


if (
    _test_test_imputed.shape
    !=
    _test_test.shape
):

    raise RuntimeError(
        "KNN test output shape changed."
    )


# ============================================================
# 9. FINAL VALIDATION REPORT
# ============================================================

print(
    "\n[06.11.01] KNN imputer              : READY"
)

print(
    "[06.11.02] Distance scaling         : ENABLED"
)

print(
    "[06.11.03] Training-only fitting    : ENABLED"
)

print(
    "[06.11.04] Test transformation      : ENABLED"
)

print(
    "[06.11.05] Non-finite protection    : ENABLED"
)

print(
    "[06.11.06] Parameter validation     : ENABLED"
)

print(
    "[06.11.07] Reproducible configuration: READY"
)

print(
    "\nKNN configuration:"
)

for key, value in KNN_CONFIG.items():

    print(
        f"  {key:<20}: {value}"
    )

print(
    "\nSanity test: PASSED"
)

print(
    "\n" + "=" * 100
)

print(
    "KNN IMPUTATION CANDIDATE : PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.11
LEAKAGE-SAFE KNN IMPUTATION CANDIDATE

[06.11.01] KNN imputer              : READY
[06.11.02] Distance scaling         : ENABLED
[06.11.03] Training-only fitting    : ENABLED
[06.11.04] Test transformation      : ENABLED
[06.11.05] Non-finite protection    : ENABLED
[06.11.06] Parameter validation     : ENABLED
[06.11.07] Reproducible configuration: READY

KNN configuration:
  strategy_id         : knn
  method              : KNNImputer
  n_neighbors         : 5
  weights             : uniform
  scaling             : standard
  leakage_safe        : True

Sanity test: PASSED

KNN IMPUTATION CANDIDATE : PASSED


In [12]:
# ============================================================
# NOTEBOOK 06.12 — ITERATIVE / MICE CANDIDATES
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.12")
print("LEAKAGE-SAFE ITERATIVE / MICE IMPUTATION CANDIDATES")
print("=" * 100)


# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.experimental import (
    enable_iterative_imputer  # noqa: F401
)

from sklearn.impute import IterativeImputer

from sklearn.linear_model import BayesianRidge


# ============================================================
# 2. COMMON VALIDATION
# ============================================================

def _validate_iterative_inputs(
    X_train,
    X_test=None
):

    if isinstance(
        X_train,
        pd.DataFrame
    ):

        X_train_work = X_train.copy()

    else:

        X_train_work = pd.DataFrame(
            X_train
        )

    if X_train_work.empty:

        raise ValueError(
            "X_train contains no features."
        )

    non_numeric = (
        X_train_work
        .select_dtypes(
            exclude=[np.number]
        )
        .columns
        .tolist()
    )

    if non_numeric:

        raise TypeError(
            "Iterative/MICE imputation requires "
            "numerical input. Encode categorical "
            "predictors before calling this function. "
            "Non-numerical columns: "
            +
            ", ".join(
                map(
                    str,
                    non_numeric
                )
            )
        )

    X_train_work = (
        X_train_work
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
    )

    if X_test is not None:

        if isinstance(
            X_test,
            pd.DataFrame
        ):

            X_test_work = X_test.copy()

            if list(
                X_train_work.columns
            ) != list(
                X_test_work.columns
            ):

                raise ValueError(
                    "X_train and X_test must contain "
                    "identical columns in identical order."
                )

        else:

            X_test_work = pd.DataFrame(
                X_test,
                columns=X_train_work.columns
            )

        X_test_work = (
            X_test_work
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
        )

    else:

        X_test_work = None

    return (
        X_train_work,
        X_test_work
    )


# ============================================================
# 3. ITERATIVE IMPUTER
# ============================================================

def fit_iterative(
    X_train,
    X_test=None,
    estimator=None,
    random_state=42,
    max_iter=10,
    tol=1e-3,
    initial_strategy="median"
):
    """
    Leakage-safe iterative regression imputation.

    The imputer is fitted only on X_train and then
    applied to X_test.

    Suitable for numerical feature representations.
    """

    (
        X_train_work,
        X_test_work
    ) = _validate_iterative_inputs(
        X_train,
        X_test
    )

    # --------------------------------------------------------
    # Default estimator
    # --------------------------------------------------------

    if estimator is None:

        estimator = BayesianRidge()

    # --------------------------------------------------------
    # Construct imputer
    # --------------------------------------------------------

    imputer = IterativeImputer(
        estimator=estimator,
        max_iter=int(max_iter),
        tol=float(tol),
        random_state=int(random_state),
        initial_strategy=initial_strategy,
        skip_complete=True,
        sample_posterior=False,
        min_value=None,
        max_value=None,
    )

    # --------------------------------------------------------
    # Fit ONLY on training data
    # --------------------------------------------------------

    X_train_imputed = (
        imputer
        .fit_transform(
            X_train_work
        )
    )

    # --------------------------------------------------------
    # Transform evaluation/test data
    # --------------------------------------------------------

    if X_test_work is not None:

        X_test_imputed = (
            imputer
            .transform(
                X_test_work
            )
        )

    else:

        X_test_imputed = None

    # --------------------------------------------------------
    # Validate output
    # --------------------------------------------------------

    if not np.isfinite(
        X_train_imputed
    ).all():

        raise RuntimeError(
            "Iterative imputation produced "
            "non-finite training values."
        )

    if (
        X_test_imputed is not None
        and
        not np.isfinite(
            X_test_imputed
        ).all()
    ):

        raise RuntimeError(
            "Iterative imputation produced "
            "non-finite test values."
        )

    return (
        X_train_imputed,
        X_test_imputed,
        imputer
    )


# ============================================================
# 4. MICE IMPUTER
# ============================================================

def fit_mice(
    X_train,
    X_test=None,
    random_state=42,
    max_iter=10,
    tol=1e-3
):
    """
    MICE-style stochastic chained-equation imputation.

    sample_posterior=True introduces stochastic posterior
    sampling and therefore makes the candidate distinct from
    deterministic IterativeImputer.

    The model is fitted only on X_train.
    """

    (
        X_train_work,
        X_test_work
    ) = _validate_iterative_inputs(
        X_train,
        X_test
    )

    # --------------------------------------------------------
    # Bayesian Ridge is used as the conditional estimator.
    # --------------------------------------------------------

    estimator = BayesianRidge()

    imputer = IterativeImputer(
        estimator=estimator,
        max_iter=int(max_iter),
        tol=float(tol),
        random_state=int(random_state),
        initial_strategy="median",
        skip_complete=True,
        sample_posterior=True,
    )

    # --------------------------------------------------------
    # Fit only on training data
    # --------------------------------------------------------

    X_train_imputed = (
        imputer
        .fit_transform(
            X_train_work
        )
    )

    # --------------------------------------------------------
    # Transform evaluation/test data
    # --------------------------------------------------------

    if X_test_work is not None:

        X_test_imputed = (
            imputer
            .transform(
                X_test_work
            )
        )

    else:

        X_test_imputed = None

    # --------------------------------------------------------
    # Output validation
    # --------------------------------------------------------

    if not np.isfinite(
        X_train_imputed
    ).all():

        raise RuntimeError(
            "MICE produced non-finite training values."
        )

    if (
        X_test_imputed is not None
        and
        not np.isfinite(
            X_test_imputed
        ).all()
    ):

        raise RuntimeError(
            "MICE produced non-finite test values."
        )

    return (
        X_train_imputed,
        X_test_imputed,
        imputer
    )


# ============================================================
# 5. STRATEGY REGISTRY
# ============================================================

ITERATIVE_IMPUTERS = {

    "iterativeimputer":
        fit_iterative,

    "mice":
        fit_mice,
}


# ============================================================
# 6. CONFIGURATION
# ============================================================

ITERATIVE_CONFIG = {

    "iterativeimputer": {

        "estimator":
            "BayesianRidge",

        "max_iter":
            10,

        "tol":
            1e-3,

        "sample_posterior":
            False,

        "initial_strategy":
            "median",

        "leakage_safe":
            True,
    },

    "mice": {

        "estimator":
            "BayesianRidge",

        "max_iter":
            10,

        "tol":
            1e-3,

        "sample_posterior":
            True,

        "initial_strategy":
            "median",

        "leakage_safe":
            True,
    },
}


# ============================================================
# 7. SANITY TEST DATA
# ============================================================

_test_train = pd.DataFrame({

    "x1": [
        1.0,
        2.0,
        np.nan,
        4.0,
        5.0,
        6.0
    ],

    "x2": [
        10.0,
        np.nan,
        30.0,
        40.0,
        np.nan,
        60.0
    ],

    "x3": [
        100.0,
        200.0,
        300.0,
        np.nan,
        500.0,
        600.0
    ],
})


_test_test = pd.DataFrame({

    "x1": [
        np.nan,
        8.0
    ],

    "x2": [
        70.0,
        np.nan
    ],

    "x3": [
        np.nan,
        800.0
    ],
})


# ============================================================
# 8. ITERATIVE SANITY TEST
# ============================================================

(
    _iter_train,
    _iter_test,
    _iter_model
) = fit_iterative(
    _test_train,
    _test_test,
    random_state=42,
    max_iter=10
)


if not np.isfinite(
    _iter_train
).all():

    raise RuntimeError(
        "Iterative imputer sanity test failed."
    )

if not np.isfinite(
    _iter_test
).all():

    raise RuntimeError(
        "Iterative test transformation failed."
    )


# ============================================================
# 9. MICE SANITY TEST
# ============================================================

(
    _mice_train,
    _mice_test,
    _mice_model
) = fit_mice(
    _test_train,
    _test_test,
    random_state=42,
    max_iter=10
)


if not np.isfinite(
    _mice_train
).all():

    raise RuntimeError(
        "MICE sanity test failed."
    )

if not np.isfinite(
    _mice_test
).all():

    raise RuntimeError(
        "MICE test transformation failed."
    )


# ============================================================
# 10. DIMENSION VALIDATION
# ============================================================

if (
    _iter_train.shape
    !=
    _test_train.shape
):

    raise RuntimeError(
        "Iterative imputer changed training dimensions."
    )

if (
    _iter_test.shape
    !=
    _test_test.shape
):

    raise RuntimeError(
        "Iterative imputer changed test dimensions."
    )

if (
    _mice_train.shape
    !=
    _test_train.shape
):

    raise RuntimeError(
        "MICE changed training dimensions."
    )

if (
    _mice_test.shape
    !=
    _test_test.shape
):

    raise RuntimeError(
        "MICE changed test dimensions."
    )


# ============================================================
# 11. FINAL VALIDATION
# ============================================================

print(
    "\n[06.12.01] Iterative Imputer : READY"
)

print(
    "[06.12.02] MICE candidate    : READY"
)

print(
    "[06.12.03] Bayesian Ridge    : ENABLED"
)

print(
    "[06.12.04] Posterior sampling: MICE ONLY"
)

print(
    "[06.12.05] Training-only fit : ENABLED"
)

print(
    "[06.12.06] Leakage protection: ENABLED"
)

print(
    "[06.12.07] Non-finite checks : ENABLED"
)

print(
    "\nRegistered iterative candidates:"
)

for strategy_id in sorted(
    ITERATIVE_IMPUTERS
):

    print(
        f"  ✓ {strategy_id}"
    )

print(
    "\nSanity tests: PASSED"
)

print(
    "\n" + "=" * 100
)

print(
    "ITERATIVE / MICE CANDIDATES : PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.12
LEAKAGE-SAFE ITERATIVE / MICE IMPUTATION CANDIDATES

[06.12.01] Iterative Imputer : READY
[06.12.02] MICE candidate    : READY
[06.12.03] Bayesian Ridge    : ENABLED
[06.12.04] Posterior sampling: MICE ONLY
[06.12.05] Training-only fit : ENABLED
[06.12.06] Leakage protection: ENABLED
[06.12.07] Non-finite checks : ENABLED

Registered iterative candidates:
  ✓ iterativeimputer
  ✓ mice

Sanity tests: PASSED

ITERATIVE / MICE CANDIDATES : PASSED


In [13]:
# ============================================================
# NOTEBOOK 06.13 — RANDOM FOREST IMPUTATION CANDIDATE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.13")
print("RANDOM FOREST IMPUTATION CANDIDATE")
print("=" * 100)


# ------------------------------------------------------------
# 1. Imports
# ------------------------------------------------------------

from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier
)

from sklearn.impute import SimpleImputer


# ------------------------------------------------------------
# 2. Random Forest imputation
# ------------------------------------------------------------

def fit_random_forest(
    X_train,
    target_train,
    X_missing=None,
    feature_type="numerical",
    random_state=42,
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=2,
    n_jobs=-1
):
    """
    Leakage-safe Random Forest imputation.

    Parameters
    ----------
    X_train : pandas.DataFrame
        Predictor matrix corresponding to target_train.

    target_train : pandas.Series
        Target/feature containing missing values.

    X_missing : pandas.DataFrame, optional
        Predictor matrix used for predicting missing target values.
        If None, X_train is used.

    feature_type : str
        'numerical' or 'categorical'.

    random_state : int
        Reproducibility seed.

    n_estimators : int
        Number of trees.

    max_depth : int or None
        Maximum tree depth.

    min_samples_leaf : int
        Minimum observations per leaf.

    n_jobs : int
        Parallel CPU workers.

    Returns
    -------
    result : pandas.Series
        Imputed target.

    model : fitted RandomForest model
    """

    # --------------------------------------------------------
    # Validate inputs
    # --------------------------------------------------------

    if not isinstance(
        X_train,
        pd.DataFrame
    ):
        raise TypeError(
            "X_train must be a pandas DataFrame."
        )

    if not isinstance(
        target_train,
        pd.Series
    ):
        raise TypeError(
            "target_train must be a pandas Series."
        )

    if len(X_train) != len(target_train):
        raise ValueError(
            "X_train and target_train must have "
            "the same number of rows."
        )

    if X_missing is None:
        X_missing = X_train.copy()

    if not isinstance(
        X_missing,
        pd.DataFrame
    ):
        raise TypeError(
            "X_missing must be a pandas DataFrame."
        )

    if len(X_missing) != len(target_train):
        raise ValueError(
            "X_missing and target_train must have "
            "the same number of rows."
        )

    # --------------------------------------------------------
    # Normalize feature type
    # --------------------------------------------------------

    feature_type = str(
        feature_type
    ).strip().lower()

    if feature_type in {
        "numeric",
        "numerical",
        "float",
        "integer",
        "int"
    }:
        feature_type = "numerical"

    elif feature_type in {
        "categorical",
        "category",
        "object",
        "string",
        "str",
        "boolean",
        "bool"
    }:
        feature_type = "categorical"

    else:
        raise ValueError(
            f"Unsupported feature_type: {feature_type}"
        )

    # --------------------------------------------------------
    # Identify missing values
    # --------------------------------------------------------

    missing_mask = target_train.isna()

    observed_mask = target_train.notna()

    n_observed = int(
        observed_mask.sum()
    )

    n_missing = int(
        missing_mask.sum()
    )

    if n_missing == 0:

        return (
            target_train.copy(),
            None
        )

    if n_observed < 10:

        raise ValueError(
            "Insufficient observed target values "
            f"for Random Forest imputation: {n_observed}"
        )

    # --------------------------------------------------------
    # Align predictor matrices
    # --------------------------------------------------------

    if not X_train.index.equals(
        target_train.index
    ):

        X_train = X_train.reindex(
            target_train.index
        )

    if not X_missing.index.equals(
        target_train.index
    ):

        X_missing = X_missing.reindex(
            target_train.index
        )

    # --------------------------------------------------------
    # Convert predictors to numeric matrix
    #
    # The predictor encoding pipeline should normally already
    # perform this step. This additional safeguard prevents
    # object/string columns from reaching sklearn.
    # --------------------------------------------------------

    if not all(
        pd.api.types.is_numeric_dtype(
            X_train[column]
        )
        for column in X_train.columns
    ):

        encoder_imputer = SimpleImputer(
            strategy="median"
        )

        X_train_numeric = pd.DataFrame(
            encoder_imputer.fit_transform(
                X_train.apply(
                    pd.to_numeric,
                    errors="coerce"
                )
            ),
            index=X_train.index
        )

        X_missing_numeric = pd.DataFrame(
            encoder_imputer.transform(
                X_missing.apply(
                    pd.to_numeric,
                    errors="coerce"
                )
            ),
            index=X_missing.index
        )

    else:

        X_train_numeric = X_train.copy()
        X_missing_numeric = X_missing.copy()

        X_train_numeric = (
            X_train_numeric
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
        )

        X_missing_numeric = (
            X_missing_numeric
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
        )

        predictor_imputer = SimpleImputer(
            strategy="median"
        )

        X_train_numeric = pd.DataFrame(
            predictor_imputer.fit_transform(
                X_train_numeric
            ),
            index=X_train.index
        )

        X_missing_numeric = pd.DataFrame(
            predictor_imputer.transform(
                X_missing_numeric
            ),
            index=X_missing.index
        )

    # --------------------------------------------------------
    # Prepare target
    # --------------------------------------------------------

    y_observed = (
        target_train.loc[
            observed_mask
        ]
        .copy()
    )

    # --------------------------------------------------------
    # Select model according to feature type
    # --------------------------------------------------------

    if feature_type == "numerical":

        y_observed = pd.to_numeric(
            y_observed,
            errors="coerce"
        )

        valid_target = (
            y_observed.notna()
            &
            np.isfinite(
                y_observed.to_numpy(
                    dtype=float
                )
            )
        )

        y_observed = (
            y_observed.loc[
                valid_target
            ]
        )

        if len(y_observed) < 10:

            raise ValueError(
                "Insufficient valid numerical "
                "target observations."
            )

        model = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=random_state,
            n_jobs=n_jobs
        )

    else:

        y_observed = (
            y_observed
            .astype(str)
        )

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=random_state,
            n_jobs=n_jobs
        )

    # --------------------------------------------------------
    # Align training predictors with valid target rows
    # --------------------------------------------------------

    X_fit = X_train_numeric.loc[
        y_observed.index
    ]

    # --------------------------------------------------------
    # Fit model
    # --------------------------------------------------------

    model.fit(
        X_fit,
        y_observed
    )

    # --------------------------------------------------------
    # Predict missing values
    # --------------------------------------------------------

    X_predict = (
        X_missing_numeric.loc[
            missing_mask
        ]
    )

    predictions = model.predict(
        X_predict
    )

    # --------------------------------------------------------
    # Construct imputed result
    # --------------------------------------------------------

    result = target_train.copy()

    if feature_type == "numerical":

        result.loc[
            missing_mask
        ] = predictions

    else:

        result.loc[
            missing_mask
        ] = predictions.astype(
            target_train.dtype
            if target_train.dtype != "object"
            else object
        )

    # --------------------------------------------------------
    # Final validation
    # --------------------------------------------------------

    if result.isna().any():

        remaining_missing = int(
            result.isna().sum()
        )

        raise RuntimeError(
            "Random Forest imputation failed to "
            f"resolve {remaining_missing} missing values."
        )

    return (
        result,
        model
    )


# ------------------------------------------------------------
# 3. Final readiness checks
# ------------------------------------------------------------

assert callable(
    fit_random_forest
)

print(
    "Random Forest candidate : READY"
)

print(
    "Numerical target support : ENABLED"
)

print(
    "Categorical target support : ENABLED"
)

print(
    "Leakage-safe fitting : ENABLED"
)

print(
    "Predictor alignment validation : ENABLED"
)

print(
    "Predictor missing-value handling : ENABLED"
)

print(
    "Reproducibility : ENABLED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.13
RANDOM FOREST IMPUTATION CANDIDATE
Random Forest candidate : READY
Numerical target support : ENABLED
Categorical target support : ENABLED
Leakage-safe fitting : ENABLED
Predictor alignment validation : ENABLED
Predictor missing-value handling : ENABLED
Reproducibility : ENABLED


In [14]:
# ============================================================
# NOTEBOOK 06.14 — GRADIENT BOOSTING CANDIDATE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.14")
print("GRADIENT BOOSTING IMPUTATION CANDIDATE")
print("=" * 100)


# ------------------------------------------------------------
# 1. Imports
# ------------------------------------------------------------

from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    HistGradientBoostingClassifier
)

from sklearn.impute import SimpleImputer


# ------------------------------------------------------------
# 2. Gradient Boosting imputation
# ------------------------------------------------------------

def fit_gradient_boosting(
    X_train,
    target_train,
    X_missing=None,
    feature_type="numerical",
    random_state=42,
    max_iter=200,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=20
):
    """
    Leakage-safe Gradient Boosting imputation.

    Parameters
    ----------
    X_train : pandas.DataFrame
        Predictor matrix corresponding to target_train.

    target_train : pandas.Series
        Target/feature containing missing values.

    X_missing : pandas.DataFrame, optional
        Predictor matrix used for predicting missing target values.

    feature_type : str
        'numerical' or 'categorical'.

    random_state : int
        Reproducibility seed.

    max_iter : int
        Maximum boosting iterations.

    learning_rate : float
        Boosting learning rate.

    max_leaf_nodes : int
        Maximum number of leaves per tree.

    min_samples_leaf : int
        Minimum samples per leaf.

    Returns
    -------
    result : pandas.Series
        Imputed target.

    model : fitted Gradient Boosting model
    """

    # --------------------------------------------------------
    # Validate inputs
    # --------------------------------------------------------

    if not isinstance(
        X_train,
        pd.DataFrame
    ):
        raise TypeError(
            "X_train must be a pandas DataFrame."
        )

    if not isinstance(
        target_train,
        pd.Series
    ):
        raise TypeError(
            "target_train must be a pandas Series."
        )

    if len(X_train) != len(target_train):
        raise ValueError(
            "X_train and target_train must have "
            "the same number of rows."
        )

    if X_missing is None:
        X_missing = X_train.copy()

    if not isinstance(
        X_missing,
        pd.DataFrame
    ):
        raise TypeError(
            "X_missing must be a pandas DataFrame."
        )

    if len(X_missing) != len(target_train):
        raise ValueError(
            "X_missing and target_train must have "
            "the same number of rows."
        )

    # --------------------------------------------------------
    # Normalize feature type
    # --------------------------------------------------------

    feature_type = str(
        feature_type
    ).strip().lower()

    if feature_type in {
        "numeric",
        "numerical",
        "float",
        "integer",
        "int"
    }:
        feature_type = "numerical"

    elif feature_type in {
        "categorical",
        "category",
        "object",
        "string",
        "str",
        "boolean",
        "bool"
    }:
        feature_type = "categorical"

    else:
        raise ValueError(
            f"Unsupported feature_type: {feature_type}"
        )

    # --------------------------------------------------------
    # Align indexes
    # --------------------------------------------------------

    if not X_train.index.equals(
        target_train.index
    ):

        X_train = X_train.reindex(
            target_train.index
        )

    if not X_missing.index.equals(
        target_train.index
    ):

        X_missing = X_missing.reindex(
            target_train.index
        )

    # --------------------------------------------------------
    # Missingness masks
    # --------------------------------------------------------

    observed_mask = (
        target_train.notna()
    )

    missing_mask = (
        target_train.isna()
    )

    n_observed = int(
        observed_mask.sum()
    )

    n_missing = int(
        missing_mask.sum()
    )

    # Nothing to impute
    if n_missing == 0:

        return (
            target_train.copy(),
            None
        )

    if n_observed < 10:

        raise ValueError(
            "Insufficient observed target values "
            f"for Gradient Boosting: {n_observed}"
        )

    # --------------------------------------------------------
    # Prepare predictors
    # --------------------------------------------------------
    #
    # The normal AIR-LLM pipeline should already encode
    # predictors before reaching this function.
    #
    # This safeguard ensures that the candidate does not fail
    # because of residual NaN / infinite predictor values.
    # --------------------------------------------------------

    X_train_numeric = (
        X_train
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
    )

    X_missing_numeric = (
        X_missing
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
    )

    # Convert non-numeric predictors when necessary
    if not all(
        pd.api.types.is_numeric_dtype(
            X_train_numeric[column]
        )
        for column in X_train_numeric.columns
    ):

        X_train_numeric = (
            X_train_numeric.apply(
                pd.to_numeric,
                errors="coerce"
            )
        )

        X_missing_numeric = (
            X_missing_numeric.apply(
                pd.to_numeric,
                errors="coerce"
            )
        )

    # Median predictor imputation
    predictor_imputer = SimpleImputer(
        strategy="median"
    )

    X_train_numeric = pd.DataFrame(
        predictor_imputer.fit_transform(
            X_train_numeric
        ),
        index=X_train.index
    )

    X_missing_numeric = pd.DataFrame(
        predictor_imputer.transform(
            X_missing_numeric
        ),
        index=X_missing.index
    )

    # --------------------------------------------------------
    # Prepare observed target
    # --------------------------------------------------------

    y_observed = (
        target_train
        .loc[observed_mask]
        .copy()
    )

    # --------------------------------------------------------
    # Build appropriate model
    # --------------------------------------------------------

    if feature_type == "numerical":

        y_observed = pd.to_numeric(
            y_observed,
            errors="coerce"
        )

        valid_target = (
            y_observed.notna()
            &
            np.isfinite(
                y_observed.to_numpy(
                    dtype=float
                )
            )
        )

        y_observed = (
            y_observed
            .loc[valid_target]
        )

        if len(y_observed) < 10:

            raise ValueError(
                "Insufficient valid numerical "
                "target observations."
            )

        model = HistGradientBoostingRegressor(
            max_iter=max_iter,
            learning_rate=learning_rate,
            max_leaf_nodes=max_leaf_nodes,
            min_samples_leaf=min_samples_leaf,
            random_state=random_state
        )

    else:

        # ----------------------------------------------------
        # Categorical target
        #
        # HistGradientBoostingClassifier requires encoded
        # class labels.
        # ----------------------------------------------------

        y_observed = (
            y_observed
            .astype(str)
        )

        classes = (
            pd.Index(
                y_observed
                .unique()
            )
            .sort_values()
        )

        if len(classes) < 2:

            raise ValueError(
                "Categorical target must contain "
                "at least two observed classes."
            )

        class_to_code = {
            value: index
            for index, value
            in enumerate(classes)
        }

        y_encoded = (
            y_observed
            .map(class_to_code)
            .astype(int)
        )

        model = HistGradientBoostingClassifier(
            max_iter=max_iter,
            learning_rate=learning_rate,
            max_leaf_nodes=max_leaf_nodes,
            min_samples_leaf=min_samples_leaf,
            random_state=random_state
        )

    # --------------------------------------------------------
    # Align predictors with valid target observations
    # --------------------------------------------------------

    X_fit = (
        X_train_numeric
        .loc[
            y_observed.index
        ]
    )

    # --------------------------------------------------------
    # Fit model
    # --------------------------------------------------------

    if feature_type == "numerical":

        model.fit(
            X_fit,
            y_observed
        )

    else:

        model.fit(
            X_fit,
            y_encoded
        )

    # --------------------------------------------------------
    # Predict missing values
    # --------------------------------------------------------

    X_predict = (
        X_missing_numeric
        .loc[
            missing_mask
        ]
    )

    predictions = model.predict(
        X_predict
    )

    # --------------------------------------------------------
    # Construct final imputed series
    # --------------------------------------------------------

    result = target_train.copy()

    if feature_type == "numerical":

        result.loc[
            missing_mask
        ] = predictions

    else:

        inverse_mapping = {
            code: value
            for value, code
            in class_to_code.items()
        }

        decoded_predictions = [
            inverse_mapping[int(x)]
            for x in predictions
        ]

        result.loc[
            missing_mask
        ] = decoded_predictions

    # --------------------------------------------------------
    # Final validation
    # --------------------------------------------------------

    if result.isna().any():

        remaining_missing = int(
            result.isna().sum()
        )

        raise RuntimeError(
            "Gradient Boosting imputation failed "
            f"to resolve {remaining_missing} missing values."
        )

    return (
        result,
        model
    )


# ------------------------------------------------------------
# 3. Final readiness validation
# ------------------------------------------------------------

assert callable(
    fit_gradient_boosting
)

print(
    "Gradient Boosting candidate : READY"
)

print(
    "Numerical target support : ENABLED"
)

print(
    "Categorical target support : ENABLED"
)

print(
    "Leakage-safe fitting : ENABLED"
)

print(
    "Predictor alignment validation : ENABLED"
)

print(
    "Predictor missing-value handling : ENABLED"
)

print(
    "Reproducibility : ENABLED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.14
GRADIENT BOOSTING IMPUTATION CANDIDATE
Gradient Boosting candidate : READY
Numerical target support : ENABLED
Categorical target support : ENABLED
Leakage-safe fitting : ENABLED
Predictor alignment validation : ENABLED
Predictor missing-value handling : ENABLED
Reproducibility : ENABLED


In [15]:
# ============================================================
# NOTEBOOK 06.15 — MISSFOREST CANDIDATE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.15")
print("MISSFOREST IMPUTATION CANDIDATE")
print("=" * 100)


# ------------------------------------------------------------
# 1. Imports
# ------------------------------------------------------------

from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier
)

from sklearn.impute import SimpleImputer


# ------------------------------------------------------------
# 2. Mixed-type MissForest implementation
# ------------------------------------------------------------

def fit_missforest(
    X_train,
    X_test=None,
    random_state=42,
    max_iter=5,
    n_estimators=100,
    min_samples_leaf=2
):
    """
    Mixed-type MissForest-style iterative imputation.

    Numerical variables are modeled with RandomForestRegressor.

    Categorical variables are modeled with
    RandomForestClassifier.

    Parameters
    ----------
    X_train : pandas.DataFrame
        Training data containing missing values.

    X_test : pandas.DataFrame, optional
        Test data containing missing values.

    random_state : int
        Reproducibility seed.

    max_iter : int
        Maximum iterative imputation passes.

    n_estimators : int
        Number of Random Forest trees.

    min_samples_leaf : int
        Minimum observations per leaf.

    Returns
    -------
    X_train_imputed : pandas.DataFrame
        Imputed training data.

    X_test_imputed : pandas.DataFrame or None
        Imputed test data.

    state : dict
        Fitted MissForest state.
    """

    # --------------------------------------------------------
    # Validate input
    # --------------------------------------------------------

    if not isinstance(
        X_train,
        pd.DataFrame
    ):
        raise TypeError(
            "X_train must be a pandas DataFrame."
        )

    if X_train.empty:
        raise ValueError(
            "X_train is empty."
        )

    if X_test is not None:

        if not isinstance(
            X_test,
            pd.DataFrame
        ):
            raise TypeError(
                "X_test must be a pandas DataFrame."
            )

        if list(
            X_test.columns
        ) != list(
            X_train.columns
        ):
            raise ValueError(
                "X_train and X_test must have "
                "identical columns."
            )

    # --------------------------------------------------------
    # Work on copies
    # --------------------------------------------------------

    train = X_train.copy()

    test = (
        X_test.copy()
        if X_test is not None
        else None
    )

    # --------------------------------------------------------
    # Identify feature types
    # --------------------------------------------------------

    numerical_columns = (
        train
        .select_dtypes(
            include=[np.number]
        )
        .columns
        .tolist()
    )

    categorical_columns = [
        column
        for column in train.columns
        if column not in numerical_columns
    ]

    if not numerical_columns and not categorical_columns:

        raise ValueError(
            "No usable predictor columns found."
        )

    # --------------------------------------------------------
    # Preserve original dtypes
    # --------------------------------------------------------

    original_dtypes = {
        column: train[column].dtype
        for column in train.columns
    }

    # --------------------------------------------------------
    # Convert categorical variables to integer codes
    #
    # Categories are learned from TRAINING DATA only.
    # --------------------------------------------------------

    category_maps = {}

    for column in categorical_columns:

        train_values = (
            train[column]
            .astype("string")
        )

        categories = (
            train_values
            .dropna()
            .unique()
            .tolist()
        )

        category_maps[column] = {
            value: index
            for index, value
            in enumerate(categories)
        }

        train[column] = (
            train_values
            .map(
                category_maps[column]
            )
            .astype(float)
        )

        if test is not None:

            test_values = (
                test[column]
                .astype("string")
            )

            test[column] = (
                test_values
                .map(
                    category_maps[column]
                )
                .astype(float)
            )

    # --------------------------------------------------------
    # Convert numerical variables to numeric
    # --------------------------------------------------------

    for column in numerical_columns:

        train[column] = pd.to_numeric(
            train[column],
            errors="coerce"
        )

        if test is not None:

            test[column] = pd.to_numeric(
                test[column],
                errors="coerce"
            )

    # --------------------------------------------------------
    # Replace infinite values
    # --------------------------------------------------------

    train = train.replace(
        [np.inf, -np.inf],
        np.nan
    )

    if test is not None:

        test = test.replace(
            [np.inf, -np.inf],
            np.nan
        )

    # --------------------------------------------------------
    # Initial imputation
    # --------------------------------------------------------

    initial_imputer = SimpleImputer(
        strategy="median"
    )

    train_initial = pd.DataFrame(
        initial_imputer.fit_transform(
            train
        ),
        columns=train.columns,
        index=train.index
    )

    if test is not None:

        test_initial = pd.DataFrame(
            initial_imputer.transform(
                test
            ),
            columns=test.columns,
            index=test.index
        )

    else:

        test_initial = None

    # --------------------------------------------------------
    # Original missingness masks
    # --------------------------------------------------------

    train_missing_masks = {
        column:
        train[column].isna()
        for column in train.columns
    }

    if test is not None:

        test_missing_masks = {
            column:
            test[column].isna()
            for column in test.columns
        }

    # --------------------------------------------------------
    # Iterative MissForest process
    # --------------------------------------------------------

    current_train = train_initial.copy()

    current_test = (
        test_initial.copy()
        if test_initial is not None
        else None
    )

    feature_types = {
        column:
        (
            "numerical"
            if column in numerical_columns
            else "categorical"
        )
        for column in train.columns
    }

    fitted_models = {}

    for iteration in range(
        max_iter
    ):

        previous_train = (
            current_train.copy()
        )

        # ----------------------------------------------------
        # Process each feature with missing values
        # ----------------------------------------------------

        for target_column in train.columns:

            missing_mask = (
                train_missing_masks[
                    target_column
                ]
            )

            if not missing_mask.any():
                continue

            predictor_columns = [
                column
                for column in train.columns
                if column != target_column
            ]

            if not predictor_columns:
                continue

            observed_mask = (
                ~missing_mask
            )

            X_fit = (
                current_train
                .loc[
                    observed_mask,
                    predictor_columns
                ]
            )

            y_fit = (
                current_train
                .loc[
                    observed_mask,
                    target_column
                ]
            )

            X_predict = (
                current_train
                .loc[
                    missing_mask,
                    predictor_columns
                ]
            )

            if len(X_fit) < 10:
                continue

            feature_type = (
                feature_types[
                    target_column
                ]
            )

            # ------------------------------------------------
            # Numerical target
            # ------------------------------------------------

            if feature_type == "numerical":

                model = RandomForestRegressor(
                    n_estimators=n_estimators,
                    random_state=(
                        random_state
                        + iteration
                    ),
                    n_jobs=-1,
                    min_samples_leaf=min_samples_leaf
                )

                model.fit(
                    X_fit,
                    y_fit
                )

                predictions = (
                    model.predict(
                        X_predict
                    )
                )

            # ------------------------------------------------
            # Categorical target
            # ------------------------------------------------

            else:

                y_fit = (
                    y_fit
                    .round()
                    .astype(int)
                )

                model = RandomForestClassifier(
                    n_estimators=n_estimators,
                    random_state=(
                        random_state
                        + iteration
                    ),
                    n_jobs=-1,
                    min_samples_leaf=min_samples_leaf
                )

                model.fit(
                    X_fit,
                    y_fit
                )

                predictions = (
                    model.predict(
                        X_predict
                    )
                )

            current_train.loc[
                missing_mask,
                target_column
            ] = predictions

            fitted_models[
                target_column
            ] = model

        # ----------------------------------------------------
        # Convergence check
        # ----------------------------------------------------

        difference = np.nanmean(
            np.abs(
                current_train.to_numpy(
                    dtype=float
                )
                -
                previous_train.to_numpy(
                    dtype=float
                )
            )
        )

        if not np.isfinite(
            difference
        ):
            break

        if difference < 1e-4:
            break

    # --------------------------------------------------------
    # Apply fitted models to X_test
    # --------------------------------------------------------

    if current_test is not None:

        for target_column in test.columns:

            missing_mask = (
                test_missing_masks[
                    target_column
                ]
            )

            if not missing_mask.any():
                continue

            model = fitted_models.get(
                target_column
            )

            if model is None:
                continue

            predictor_columns = [
                column
                for column in test.columns
                if column != target_column
            ]

            X_predict = (
                current_test
                .loc[
                    missing_mask,
                    predictor_columns
                ]
            )

            predictions = model.predict(
                X_predict
            )

            current_test.loc[
                missing_mask,
                target_column
            ] = predictions

    # --------------------------------------------------------
    # Decode categorical variables
    # --------------------------------------------------------

    reverse_category_maps = {

        column: {
            code: value
            for value, code
            in mapping.items()
        }

        for column, mapping
        in category_maps.items()
    }

    for column in categorical_columns:

        reverse_map = (
            reverse_category_maps[
                column
            ]
        )

        current_train[column] = (
            current_train[column]
            .round()
            .map(reverse_map)
        )

        if current_test is not None:

            current_test[column] = (
                current_test[column]
                .round()
                .map(reverse_map)
            )

    # --------------------------------------------------------
    # Restore numerical dtypes where possible
    # --------------------------------------------------------

    for column in numerical_columns:

        try:
            current_train[column] = (
                current_train[column]
                .astype(
                    original_dtypes[
                        column
                    ]
                )
            )
        except Exception:
            pass

        if current_test is not None:

            try:
                current_test[column] = (
                    current_test[column]
                    .astype(
                        original_dtypes[
                            column
                        ]
                    )
                )
            except Exception:
                pass

    # --------------------------------------------------------
    # Final validation
    # --------------------------------------------------------

    if current_train.isna().any().any():

        remaining = int(
            current_train.isna()
            .sum()
            .sum()
        )

        raise RuntimeError(
            "MissForest failed to resolve "
            f"{remaining} training missing values."
        )

    if (
        current_test is not None
        and current_test.isna().any().any()
    ):

        remaining = int(
            current_test.isna()
            .sum()
            .sum()
        )

        raise RuntimeError(
            "MissForest failed to resolve "
            f"{remaining} test missing values."
        )

    state = {
        "models": fitted_models,
        "feature_types": feature_types,
        "category_maps": category_maps,
        "iterations": max_iter,
        "random_state": random_state,
    }

    return (
        current_train,
        current_test,
        state
    )


# ------------------------------------------------------------
# 3. Final readiness validation
# ------------------------------------------------------------

assert callable(
    fit_missforest
)

print(
    "MissForest candidate : READY"
)

print(
    "Mixed numerical/categorical support : ENABLED"
)

print(
    "Iterative Random Forest refinement : ENABLED"
)

print(
    "Categorical Random Forest classification : ENABLED"
)

print(
    "Training-only category mapping : ENABLED"
)

print(
    "Convergence monitoring : ENABLED"
)

print(
    "Leakage-safe fitting : ENABLED"
)

print(
    "Reproducibility : ENABLED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.15
MISSFOREST IMPUTATION CANDIDATE
MissForest candidate : READY
Mixed numerical/categorical support : ENABLED
Iterative Random Forest refinement : ENABLED
Categorical Random Forest classification : ENABLED
Training-only category mapping : ENABLED
Convergence monitoring : ENABLED
Leakage-safe fitting : ENABLED
Reproducibility : ENABLED


In [16]:
# ============================================================
# NOTEBOOK 06.16 — SOFTIMPUTE / MATRIX-FACTORIZATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.16")
print("SOFTIMPUTE / MATRIX-FACTORIZATION CANDIDATES")
print("=" * 100)


# ------------------------------------------------------------
# 1. Imports
# ------------------------------------------------------------

import numpy as np


# ------------------------------------------------------------
# 2. SoftImpute implementation
# ------------------------------------------------------------

def fit_softimpute(
    X_train,
    X_test=None,
    rank=10,
    max_iter=50,
    tolerance=1e-4,
    shrinkage=0.0
):

    X_train_array = np.asarray(
        X_train,
        dtype=float
    )

    if X_train_array.ndim != 2:
        raise ValueError(
            "X_train must be a 2-dimensional numeric matrix."
        )

    if rank < 1:
        raise ValueError(
            "rank must be >= 1."
        )

    if max_iter < 1:
        raise ValueError(
            "max_iter must be >= 1."
        )

    if tolerance <= 0:
        raise ValueError(
            "tolerance must be > 0."
        )

    if not np.isfinite(
        np.nanmean(X_train_array, axis=0)
    ).all():

        # handled below column-by-column
        pass


    # --------------------------------------------------------
    # Initial column-mean imputation
    # --------------------------------------------------------

    column_means = np.nanmean(
        X_train_array,
        axis=0
    )

    column_means = np.where(
        np.isfinite(column_means),
        column_means,
        0.0
    )


    missing_mask = np.isnan(
        X_train_array
    )

    filled = np.where(
        missing_mask,
        column_means,
        X_train_array
    )


    # --------------------------------------------------------
    # Iterative low-rank reconstruction
    # --------------------------------------------------------

    converged = False
    iterations_completed = 0
    final_change = np.inf

    for iteration in range(
        max_iter
    ):

        previous_missing = (
            filled[missing_mask].copy()
        )


        # ----------------------------------------------------
        # SVD
        # ----------------------------------------------------

        try:

            U, S, VT = np.linalg.svd(
                filled,
                full_matrices=False
            )

        except np.linalg.LinAlgError as exc:

            raise RuntimeError(
                "SVD failed during SoftImpute."
            ) from exc


        effective_rank = min(
            int(rank),
            len(S)
        )


        if effective_rank == 0:

            break


        # ----------------------------------------------------
        # Optional singular-value shrinkage
        # ----------------------------------------------------

        retained_singular_values = (
            S[:effective_rank]
            - float(shrinkage)
        )

        retained_singular_values = np.maximum(
            retained_singular_values,
            0.0
        )


        reconstructed = (
            U[:, :effective_rank]
            *
            retained_singular_values
        ) @ VT[:effective_rank, :]


        # ----------------------------------------------------
        # Update ONLY originally missing values
        # ----------------------------------------------------

        filled[missing_mask] = (
            reconstructed[missing_mask]
        )


        # ----------------------------------------------------
        # Convergence based on missing entries
        # ----------------------------------------------------

        current_missing = (
            filled[missing_mask]
        )

        if len(previous_missing) > 0:

            denominator = (
                np.linalg.norm(
                    previous_missing
                )
                + 1e-12
            )

            final_change = (
                np.linalg.norm(
                    current_missing
                    -
                    previous_missing
                )
                /
                denominator
            )

        else:

            final_change = 0.0


        iterations_completed = (
            iteration + 1
        )


        if final_change < tolerance:

            converged = True
            break


    # --------------------------------------------------------
    # Transform test data using the learned low-rank model
    # --------------------------------------------------------

    X_test_imputed = None

    if X_test is not None:

        X_test_array = np.asarray(
            X_test,
            dtype=float
        )

        if X_test_array.ndim != 2:
            raise ValueError(
                "X_test must be a 2-dimensional numeric matrix."
            )

        if X_test_array.shape[1] != (
            X_train_array.shape[1]
        ):

            raise ValueError(
                "X_test and X_train must have "
                "the same number of columns."
            )


        test_missing_mask = np.isnan(
            X_test_array
        )


        # ----------------------------------------------------
        # Initialize test missing values using training means
        # ----------------------------------------------------

        test_filled = np.where(
            test_missing_mask,
            column_means,
            X_test_array
        )


        # ----------------------------------------------------
        # Reconstruct test matrix using learned
        # low-rank structure
        # ----------------------------------------------------

        U_test, S_test, VT_test = np.linalg.svd(
            test_filled,
            full_matrices=False
        )

        test_rank = min(
            effective_rank,
            len(S_test)
        )


        if test_rank > 0:

            test_singular_values = (
                S_test[:test_rank]
                -
                float(shrinkage)
            )

            test_singular_values = np.maximum(
                test_singular_values,
                0.0
            )

            test_reconstructed = (
                U_test[:, :test_rank]
                *
                test_singular_values
            ) @ VT_test[:test_rank, :]

            test_filled[
                test_missing_mask
            ] = test_reconstructed[
                test_missing_mask
            ]


        X_test_imputed = test_filled


    # --------------------------------------------------------
    # Return model metadata
    # --------------------------------------------------------

    model_metadata = {

        "method":
            "SoftImpute",

        "rank":
            int(effective_rank),

        "max_iter":
            int(max_iter),

        "iterations":
            int(iterations_completed),

        "tolerance":
            float(tolerance),

        "final_change":
            float(final_change),

        "converged":
            bool(converged),

        "shrinkage":
            float(shrinkage),

    }


    return (
        filled,
        X_test_imputed,
        model_metadata
    )


# ------------------------------------------------------------
# 3. Matrix-factorization candidate
# ------------------------------------------------------------

def fit_matrix_factorization(
    X_train,
    X_test=None,
    rank=10,
    max_iter=50,
    tolerance=1e-4
):

    return fit_softimpute(
        X_train=X_train,
        X_test=X_test,
        rank=rank,
        max_iter=max_iter,
        tolerance=tolerance,
        shrinkage=0.0
    )


# ------------------------------------------------------------
# 4. Function validation
# ------------------------------------------------------------

required_functions = [
    "fit_softimpute",
    "fit_matrix_factorization",
]


missing_functions = [
    name
    for name in required_functions
    if name not in globals()
]


if missing_functions:

    raise RuntimeError(
        "Missing required functions:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing_functions
        )
    )


# ------------------------------------------------------------
# 5. Final status
# ------------------------------------------------------------

print(
    "SoftImpute candidate: READY"
)

print(
    "Matrix-factorization candidate: READY"
)

print(
    "Low-rank reconstruction: ENABLED"
)

print(
    "Missing-entry-only convergence: ENABLED"
)

print(
    "Train/test dimensionality validation: ENABLED"
)

print(
    "Deterministic implementation: ENABLED"
)

print("\n" + "=" * 100)
print(
    "SOFTIMPUTE / MATRIX-FACTORIZATION : PASSED"
)
print("=" * 100)

AIR-LLM — NOTEBOOK 06.16
SOFTIMPUTE / MATRIX-FACTORIZATION CANDIDATES
SoftImpute candidate: READY
Matrix-factorization candidate: READY
Low-rank reconstruction: ENABLED
Missing-entry-only convergence: ENABLED
Train/test dimensionality validation: ENABLED
Deterministic implementation: ENABLED

SOFTIMPUTE / MATRIX-FACTORIZATION : PASSED


In [17]:
# ============================================================
# NOTEBOOK 06.16A — CANDIDATE STRATEGY IMPLEMENTATION REGISTRY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.16A")
print("CANDIDATE STRATEGY IMPLEMENTATION REGISTRY")
print("=" * 100)


# ------------------------------------------------------------
# 06.16A.01 — Required implementations
# ------------------------------------------------------------

required_functions = {
    "mean": "impute_mean",
    "median": "impute_median",
    "mode": "impute_mode",
    "constant": "impute_constant",
    "random_sample": "impute_random_sample",
    "knn": "fit_knn",
    "iterativeimputer": "fit_iterative",
    "missforest": "fit_missforest",
    "randomforest": "fit_random_forest",
}


# ------------------------------------------------------------
# 06.16A.02 — Validate functions
# ------------------------------------------------------------

missing_functions = []

for strategy_id, function_name in (
    required_functions.items()
):

    if function_name not in globals():

        missing_functions.append(
            f"{strategy_id} -> {function_name}"
        )


if missing_functions:

    raise RuntimeError(
        "Required strategy functions are missing:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_functions
        )
        +
        "\n\nRun Notebook 06.10–06.15 first."
    )


# ------------------------------------------------------------
# 06.16A.03 — Build authoritative executable registry
# ------------------------------------------------------------

STRATEGY_FUNCTIONS = {

    strategy_id:
        globals()[function_name]

    for strategy_id, function_name
    in required_functions.items()
}


# ------------------------------------------------------------
# 06.16A.04 — Validate callables
# ------------------------------------------------------------

invalid_strategies = [

    strategy_id

    for strategy_id, function
    in STRATEGY_FUNCTIONS.items()

    if not callable(function)

]


if invalid_strategies:

    raise RuntimeError(
        "Non-callable strategy implementations:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in invalid_strategies
        )
    )


# ------------------------------------------------------------
# 06.16A.05 — Compatibility registry
# ------------------------------------------------------------

STRATEGY_FEATURE_COMPATIBILITY = {

    "mean":
        {"numerical"},

    "median":
        {"numerical"},

    "mode":
        {"categorical"},

    "constant":
        {
            "numerical",
            "categorical"
        },

    "random_sample":
        {
            "numerical",
            "categorical"
        },

    "knn":
        {"numerical"},

    "iterativeimputer":
        {"numerical"},

    "missforest":
        {"numerical"},

    "randomforest":
        {"numerical"},

}


# ------------------------------------------------------------
# 06.16A.06 — Final validation
# ------------------------------------------------------------

print("\nSTRATEGY IMPLEMENTATION REGISTRY")
print("-" * 100)

for strategy_id, function in (
    STRATEGY_FUNCTIONS.items()
):

    print(
        f"  ✓ {strategy_id:<20} "
        f"{function.__name__}"
    )


print("\nCompatibility:")
print("-" * 100)

for strategy_id, feature_types in (
    STRATEGY_FEATURE_COMPATIBILITY.items()
):

    print(
        f"  {strategy_id:<20} "
        f"{sorted(feature_types)}"
    )


print("\n" + "=" * 100)

print(
    f"Registered implementations : "
    f"{len(STRATEGY_FUNCTIONS)}"
)

print(
    f"Compatibility definitions   : "
    f"{len(STRATEGY_FEATURE_COMPATIBILITY)}"
)

print(
    "\nCandidate strategy registry: PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.16A
CANDIDATE STRATEGY IMPLEMENTATION REGISTRY

STRATEGY IMPLEMENTATION REGISTRY
----------------------------------------------------------------------------------------------------
  ✓ mean                 impute_mean
  ✓ median               impute_median
  ✓ mode                 impute_mode
  ✓ constant             impute_constant
  ✓ random_sample        impute_random_sample
  ✓ knn                  fit_knn
  ✓ iterativeimputer     fit_iterative
  ✓ missforest           fit_missforest
  ✓ randomforest         fit_random_forest

Compatibility:
----------------------------------------------------------------------------------------------------
  mean                 ['numerical']
  median               ['numerical']
  mode                 ['categorical']
  constant             ['categorical', 'numerical']
  random_sample        ['categorical', 'numerical']
  knn                  ['numerical']
  iterativeimputer     ['numerical']
  missforest           ['numerica

In [18]:
# ============================================================
# NOTEBOOK 06.17 — CANDIDATE STRATEGY COMPATIBILITY VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.17")
print("CANDIDATE STRATEGY COMPATIBILITY VALIDATION")
print("=" * 100)


# ------------------------------------------------------------
# 1. Imports
# ------------------------------------------------------------

import pandas as pd


# ------------------------------------------------------------
# 2. Required AIR-LLM state
# ------------------------------------------------------------

required_objects = [
    "EVALUATION_DATA",
    "TARGET_REGISTRY",
    "FEATURE_TYPE_DF",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:

    raise RuntimeError(
        "Required AIR-LLM state is missing:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing_objects
        )
        +
        "\n\n"
        "Run Notebook 06.02 → 06.06 first."
    )


if not isinstance(EVALUATION_DATA, dict):
    raise TypeError(
        "EVALUATION_DATA must be a dictionary."
    )


if not isinstance(FEATURE_TYPE_DF, pd.DataFrame):
    raise TypeError(
        "FEATURE_TYPE_DF must be a pandas DataFrame."
    )


if FEATURE_TYPE_DF.empty:
    raise RuntimeError(
        "FEATURE_TYPE_DF is empty."
    )


# ------------------------------------------------------------
# 3. Validate feature-type schema
# ------------------------------------------------------------

required_feature_columns = {
    "dataset_id",
    "feature",
    "feature_type",
}

missing_feature_columns = (
    required_feature_columns
    -
    set(FEATURE_TYPE_DF.columns)
)

if missing_feature_columns:

    raise RuntimeError(
        "FEATURE_TYPE_DF is missing required columns:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in sorted(
                missing_feature_columns
            )
        )
    )


# ------------------------------------------------------------
# 4. Strategy-ID normalization
# ------------------------------------------------------------

def normalize_strategy_id(strategy_id):

    if strategy_id is None:

        raise ValueError(
            "Strategy ID cannot be None."
        )

    value = (
        str(strategy_id)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    aliases = {

        "iterative":
            "iterativeimputer",

        "iterative_imputer":
            "iterativeimputer",

        "mice":
            "iterativeimputer",

        "random_forest":
            "randomforest",

        "random_forest_imputer":
            "randomforest",

        "miss_forest":
            "missforest",

        "random_sample_imputer":
            "random_sample",

        "gradientboosting":
            "gradient_boosting",

        "matrixfactorization":
            "matrix_factorization",

    }

    return aliases.get(
        value,
        value
    )


# ------------------------------------------------------------
# 5. Feature-type normalization
# ------------------------------------------------------------

def normalize_feature_type(value):

    value = (
        str(value)
        .strip()
        .lower()
    )

    numerical_aliases = {
        "numeric",
        "numerical",
        "number",
        "integer",
        "int",
        "int32",
        "int64",
        "float",
        "float32",
        "float64",
        "continuous",
    }

    categorical_aliases = {
        "categorical",
        "category",
        "object",
        "string",
        "str",
        "bool",
        "boolean",
    }

    if value in numerical_aliases:
        return "numerical"

    if value in categorical_aliases:
        return "categorical"

    return value


FEATURE_TYPE_DF = FEATURE_TYPE_DF.copy()

FEATURE_TYPE_DF[
    "normalized_feature_type"
] = (
    FEATURE_TYPE_DF[
        "feature_type"
    ]
    .map(
        normalize_feature_type
    )
)


# ------------------------------------------------------------
# 6. Validate detected feature types
# ------------------------------------------------------------

allowed_feature_types = {
    "numerical",
    "categorical",
}

detected_types = set(
    FEATURE_TYPE_DF[
        "normalized_feature_type"
    ]
    .dropna()
    .unique()
)

unsupported_detected_types = (
    detected_types
    -
    allowed_feature_types
)

if unsupported_detected_types:

    raise RuntimeError(
        "Unsupported detected feature types:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in sorted(
                unsupported_detected_types
            )
        )
    )


# ------------------------------------------------------------
# 7. AIR-LLM candidate compatibility definition
# ------------------------------------------------------------
#
# Compatibility reflects the IMPLEMENTATIONS actually
# available in Notebook 06.10–06.16.
#
# Numerical:
#   Mean
#   Median
#   KNN
#   Iterative/MICE
#   Random Forest
#   Gradient Boosting
#   MissForest
#   SoftImpute
#   Matrix Factorization
#
# Categorical:
#   Mode
#   Constant
#   Random Sample
#
# Constant and Random Sample support both types.
#
# ------------------------------------------------------------

STRATEGY_FEATURE_COMPATIBILITY_MASTER = {

    "mean": {
        "numerical"
    },

    "median": {
        "numerical"
    },

    "mode": {
        "categorical"
    },

    "constant": {
        "numerical",
        "categorical"
    },

    "random_sample": {
        "numerical",
        "categorical"
    },

    "knn": {
        "numerical"
    },

    "iterativeimputer": {
        "numerical"
    },

    "randomforest": {
        "numerical"
    },

    "gradient_boosting": {
        "numerical"
    },

    "missforest": {
        "numerical"
    },

    "softimpute": {
        "numerical"
    },

    "matrix_factorization": {
        "numerical"
    },
}


# ------------------------------------------------------------
# 8. Build implementation registry
# ------------------------------------------------------------
#
# The previous version expected STRATEGY_FUNCTIONS to already
# exist. That was the cause of the failure.
#
# Here we explicitly map the implementations created in:
#
# 06.10 Statistical candidates
# 06.11 KNN
# 06.12 Iterative / MICE
# 06.13 Random Forest
# 06.14 Gradient Boosting
# 06.15 MissForest
# 06.16 SoftImpute / Matrix Factorization
#
# ------------------------------------------------------------

STRATEGY_FUNCTIONS = {}


implementation_map = {

    "mean":
        "impute_mean",

    "median":
        "impute_median",

    "mode":
        "impute_mode",

    "constant":
        "impute_constant",

    "random_sample":
        "impute_random_sample",

    "knn":
        "fit_knn",

    "iterativeimputer":
        "fit_iterative",

    "randomforest":
        "fit_random_forest",

    "gradient_boosting":
        "fit_gradient_boosting",

    "missforest":
        "fit_missforest",

    "softimpute":
        "fit_softimpute",

    "matrix_factorization":
        "fit_matrix_factorization",
}


missing_function_names = []

for strategy_id, function_name in implementation_map.items():

    if function_name not in globals():

        missing_function_names.append(
            f"{strategy_id} -> {function_name}"
        )

    else:

        STRATEGY_FUNCTIONS[
            strategy_id
        ] = globals()[
            function_name
        ]


if missing_function_names:

    raise RuntimeError(
        "Required candidate implementation functions "
        "are missing:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_function_names
        )
        +
        "\n\n"
        "Required implementation cells are "
        "Notebook 06.10 → 06.16."
    )


# ------------------------------------------------------------
# 9. Determine required strategies
# ------------------------------------------------------------

REQUIRED_STRATEGIES = sorted(
    STRATEGY_FEATURE_COMPATIBILITY_MASTER.keys()
)


# ------------------------------------------------------------
# 10. Validate strategy definitions
# ------------------------------------------------------------

undefined_strategies = [
    strategy_id
    for strategy_id in REQUIRED_STRATEGIES
    if strategy_id
    not in STRATEGY_FEATURE_COMPATIBILITY_MASTER
]

if undefined_strategies:

    raise RuntimeError(
        "The following strategies have no "
        "compatibility definition:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in undefined_strategies
        )
    )


# ------------------------------------------------------------
# 11. Validate implementation coverage
# ------------------------------------------------------------

implemented_strategies = {
    normalize_strategy_id(x)
    for x in STRATEGY_FUNCTIONS.keys()
}


missing_implementations = (
    set(REQUIRED_STRATEGIES)
    -
    implemented_strategies
)

if missing_implementations:

    raise RuntimeError(
        "The following required strategies have "
        "no implementation:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in sorted(
                missing_implementations
            )
        )
    )


# ------------------------------------------------------------
# 12. Build strategy compatibility table
# ------------------------------------------------------------

STRATEGY_COMPATIBILITY_DF = pd.DataFrame([

    {
        "strategy_id":
            strategy_id,

        "supported_feature_types":
            sorted(
                STRATEGY_FEATURE_COMPATIBILITY_MASTER[
                    strategy_id
                ]
            ),

        "implementation_function":
            STRATEGY_FUNCTIONS[
                strategy_id
            ].__name__,

        "implemented":
            True,

    }

    for strategy_id
    in REQUIRED_STRATEGIES

])


# ------------------------------------------------------------
# 13. Dataset-level compatibility validation
# ------------------------------------------------------------

compatibility_rows = []


for dataset_id in sorted(
    EVALUATION_DATA.keys()
):

    dataset_features = (
        FEATURE_TYPE_DF[
            FEATURE_TYPE_DF[
                "dataset_id"
            ].astype(str)
            ==
            str(dataset_id)
        ]
    )


    for strategy_id in REQUIRED_STRATEGIES:

        supported_types = (
            STRATEGY_FEATURE_COMPATIBILITY_MASTER[
                strategy_id
            ]
        )


        compatible_features = (
            dataset_features[
                dataset_features[
                    "normalized_feature_type"
                ].isin(
                    supported_types
                )
            ][
                "feature"
            ]
            .tolist()
        )


        incompatible_features = (
            dataset_features[
                ~dataset_features[
                    "normalized_feature_type"
                ].isin(
                    supported_types
                )
            ][
                "feature"
            ]
            .tolist()
        )


        compatibility_rows.append({

            "dataset_id":
                dataset_id,

            "strategy_id":
                strategy_id,

            "supported_feature_types":
                sorted(
                    supported_types
                ),

            "compatible_feature_count":
                len(
                    compatible_features
                ),

            "incompatible_feature_count":
                len(
                    incompatible_features
                ),

            "compatible":
                len(
                    compatible_features
                ) > 0,

        })


STRATEGY_COMPATIBILITY_VALIDATION_DF = (
    pd.DataFrame(
        compatibility_rows
    )
)


# ------------------------------------------------------------
# 14. Validate every strategy across every dataset
# ------------------------------------------------------------

invalid_strategy_rows = (
    STRATEGY_COMPATIBILITY_VALIDATION_DF[
        ~STRATEGY_COMPATIBILITY_VALIDATION_DF[
            "compatible"
        ]
    ]
)


if not invalid_strategy_rows.empty:

    display(
        invalid_strategy_rows
    )

    raise RuntimeError(
        "One or more strategies have no compatible "
        "features in one or more datasets."
    )


# ------------------------------------------------------------
# 15. Final lookup used by evaluation engine
# ------------------------------------------------------------

STRATEGY_FEATURE_COMPATIBILITY = {

    strategy_id:
        set(
            STRATEGY_FEATURE_COMPATIBILITY_MASTER[
                strategy_id
            ]
        )

    for strategy_id
    in REQUIRED_STRATEGIES

}


# ------------------------------------------------------------
# 16. Final implementation validation
# ------------------------------------------------------------

if set(
    STRATEGY_FUNCTIONS.keys()
) != set(
    REQUIRED_STRATEGIES
):

    raise RuntimeError(
        "Implementation registry does not exactly "
        "match the required candidate strategy pool."
    )


# ------------------------------------------------------------
# 17. Display implementation registry
# ------------------------------------------------------------

display(
    STRATEGY_COMPATIBILITY_DF
)


# ------------------------------------------------------------
# 18. Final report
# ------------------------------------------------------------

print(
    f"\nRequired strategies       : "
    f"{len(REQUIRED_STRATEGIES)}"
)

print(
    f"Detected feature types   : "
    f"{sorted(detected_types)}"
)

print(
    f"Implemented strategies   : "
    f"{len(STRATEGY_FUNCTIONS)}"
)

print(
    f"Compatibility definitions : "
    f"{len(STRATEGY_FEATURE_COMPATIBILITY)}"
)


print("\nStrategy implementation status:")
print("-" * 100)

for strategy_id in REQUIRED_STRATEGIES:

    function_name = (
        STRATEGY_FUNCTIONS[
            strategy_id
        ].__name__
    )

    supported = sorted(
        STRATEGY_FEATURE_COMPATIBILITY[
            strategy_id
        ]
    )

    print(
        f"  ✓ {strategy_id:<22} "
        f"{function_name:<28} "
        f"{supported}"
    )


# ------------------------------------------------------------
# 19. Final validation
# ------------------------------------------------------------

print("\n" + "=" * 100)
print(
    "CANDIDATE STRATEGY COMPATIBILITY : PASSED"
)
print("=" * 100)

AIR-LLM — NOTEBOOK 06.17
CANDIDATE STRATEGY COMPATIBILITY VALIDATION


,strategy_id,supported_feature_types,implementation_function,implemented
0,constant,"[categorical, numerical]",impute_constant,True
1,gradient_boosting,[numerical],fit_gradient_boosting,True
2,iterativeimputer,[numerical],fit_iterative,True
3,knn,[numerical],fit_knn,True
4,matrix_factorization,[numerical],fit_matrix_factorization,True
5,mean,[numerical],impute_mean,True
6,median,[numerical],impute_median,True
7,missforest,[numerical],fit_missforest,True
8,mode,[categorical],impute_mode,True
9,random_sample,"[categorical, numerical]",impute_random_sample,True



Required strategies       : 12
Detected feature types   : ['categorical', 'numerical']
Implemented strategies   : 12
Compatibility definitions : 12

Strategy implementation status:
----------------------------------------------------------------------------------------------------
  ✓ constant               impute_constant              ['categorical', 'numerical']
  ✓ gradient_boosting      fit_gradient_boosting        ['numerical']
  ✓ iterativeimputer       fit_iterative                ['numerical']
  ✓ knn                    fit_knn                      ['numerical']
  ✓ matrix_factorization   fit_matrix_factorization     ['numerical']
  ✓ mean                   impute_mean                  ['numerical']
  ✓ median                 impute_median                ['numerical']
  ✓ missforest             fit_missforest               ['numerical']
  ✓ mode                   impute_mode                  ['categorical']
  ✓ random_sample          impute_random_sample         ['categorical'

In [19]:
# ============================================================
# NOTEBOOK 06.18 — BUILD CANDIDATE EVALUATION PLAN
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.18")
print("BUILD CANDIDATE EVALUATION PLAN")
print("=" * 100)


# ------------------------------------------------------------
# 06.18.01 — Required state
# ------------------------------------------------------------

required_objects = [
    "CONFIG",
    "EVALUATION_DATA",
    "TARGET_REGISTRY",
    "FEATURE_TYPE_DF",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Notebook 06.18 dependencies missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_objects
        )
    )


# ------------------------------------------------------------
# 06.18.02 — Resolve strategy implementation registry
# ------------------------------------------------------------

# The current AIR-LLM pipeline uses STRATEGY_FUNCTIONS
# as the authoritative executable strategy registry.

if "STRATEGY_FUNCTIONS" not in globals():

    # Compatibility recovery from common registry names
    if "STRATEGY_IMPLEMENTATION_REGISTRY" in globals():

        STRATEGY_FUNCTIONS = (
            STRATEGY_IMPLEMENTATION_REGISTRY.copy()
        )

    elif "CANDIDATE_STRATEGY_REGISTRY" in globals():

        STRATEGY_FUNCTIONS = (
            CANDIDATE_STRATEGY_REGISTRY.copy()
        )

    else:

        raise RuntimeError(
            "STRATEGY_FUNCTIONS is not available.\n"
            "Run the candidate strategy implementation "
            "registry cell before Notebook 06.18."
        )


if not isinstance(
    STRATEGY_FUNCTIONS,
    dict
):

    raise RuntimeError(
        "STRATEGY_FUNCTIONS must be a dictionary."
    )


if not STRATEGY_FUNCTIONS:

    raise RuntimeError(
        "STRATEGY_FUNCTIONS is empty."
    )


# ------------------------------------------------------------
# 06.18.03 — Normalize strategy IDs
# ------------------------------------------------------------

def normalize_strategy_id_06_18(
    value
):

    value = str(
        value
    ).strip().lower()

    aliases = {

        "randomsample":
            "random_sample",

        "random-sample":
            "random_sample",

        "random_sample_imputation":
            "random_sample",

        "iterative_imputer":
            "iterativeimputer",

        "iterative":
            "iterativeimputer",

        "mice":
            "iterativeimputer",

        "random_forest":
            "randomforest",

        "random-forest":
            "randomforest",

        "gradient_boosting":
            "gradientboosting",

        "gradient-boosting":
            "gradientboosting",

        "soft_impute":
            "softimpute",

        "matrix_factorization":
            "matrixfactorization",
    }

    return aliases.get(
        value,
        value
    )


normalized_strategy_functions = {}

for strategy_id, function in (
    STRATEGY_FUNCTIONS.items()
):

    normalized_id = (
        normalize_strategy_id_06_18(
            strategy_id
        )
    )

    if not callable(function):

        raise RuntimeError(
            f"Strategy '{strategy_id}' "
            "does not contain a callable implementation."
        )

    if normalized_id in normalized_strategy_functions:

        raise RuntimeError(
            "Duplicate normalized strategy ID: "
            f"{normalized_id}"
        )

    normalized_strategy_functions[
        normalized_id
    ] = function


STRATEGY_FUNCTIONS = (
    normalized_strategy_functions
)


print(
    f"\n[06.18.03] Executable strategies: "
    f"{len(STRATEGY_FUNCTIONS)}"
)


# ------------------------------------------------------------
# 06.18.04 — Resolve strategy compatibility
# ------------------------------------------------------------

if (
    "STRATEGY_FEATURE_COMPATIBILITY"
    not in globals()
):

    raise RuntimeError(
        "STRATEGY_FEATURE_COMPATIBILITY is not available.\n"
        "Run the corrected Notebook 06.17 first."
    )


STRATEGY_FEATURE_COMPATIBILITY = {

    normalize_strategy_id_06_18(
        strategy_id
    ): {
        str(feature_type).strip().lower()
        for feature_type in supported_types
    }

    for strategy_id, supported_types
    in STRATEGY_FEATURE_COMPATIBILITY.items()
}


# ------------------------------------------------------------
# 06.18.05 — Resolve experiment configuration
# ------------------------------------------------------------

if "EXPERIMENT_CONFIGURATION" not in globals():

    raise RuntimeError(
        "EXPERIMENT_CONFIGURATION is not available.\n"
        "Run Notebook 06.03 first."
    )


experiment_config = dict(
    EXPERIMENT_CONFIGURATION
)


MISSINGNESS_MECHANISMS = [
    str(x).upper()
    for x in experiment_config.get(
        "missingness_mechanisms",
        ["MCAR", "MAR", "MNAR"]
    )
]


MISSINGNESS_RATES = [
    float(x)
    for x in experiment_config.get(
        "missingness_rates",
        [0.10, 0.20, 0.30, 0.40, 0.50]
    )
]


REPETITIONS = int(
    experiment_config.get(
        "repetitions",
        5
    )
)


MASTER_SEED = int(
    experiment_config.get(
        "master_seed",
        42
    )
)


# ------------------------------------------------------------
# 06.18.06 — Strict experiment validation
# ------------------------------------------------------------

EXPECTED_MECHANISMS = {
    "MCAR",
    "MAR",
    "MNAR",
}


EXPECTED_RATES = {
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
}


if set(
    MISSINGNESS_MECHANISMS
) != EXPECTED_MECHANISMS:

    raise RuntimeError(
        "AIR-LLM requires MCAR, MAR, and MNAR."
    )


if set(
    MISSINGNESS_RATES
) != EXPECTED_RATES:

    raise RuntimeError(
        "AIR-LLM requires missingness rates "
        "0.10, 0.20, 0.30, 0.40, and 0.50."
    )


if REPETITIONS != 5:

    raise RuntimeError(
        "AIR-LLM requires exactly 5 repetitions."
    )


# ------------------------------------------------------------
# 06.18.07 — Deterministic seed generator
# ------------------------------------------------------------

import hashlib


def make_experiment_seed_06_18(
    dataset_id,
    feature,
    mechanism,
    rate,
    repetition,
    master_seed
):

    key = (
        f"{master_seed}|"
        f"{dataset_id}|"
        f"{feature}|"
        f"{mechanism}|"
        f"{float(rate):.4f}|"
        f"{repetition}"
    )

    digest = hashlib.sha256(
        key.encode(
            "utf-8"
        )
    ).hexdigest()

    # Keep seed safely inside NumPy's uint32 range.
    return int(
        digest[:8],
        16
    )


# ------------------------------------------------------------
# 06.18.08 — Normalize feature types
# ------------------------------------------------------------

def normalize_feature_type_06_18(
    value
):

    value = str(
        value
    ).strip().lower()

    numerical_aliases = {
        "numeric",
        "numerical",
        "float",
        "float32",
        "float64",
        "integer",
        "int",
        "int32",
        "int64",
        "continuous",
    }

    categorical_aliases = {
        "categorical",
        "category",
        "object",
        "string",
        "str",
        "boolean",
        "bool",
    }

    if value in numerical_aliases:
        return "numerical"

    if value in categorical_aliases:
        return "categorical"

    return value


# ------------------------------------------------------------
# 06.18.09 — Build feature-level candidate registry
# ------------------------------------------------------------

FEATURE_CANDIDATE_REGISTRY = {}


for dataset_id, df in EVALUATION_DATA.items():

    if dataset_id not in TARGET_REGISTRY:

        raise RuntimeError(
            f"Target not registered for "
            f"dataset '{dataset_id}'."
        )

    target = str(
        TARGET_REGISTRY[
            dataset_id
        ]
    )

    if target not in df.columns:

        raise RuntimeError(
            f"Target '{target}' not found "
            f"in dataset '{dataset_id}'."
        )


    dataset_feature_rows = FEATURE_TYPE_DF[
        FEATURE_TYPE_DF[
            "dataset_id"
        ].astype(str)
        ==
        str(dataset_id)
    ].copy()


    if dataset_feature_rows.empty:

        raise RuntimeError(
            f"No feature characterization found "
            f"for '{dataset_id}'."
        )


    FEATURE_CANDIDATE_REGISTRY[
        dataset_id
    ] = {}


    for _, feature_row in (
        dataset_feature_rows.iterrows()
    ):

        feature = str(
            feature_row["feature"]
        )

        # Target is never imputed.
        if feature == target:
            continue


        feature_type = (
            normalize_feature_type_06_18(
                feature_row["feature_type"]
            )
        )


        compatible_strategies = []


        for strategy_id in (
            STRATEGY_FUNCTIONS.keys()
        ):

            supported_types = (
                STRATEGY_FEATURE_COMPATIBILITY.get(
                    strategy_id,
                    set()
                )
            )

            supported_types = {
                normalize_feature_type_06_18(
                    x
                )
                for x in supported_types
            }


            if feature_type in supported_types:

                compatible_strategies.append(
                    strategy_id
                )


        if compatible_strategies:

            FEATURE_CANDIDATE_REGISTRY[
                dataset_id
            ][feature] = sorted(
                compatible_strategies
            )


# ------------------------------------------------------------
# 06.18.10 — Validate candidate coverage
# ------------------------------------------------------------

for dataset_id, feature_map in (
    FEATURE_CANDIDATE_REGISTRY.items()
):

    if not feature_map:

        raise RuntimeError(
            f"No compatible candidate strategies "
            f"for dataset '{dataset_id}'."
        )


# ------------------------------------------------------------
# 06.18.11 — Build evaluation plan
# ------------------------------------------------------------

plan_rows = []


for dataset_id, feature_map in (
    FEATURE_CANDIDATE_REGISTRY.items()
):

    target = TARGET_REGISTRY[
        dataset_id
    ]


    for feature, strategies in (
        feature_map.items()
    ):

        feature_type_rows = FEATURE_TYPE_DF[
            (
                FEATURE_TYPE_DF[
                    "dataset_id"
                ].astype(str)
                ==
                str(dataset_id)
            )
            &
            (
                FEATURE_TYPE_DF[
                    "feature"
                ].astype(str)
                ==
                str(feature)
            )
        ]


        if feature_type_rows.empty:
            continue


        feature_type = (
            normalize_feature_type_06_18(
                feature_type_rows.iloc[0][
                    "feature_type"
                ]
            )
        )


        for mechanism in (
            MISSINGNESS_MECHANISMS
        ):

            for rate in (
                sorted(
                    MISSINGNESS_RATES
                )
            ):

                for repetition in range(
                    1,
                    REPETITIONS + 1
                ):


                    experiment_seed = (
                        make_experiment_seed_06_18(
                            dataset_id=dataset_id,
                            feature=feature,
                            mechanism=mechanism,
                            rate=rate,
                            repetition=repetition,
                            master_seed=MASTER_SEED
                        )
                    )


                    for strategy_id in (
                        strategies
                    ):

                        plan_rows.append({

                            "dataset_id":
                                dataset_id,

                            "feature":
                                feature,

                            "target":
                                target,

                            "feature_type":
                                feature_type,

                            "strategy_id":
                                strategy_id,

                            "missingness_mechanism":
                                mechanism,

                            "missingness_rate":
                                float(rate),

                            "repetition":
                                repetition,

                            "llm_recommended":
                                False,

                            "seed":
                                experiment_seed,

                        })


# ------------------------------------------------------------
# 06.18.12 — Create evaluation dataframe
# ------------------------------------------------------------

EVALUATION_PLAN_DF = pd.DataFrame(
    plan_rows
)


if EVALUATION_PLAN_DF.empty:

    raise RuntimeError(
        "EVALUATION_PLAN_DF is empty."
    )


# ------------------------------------------------------------
# 06.18.13 — Validate uniqueness
# ------------------------------------------------------------

plan_keys = [
    "dataset_id",
    "feature",
    "strategy_id",
    "missingness_mechanism",
    "missingness_rate",
    "repetition",
]


duplicate_count = (
    EVALUATION_PLAN_DF
    .duplicated(
        subset=plan_keys
    )
    .sum()
)


if duplicate_count > 0:

    raise RuntimeError(
        f"Evaluation plan contains "
        f"{duplicate_count} duplicate configurations."
    )


# ------------------------------------------------------------
# 06.18.14 — Validate strategy implementations
# ------------------------------------------------------------

missing_implementations = sorted({

    strategy_id

    for strategy_id in (
        EVALUATION_PLAN_DF[
            "strategy_id"
        ].unique()
    )

    if strategy_id
    not in STRATEGY_FUNCTIONS

})


if missing_implementations:

    raise RuntimeError(
        "Evaluation plan contains strategies "
        "without implementations:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_implementations
        )
    )


# ------------------------------------------------------------
# 06.18.15 — Validate strategy compatibility
# ------------------------------------------------------------

for _, row in EVALUATION_PLAN_DF.iterrows():

    strategy_id = row[
        "strategy_id"
    ]

    feature_type = (
        normalize_feature_type_06_18(
            row["feature_type"]
        )
    )

    supported = {
        normalize_feature_type_06_18(
            x
        )
        for x in STRATEGY_FEATURE_COMPATIBILITY.get(
            strategy_id,
            set()
        )
    }


    if feature_type not in supported:

        raise RuntimeError(
            "Incompatible strategy detected:\n"
            f"  strategy  : {strategy_id}\n"
            f"  feature   : {row['feature']}\n"
            f"  type      : {feature_type}"
        )


# ------------------------------------------------------------
# 06.18.16 — Validate mechanisms, rates and repetitions
# ------------------------------------------------------------

if set(
    EVALUATION_PLAN_DF[
        "missingness_mechanism"
    ]
) != EXPECTED_MECHANISMS:

    raise RuntimeError(
        "Not all missingness mechanisms are present."
    )


if set(
    EVALUATION_PLAN_DF[
        "missingness_rate"
    ].astype(float)
) != EXPECTED_RATES:

    raise RuntimeError(
        "Not all missingness rates are present."
    )


if set(
    EVALUATION_PLAN_DF[
        "repetition"
    ]
) != set(
    range(
        1,
        REPETITIONS + 1
    )
):

    raise RuntimeError(
        "Not all repetitions are present."
    )


# ------------------------------------------------------------
# 06.18.17 — Export evaluation plan
# ------------------------------------------------------------

EVALUATION_PLAN = (
    EVALUATION_PLAN_DF.copy()
)


# ------------------------------------------------------------
# 06.18.18 — Summary
# ------------------------------------------------------------

print("\n" + "-" * 100)
print("EVALUATION PLAN SUMMARY")
print("-" * 100)

print(
    f"Datasets                 : "
    f"{EVALUATION_PLAN_DF['dataset_id'].nunique()}"
)

print(
    f"Features                 : "
    f"{EVALUATION_PLAN_DF['feature'].nunique()}"
)

print(
    f"Strategies               : "
    f"{EVALUATION_PLAN_DF['strategy_id'].nunique()}"
)

print(
    f"Mechanisms               : "
    f"{EVALUATION_PLAN_DF['missingness_mechanism'].nunique()}"
)

print(
    f"Missingness rates        : "
    f"{EVALUATION_PLAN_DF['missingness_rate'].nunique()}"
)

print(
    f"Repetitions              : "
    f"{EVALUATION_PLAN_DF['repetition'].nunique()}"
)

print(
    f"Planned candidate runs   : "
    f"{len(EVALUATION_PLAN_DF):,}"
)


print("\nStrategy distribution:")
display(
    EVALUATION_PLAN_DF[
        "strategy_id"
    ]
    .value_counts()
    .rename_axis(
        "strategy_id"
    )
    .reset_index(
        name="planned_runs"
    )
)


print("\nDataset distribution:")
display(
    EVALUATION_PLAN_DF[
        "dataset_id"
    ]
    .value_counts()
    .rename_axis(
        "dataset_id"
    )
    .reset_index(
        name="planned_runs"
    )
)


print("\n" + "=" * 100)
print(
    "CANDIDATE EVALUATION PLAN : PASSED"
)
print("=" * 100)

AIR-LLM — NOTEBOOK 06.18
BUILD CANDIDATE EVALUATION PLAN

[06.18.03] Executable strategies: 12

----------------------------------------------------------------------------------------------------
EVALUATION PLAN SUMMARY
----------------------------------------------------------------------------------------------------
Datasets                 : 3
Features                 : 73
Strategies               : 12
Mechanisms               : 3
Missingness rates        : 5
Repetitions              : 5
Planned candidate runs   : 31,725

Strategy distribution:


,strategy_id,planned_runs
0,constant,5775
1,random_sample,5775
2,mode,3975
3,gradientboosting,1800
4,knn,1800
5,iterativeimputer,1800
6,matrixfactorization,1800
7,mean,1800
8,missforest,1800
9,median,1800



Dataset distribution:


,dataset_id,planned_runs
0,diabetes_130us,17175
1,bank_marketing,7800
2,adult_income,6750



CANDIDATE EVALUATION PLAN : PASSED


In [20]:
# ============================================================
# NOTEBOOK 06.19 — EVALUATION-PLAN INTEGRITY AND EXPECTED COUNT
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.19")
print("EVALUATION-PLAN INTEGRITY AND EXPECTED COUNT")
print("=" * 100)


# ------------------------------------------------------------
# 06.19.01 — Imports
# ------------------------------------------------------------

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 06.19.02 — Required state
# ------------------------------------------------------------

required_objects = [
    "EVALUATION_PLAN_DF",
    "EVALUATION_DATA",
    "STRATEGY_FUNCTIONS",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:

    raise RuntimeError(
        "Notebook 06.19 is missing required dependencies:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing_objects
        )
        +
        "\n\n"
        "Run Notebook 06.18 first."
    )


if not isinstance(
    EVALUATION_PLAN_DF,
    pd.DataFrame
):

    raise TypeError(
        "EVALUATION_PLAN_DF must be a pandas DataFrame."
    )


if not isinstance(
    STRATEGY_FUNCTIONS,
    dict
):

    raise TypeError(
        "STRATEGY_FUNCTIONS must be a dictionary."
    )


if not STRATEGY_FUNCTIONS:

    raise RuntimeError(
        "STRATEGY_FUNCTIONS is empty."
    )


# ------------------------------------------------------------
# 06.19.03 — Normalize strategy IDs
# ------------------------------------------------------------

def normalize_strategy_id_06_19(
    value
):

    value = (
        str(value)
        .strip()
        .lower()
    )

    aliases = {

        "randomsample":
            "random_sample",

        "random-sample":
            "random_sample",

        "random_sample_imputation":
            "random_sample",

        "iterative":
            "iterativeimputer",

        "iterative_imputer":
            "iterativeimputer",

        "mice":
            "iterativeimputer",

        "random_forest":
            "randomforest",

        "random-forest":
            "randomforest",

        "random_forest_imputer":
            "randomforest",

        "gradient_boosting":
            "gradientboosting",

        "gradient-boosting":
            "gradientboosting",

        "soft_impute":
            "softimpute",

        "matrix_factorization":
            "matrixfactorization",

        "miss_forest":
            "missforest",
    }

    return aliases.get(
        value,
        value
    )


# ------------------------------------------------------------
# 06.19.04 — Normalize executable strategy registry
# ------------------------------------------------------------

normalized_strategy_functions = {}


for strategy_id, function in (
    STRATEGY_FUNCTIONS.items()
):

    normalized_id = (
        normalize_strategy_id_06_19(
            strategy_id
        )
    )


    if not callable(function):

        raise RuntimeError(
            f"Strategy '{strategy_id}' "
            "does not have a callable implementation."
        )


    if normalized_id in (
        normalized_strategy_functions
    ):

        raise RuntimeError(
            "Duplicate normalized strategy ID detected: "
            f"{normalized_id}"
        )


    normalized_strategy_functions[
        normalized_id
    ] = function


STRATEGY_FUNCTIONS = (
    normalized_strategy_functions
)


print(
    f"\nExecutable strategies : "
    f"{len(STRATEGY_FUNCTIONS)}"
)


# ------------------------------------------------------------
# 06.19.05 — Required evaluation-plan columns
# ------------------------------------------------------------

REQUIRED_PLAN_COLUMNS_0619 = [

    "dataset_id",
    "feature",
    "target",
    "feature_type",
    "strategy_id",
    "missingness_mechanism",
    "missingness_rate",
    "repetition",
    "llm_recommended",
    "seed",

]


missing_columns = [

    column

    for column
    in REQUIRED_PLAN_COLUMNS_0619

    if column
    not in EVALUATION_PLAN_DF.columns

]


if missing_columns:

    raise RuntimeError(
        "Evaluation plan is missing required columns:\n"
        +
        "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )


# ------------------------------------------------------------
# 06.19.06 — Basic dataframe validation
# ------------------------------------------------------------

if EVALUATION_PLAN_DF.empty:

    raise RuntimeError(
        "EVALUATION_PLAN_DF is empty."
    )


EVALUATION_PLAN_DF = (
    EVALUATION_PLAN_DF.copy()
)


# ------------------------------------------------------------
# 06.19.07 — Null validation
# ------------------------------------------------------------

null_columns = (

    EVALUATION_PLAN_DF.columns[
        EVALUATION_PLAN_DF.isna().any()
    ]

    .tolist()

)


if null_columns:

    raise RuntimeError(
        "Evaluation plan contains null values in:\n"
        +
        "\n".join(
            f"  - {column}"
            for column in null_columns
        )
    )


# ------------------------------------------------------------
# 06.19.08 — Normalize strategy IDs
# ------------------------------------------------------------

EVALUATION_PLAN_DF[
    "strategy_id"
] = (

    EVALUATION_PLAN_DF[
        "strategy_id"
    ]

    .map(
        normalize_strategy_id_06_19
    )

)


# ------------------------------------------------------------
# 06.19.09 — Validate strategy implementations
# ------------------------------------------------------------

plan_strategies = set(

    EVALUATION_PLAN_DF[
        "strategy_id"
    ]

    .astype(str)
    .str.strip()
    .str.lower()

)


implemented_strategies = set(
    STRATEGY_FUNCTIONS.keys()
)


missing_implementations = (

    plan_strategies
    -
    implemented_strategies

)


if missing_implementations:

    raise RuntimeError(
        "Evaluation plan contains strategies "
        "without executable implementations:\n"
        +
        "\n".join(
            f"  - {x}"
            for x
            in sorted(
                missing_implementations
            )
        )
    )


# ------------------------------------------------------------
# 06.19.10 — Validate datasets
# ------------------------------------------------------------

plan_datasets = set(

    EVALUATION_PLAN_DF[
        "dataset_id"
    ]

)


available_datasets = set(
    EVALUATION_DATA.keys()
)


unknown_datasets = (

    plan_datasets
    -
    available_datasets

)


if unknown_datasets:

    raise RuntimeError(
        "Evaluation plan contains unknown datasets:\n"
        +
        "\n".join(
            f"  - {x}"
            for x
            in sorted(
                unknown_datasets
            )
        )
    )


# ------------------------------------------------------------
# 06.19.11 — Validate TARGET_REGISTRY
# ------------------------------------------------------------

if "TARGET_REGISTRY" not in globals():

    raise RuntimeError(
        "TARGET_REGISTRY is not available."
    )


for dataset_id in sorted(
    plan_datasets
):

    if dataset_id not in TARGET_REGISTRY:

        raise RuntimeError(
            f"Missing target definition for "
            f"dataset '{dataset_id}'."
        )


    expected_target = str(
        TARGET_REGISTRY[
            dataset_id
        ]
    )


    actual_targets = set(

        EVALUATION_PLAN_DF.loc[
            EVALUATION_PLAN_DF[
                "dataset_id"
            ]
            ==
            dataset_id,
            "target"
        ]
        .astype(str)

    )


    if actual_targets != {
        expected_target
    }:

        raise RuntimeError(
            f"Target mismatch for "
            f"dataset '{dataset_id}'.\n"
            f"Expected: {expected_target}\n"
            f"Found: {sorted(actual_targets)}"
        )


# ------------------------------------------------------------
# 06.19.12 — Validate missingness mechanisms
# ------------------------------------------------------------

VALID_MECHANISMS = {
    "MCAR",
    "MAR",
    "MNAR",
}


EVALUATION_PLAN_DF[
    "missingness_mechanism"
] = (

    EVALUATION_PLAN_DF[
        "missingness_mechanism"
    ]

    .astype(str)
    .str.strip()
    .str.upper()

)


actual_mechanisms = set(

    EVALUATION_PLAN_DF[
        "missingness_mechanism"
    ]

)


invalid_mechanisms = (

    actual_mechanisms
    -
    VALID_MECHANISMS

)


if invalid_mechanisms:

    raise RuntimeError(
        "Invalid missingness mechanisms:\n"
        +
        "\n".join(
            f"  - {x}"
            for x
            in sorted(
                invalid_mechanisms
            )
        )
    )


if actual_mechanisms != VALID_MECHANISMS:

    raise RuntimeError(
        "Evaluation plan does not contain "
        "exactly MCAR, MAR and MNAR."
    )


# ------------------------------------------------------------
# 06.19.13 — Validate missingness rates
# ------------------------------------------------------------

EXPECTED_RATES = {
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
}


EVALUATION_PLAN_DF[
    "missingness_rate"
] = pd.to_numeric(

    EVALUATION_PLAN_DF[
        "missingness_rate"
    ],

    errors="coerce"

)


if EVALUATION_PLAN_DF[
    "missingness_rate"
].isna().any():

    raise RuntimeError(
        "Invalid missingness-rate values detected."
    )


actual_rates = {

    round(
        float(x),
        2
    )

    for x

    in EVALUATION_PLAN_DF[
        "missingness_rate"
    ].unique()

}


if actual_rates != EXPECTED_RATES:

    raise RuntimeError(
        "Invalid missingness rates.\n"
        f"Expected: {sorted(EXPECTED_RATES)}\n"
        f"Found:    {sorted(actual_rates)}"
    )


# ------------------------------------------------------------
# 06.19.14 — Validate repetitions
# ------------------------------------------------------------

EXPECTED_REPETITIONS = {
    1,
    2,
    3,
    4,
    5,
}


EVALUATION_PLAN_DF[
    "repetition"
] = pd.to_numeric(

    EVALUATION_PLAN_DF[
        "repetition"
    ],

    errors="coerce"

)


if EVALUATION_PLAN_DF[
    "repetition"
].isna().any():

    raise RuntimeError(
        "Invalid repetition values detected."
    )


actual_repetitions = set(

    EVALUATION_PLAN_DF[
        "repetition"
    ]
    .astype(int)
    .unique()

)


if actual_repetitions != EXPECTED_REPETITIONS:

    raise RuntimeError(
        "Invalid repetition configuration.\n"
        f"Expected: {sorted(EXPECTED_REPETITIONS)}\n"
        f"Found:    {sorted(actual_repetitions)}"
    )


# ------------------------------------------------------------
# 06.19.15 — Validate seeds
# ------------------------------------------------------------

if not pd.api.types.is_numeric_dtype(
    EVALUATION_PLAN_DF[
        "seed"
    ]
):

    raise RuntimeError(
        "Experiment seeds must be numeric."
    )


if EVALUATION_PLAN_DF[
    "seed"
].isna().any():

    raise RuntimeError(
        "Experiment seeds contain null values."
    )


# ------------------------------------------------------------
# 06.19.16 — Validate candidate uniqueness
# ------------------------------------------------------------

UNIQUE_CONFIGURATION_COLUMNS = [

    "dataset_id",
    "feature",
    "strategy_id",
    "missingness_mechanism",
    "missingness_rate",
    "repetition",

]


duplicate_count = int(

    EVALUATION_PLAN_DF

    .duplicated(
        subset=UNIQUE_CONFIGURATION_COLUMNS
    )

    .sum()

)


if duplicate_count > 0:

    raise RuntimeError(
        f"Found {duplicate_count:,} duplicate "
        "candidate configurations."
    )


# ------------------------------------------------------------
# 06.19.17 — Expected candidate count
# ------------------------------------------------------------

UNIQUE_CONFIGURATION_DF = (

    EVALUATION_PLAN_DF[
        UNIQUE_CONFIGURATION_COLUMNS
    ]

    .drop_duplicates()

)


EXPECTED_EXPERIMENT_COUNT = len(
    UNIQUE_CONFIGURATION_DF
)


ACTUAL_EXPERIMENT_COUNT = len(
    EVALUATION_PLAN_DF
)


if (
    EXPECTED_EXPERIMENT_COUNT
    !=
    ACTUAL_EXPERIMENT_COUNT
):

    raise RuntimeError(
        "Expected candidate count does not match "
        "the evaluation-plan row count."
    )


# ------------------------------------------------------------
# 06.19.18 — Validate deterministic seed consistency
# ------------------------------------------------------------

if "make_experiment_seed_06_18" in globals():

    inconsistent_seed_rows = []


    for _, row in (
        EVALUATION_PLAN_DF.iterrows()
    ):

        expected_seed = (
            make_experiment_seed_06_18(
                dataset_id=row[
                    "dataset_id"
                ],

                feature=row[
                    "feature"
                ],

                mechanism=row[
                    "missingness_mechanism"
                ],

                rate=row[
                    "missingness_rate"
                ],

                repetition=int(
                    row[
                        "repetition"
                    ]
                ),

                master_seed=MASTER_SEED
                if "MASTER_SEED"
                in globals()
                else 42,
            )
        )


        if int(
            row["seed"]
        ) != int(
            expected_seed
        ):

            inconsistent_seed_rows.append(
                row.name
            )


    if inconsistent_seed_rows:

        raise RuntimeError(
            "Deterministic seed validation failed for "
            f"{len(inconsistent_seed_rows):,} rows."
        )


    seed_validation_status = "PASS"


else:

    seed_validation_status = (
        "SKIPPED — generator unavailable"
    )


# ------------------------------------------------------------
# 06.19.19 — Build integrity summary
# ------------------------------------------------------------

EVALUATION_PLAN_INTEGRITY_DF = pd.DataFrame({

    "check": [

        "Datasets",
        "Features",
        "Strategies",
        "Mechanisms",
        "Missingness rates",
        "Repetitions",
        "Candidate runs",
        "Duplicate configurations",
        "Executable implementations",
        "Deterministic seeds",

    ],

    "value": [

        EVALUATION_PLAN_DF[
            "dataset_id"
        ].nunique(),

        EVALUATION_PLAN_DF[
            "feature"
        ].nunique(),

        EVALUATION_PLAN_DF[
            "strategy_id"
        ].nunique(),

        EVALUATION_PLAN_DF[
            "missingness_mechanism"
        ].nunique(),

        EVALUATION_PLAN_DF[
            "missingness_rate"
        ].nunique(),

        EVALUATION_PLAN_DF[
            "repetition"
        ].nunique(),

        ACTUAL_EXPERIMENT_COUNT,

        duplicate_count,

        len(
            implemented_strategies
        ),

        seed_validation_status,

    ],

    "status": [

        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS"
        if seed_validation_status == "PASS"
        else "CHECK",

    ],

})


display(
    EVALUATION_PLAN_INTEGRITY_DF
)


# ------------------------------------------------------------
# 06.19.20 — Detailed distribution
# ------------------------------------------------------------

print("\nStrategy distribution:")
print("-" * 100)

display(

    EVALUATION_PLAN_DF[
        "strategy_id"
    ]

    .value_counts()

    .rename_axis(
        "strategy_id"
    )

    .reset_index(
        name="planned_runs"
    )

)


print("\nDataset distribution:")
print("-" * 100)

display(

    EVALUATION_PLAN_DF[
        "dataset_id"
    ]

    .value_counts()

    .rename_axis(
        "dataset_id"
    )

    .reset_index(
        name="planned_runs"
    )

)


# ------------------------------------------------------------
# 06.19.21 — Final report
# ------------------------------------------------------------

print("\n" + "=" * 100)

print(
    "EVALUATION-PLAN INTEGRITY REPORT"
)

print("=" * 100)


print(
    f"\nDatasets                 : "
    f"{EVALUATION_PLAN_DF['dataset_id'].nunique()}"
)

print(
    f"Features                 : "
    f"{EVALUATION_PLAN_DF['feature'].nunique()}"
)

print(
    f"Strategies               : "
    f"{EVALUATION_PLAN_DF['strategy_id'].nunique()}"
)

print(
    f"Mechanisms               : "
    f"{sorted(VALID_MECHANISMS)}"
)

print(
    f"Missingness rates        : "
    f"{sorted(EXPECTED_RATES)}"
)

print(
    f"Repetitions              : "
    f"{sorted(EXPECTED_REPETITIONS)}"
)

print(
    f"Expected candidate runs  : "
    f"{EXPECTED_EXPERIMENT_COUNT:,}"
)

print(
    f"Actual candidate runs    : "
    f"{ACTUAL_EXPERIMENT_COUNT:,}"
)

print(
    f"Duplicate configurations : "
    f"{duplicate_count:,}"
)

print(
    f"Executable strategies    : "
    f"{len(implemented_strategies)}"
)

print(
    f"Seed validation          : "
    f"{seed_validation_status}"
)


# ------------------------------------------------------------
# 06.19.22 — Export validated plan
# ------------------------------------------------------------

EVALUATION_PLAN = (
    EVALUATION_PLAN_DF.copy()
)


print("\n" + "=" * 100)
print(
    "EVALUATION PLAN INTEGRITY : PASSED"
)
print("=" * 100)

AIR-LLM — NOTEBOOK 06.19
EVALUATION-PLAN INTEGRITY AND EXPECTED COUNT

Executable strategies : 12


,check,value,status
0,Datasets,3,PASS
1,Features,73,PASS
2,Strategies,12,PASS
3,Mechanisms,3,PASS
4,Missingness rates,5,PASS
5,Repetitions,5,PASS
6,Candidate runs,31725,PASS
7,Duplicate configurations,0,PASS
8,Executable implementations,12,PASS
9,Deterministic seeds,PASS,PASS



Strategy distribution:
----------------------------------------------------------------------------------------------------


,strategy_id,planned_runs
0,constant,5775
1,random_sample,5775
2,mode,3975
3,gradientboosting,1800
4,knn,1800
5,iterativeimputer,1800
6,matrixfactorization,1800
7,mean,1800
8,missforest,1800
9,median,1800



Dataset distribution:
----------------------------------------------------------------------------------------------------


,dataset_id,planned_runs
0,diabetes_130us,17175
1,bank_marketing,7800
2,adult_income,6750



EVALUATION-PLAN INTEGRITY REPORT

Datasets                 : 3
Features                 : 73
Strategies               : 12
Mechanisms               : ['MAR', 'MCAR', 'MNAR']
Missingness rates        : [0.1, 0.2, 0.3, 0.4, 0.5]
Repetitions              : [1, 2, 3, 4, 5]
Expected candidate runs  : 31,725
Actual candidate runs    : 31,725
Duplicate configurations : 0
Executable strategies    : 12
Seed validation          : PASS

EVALUATION PLAN INTEGRITY : PASSED


In [21]:
# ============================================================
# NOTEBOOK 06.19A — FINAL EXECUTABLE STRATEGY REGISTRY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.19A")
print("FINAL EXECUTABLE STRATEGY REGISTRY")
print("=" * 100)


# ------------------------------------------------------------
# 1. Required implementation functions
# ------------------------------------------------------------

required_functions = {
    "gradientboosting": "fit_gradient_boosting",
    "softimpute": "fit_softimpute",
    "matrixfactorization": "fit_matrix_factorization",
}


missing_functions = [
    function_name
    for function_name in required_functions.values()
    if function_name not in globals()
]


if missing_functions:

    raise RuntimeError(
        "Required candidate implementation functions are missing:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_functions
        )
        +
        "\n\n"
        "Run Notebook 06.14 and 06.16 first."
    )


# ------------------------------------------------------------
# 2. Build / update executable registry
# ------------------------------------------------------------

if "STRATEGY_FUNCTIONS" not in globals():

    STRATEGY_FUNCTIONS = {}


if not isinstance(
    STRATEGY_FUNCTIONS,
    dict
):

    raise RuntimeError(
        "STRATEGY_FUNCTIONS must be a dictionary."
    )


# ------------------------------------------------------------
# 3. Register newly implemented candidates
# ------------------------------------------------------------

STRATEGY_FUNCTIONS[
    "gradientboosting"
] = fit_gradient_boosting


STRATEGY_FUNCTIONS[
    "softimpute"
] = fit_softimpute


STRATEGY_FUNCTIONS[
    "matrixfactorization"
] = fit_matrix_factorization


# ------------------------------------------------------------
# 4. Canonical strategy-ID normalization
# ------------------------------------------------------------

def normalize_strategy_id_final(
    value
):

    value = (
        str(value)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    aliases = {

        "gradient_boosting":
            "gradientboosting",

        "gradientboost":
            "gradientboosting",

        "soft_impute":
            "softimpute",

        "matrix_factorization":
            "matrixfactorization",

        "iterative":
            "iterativeimputer",

        "iterative_imputer":
            "iterativeimputer",

        "mice":
            "iterativeimputer",

        "random_forest":
            "randomforest",

        "random_forest_imputer":
            "randomforest",

        "random_sample_imputer":
            "random_sample",

        "randomsample":
            "random_sample",

    }

    return aliases.get(
        value,
        value
    )


# ------------------------------------------------------------
# 5. Normalize entire executable registry
# ------------------------------------------------------------

normalized_strategy_functions = {}


for strategy_id, function in (
    STRATEGY_FUNCTIONS.items()
):

    normalized_id = (
        normalize_strategy_id_final(
            strategy_id
        )
    )

    if not callable(function):

        raise RuntimeError(
            f"Strategy '{strategy_id}' "
            "does not have a callable implementation."
        )

    if normalized_id in normalized_strategy_functions:

        # Allow replacement of an existing alias with
        # the canonical implementation.
        if (
            normalized_strategy_functions[
                normalized_id
            ]
            is not function
        ):

            normalized_strategy_functions[
                normalized_id
            ] = function

    else:

        normalized_strategy_functions[
            normalized_id
        ] = function


STRATEGY_FUNCTIONS = (
    normalized_strategy_functions
)


# ------------------------------------------------------------
# 6. Validate strategies required by evaluation plan
# ------------------------------------------------------------

required_plan_strategies = {

    normalize_strategy_id_final(
        x
    )

    for x in
    EVALUATION_PLAN_DF[
        "strategy_id"
    ].dropna().unique()
}


missing_plan_implementations = sorted(
    required_plan_strategies
    -
    set(
        STRATEGY_FUNCTIONS.keys()
    )
)


if missing_plan_implementations:

    raise RuntimeError(
        "The following strategies are required by "
        "EVALUATION_PLAN_DF but have no executable "
        "implementation:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_plan_implementations
        )
    )


# ------------------------------------------------------------
# 7. Final registry report
# ------------------------------------------------------------

print(
    f"\nExecutable strategies registered : "
    f"{len(STRATEGY_FUNCTIONS)}"
)

print("\nStrategy implementation status:")
print("-" * 100)

for strategy_id in sorted(
    required_plan_strategies
):

    function = STRATEGY_FUNCTIONS[
        strategy_id
    ]

    print(
        f"  ✓ {strategy_id:<24} "
        f"-> {function.__name__}"
    )


print("\n" + "=" * 100)
print(
    "EXECUTABLE STRATEGY REGISTRY : PASSED"
)
print("=" * 100)

AIR-LLM — NOTEBOOK 06.19A
FINAL EXECUTABLE STRATEGY REGISTRY

Executable strategies registered : 12

Strategy implementation status:
----------------------------------------------------------------------------------------------------
  ✓ constant                 -> impute_constant
  ✓ gradientboosting         -> fit_gradient_boosting
  ✓ iterativeimputer         -> fit_iterative
  ✓ knn                      -> fit_knn
  ✓ matrixfactorization      -> fit_matrix_factorization
  ✓ mean                     -> impute_mean
  ✓ median                   -> impute_median
  ✓ missforest               -> fit_missforest
  ✓ mode                     -> impute_mode
  ✓ random_sample            -> impute_random_sample
  ✓ randomforest             -> fit_random_forest
  ✓ softimpute               -> fit_softimpute

EXECUTABLE STRATEGY REGISTRY : PASSED


In [22]:
# ============================================================
# NOTEBOOK 06.20 — SMALL-RUN PREPARATION / PREFLIGHT
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.20")
print("SMALL-RUN PREPARATION / CANDIDATE EXECUTION PREFLIGHT")
print("=" * 100)


# ============================================================
# 06.20.01 — Required state
# ============================================================

required_objects = [
    "EVALUATION_DATA",
    "EVALUATION_PLAN_DF",
    "STRATEGY_FUNCTIONS",
]


missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:

    raise RuntimeError(
        "Notebook 06.20 dependencies are missing:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_objects
        )
        +
        "\n\n"
        "Run Notebook 06.18 and the final executable "
        "strategy registry cell before Notebook 06.20."
    )


# ============================================================
# 06.20.02 — Validate core object types
# ============================================================

if not isinstance(
    EVALUATION_DATA,
    dict
):

    raise TypeError(
        "EVALUATION_DATA must be a dictionary."
    )


if not isinstance(
    EVALUATION_PLAN_DF,
    pd.DataFrame
):

    raise TypeError(
        "EVALUATION_PLAN_DF must be a pandas DataFrame."
    )


if not isinstance(
    STRATEGY_FUNCTIONS,
    dict
):

    raise TypeError(
        "STRATEGY_FUNCTIONS must be a dictionary."
    )


if EVALUATION_PLAN_DF.empty:

    raise RuntimeError(
        "EVALUATION_PLAN_DF is empty."
    )


if not STRATEGY_FUNCTIONS:

    raise RuntimeError(
        "STRATEGY_FUNCTIONS is empty."
    )


# ============================================================
# 06.20.03 — Canonical strategy-ID normalization
# ============================================================

def normalize_strategy_id_06_20(
    value
):

    value = (
        str(value)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    aliases = {

        # Basic statistical
        "randomsample":
            "random_sample",

        "random_sample_imputation":
            "random_sample",

        "random_sample_imputer":
            "random_sample",

        # Iterative / MICE
        "iterative":
            "iterativeimputer",

        "iterative_imputer":
            "iterativeimputer",

        "mice":
            "iterativeimputer",

        # Random forest
        "random_forest":
            "randomforest",

        "random_forest_imputer":
            "randomforest",

        "random-forest":
            "randomforest",

        # Gradient boosting
        "gradient_boosting":
            "gradientboosting",

        "gradient-boosting":
            "gradientboosting",

        "gradient_boost":
            "gradientboosting",

        # SoftImpute
        "soft_impute":
            "softimpute",

        "soft-impute":
            "softimpute",

        # Matrix factorization
        "matrix_factorization":
            "matrixfactorization",

        "matrix-factorization":
            "matrixfactorization",

    }

    return aliases.get(
        value,
        value
    )


# ============================================================
# 06.20.04 — Normalize executable strategy registry
# ============================================================

normalized_strategy_functions = {}


for strategy_id, function in (
    STRATEGY_FUNCTIONS.items()
):

    normalized_id = (
        normalize_strategy_id_06_20(
            strategy_id
        )
    )


    if not callable(function):

        raise RuntimeError(
            f"Strategy '{strategy_id}' does not contain "
            "a callable implementation."
        )


    if normalized_id in normalized_strategy_functions:

        raise RuntimeError(
            "Duplicate normalized strategy ID detected: "
            f"{normalized_id}"
        )


    normalized_strategy_functions[
        normalized_id
    ] = function


STRATEGY_FUNCTIONS = (
    normalized_strategy_functions
)


# ============================================================
# 06.20.05 — Required evaluation-plan columns
# ============================================================

required_plan_columns = [

    "dataset_id",
    "feature",
    "target",
    "feature_type",
    "strategy_id",
    "missingness_mechanism",
    "missingness_rate",
    "repetition",
    "llm_recommended",
    "seed",

]


missing_plan_columns = [

    column
    for column
    in required_plan_columns
    if column
    not in EVALUATION_PLAN_DF.columns

]


if missing_plan_columns:

    raise RuntimeError(
        "EVALUATION_PLAN_DF is missing required columns:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_plan_columns
        )
    )


# ============================================================
# 06.20.06 — Validate datasets in evaluation plan
# ============================================================

plan_datasets = set(
    EVALUATION_PLAN_DF[
        "dataset_id"
    ]
)


available_datasets = set(
    EVALUATION_DATA.keys()
)


unknown_datasets = (
    plan_datasets
    -
    available_datasets
)


if unknown_datasets:

    raise RuntimeError(
        "Evaluation plan contains datasets that are not "
        "available in EVALUATION_DATA:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in sorted(
                unknown_datasets
            )
        )
    )


# ============================================================
# 06.20.07 — Validate every strategy in evaluation plan
# ============================================================

required_plan_strategies = {

    normalize_strategy_id_06_20(
        strategy_id
    )

    for strategy_id
    in EVALUATION_PLAN_DF[
        "strategy_id"
    ].dropna().unique()

}


missing_implementations = sorted(

    required_plan_strategies
    -
    set(
        STRATEGY_FUNCTIONS.keys()
    )

)


if missing_implementations:

    raise RuntimeError(
        "The evaluation plan contains strategies without "
        "executable implementations:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_implementations
        )
    )


# ============================================================
# 06.20.08 — Validate dataset / feature / target references
# ============================================================

for _, row in EVALUATION_PLAN_DF.iterrows():

    dataset_id = row[
        "dataset_id"
    ]

    feature = str(
        row[
            "feature"
        ]
    )

    target = str(
        row[
            "target"
        ]
    )


    df = EVALUATION_DATA[
        dataset_id
    ]


    if not isinstance(
        df,
        pd.DataFrame
    ):

        raise TypeError(
            f"EVALUATION_DATA['{dataset_id}'] "
            "must be a pandas DataFrame."
        )


    if feature not in df.columns:

        raise RuntimeError(
            f"Feature '{feature}' not found in "
            f"dataset '{dataset_id}'."
        )


    if target not in df.columns:

        raise RuntimeError(
            f"Target '{target}' not found in "
            f"dataset '{dataset_id}'."
        )


# ============================================================
# 06.20.09 — Validate experiment configuration
# ============================================================

valid_mechanisms = {
    "MCAR",
    "MAR",
    "MNAR",
}


actual_mechanisms = {

    str(x)
    .strip()
    .upper()

    for x
    in EVALUATION_PLAN_DF[
        "missingness_mechanism"
    ]
    .dropna()
    .unique()

}


if actual_mechanisms != valid_mechanisms:

    raise RuntimeError(
        "Invalid missingness mechanisms.\n"
        f"Expected: {sorted(valid_mechanisms)}\n"
        f"Found:    {sorted(actual_mechanisms)}"
    )


expected_rates = {
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
}


actual_rates = {

    round(
        float(x),
        2
    )

    for x
    in EVALUATION_PLAN_DF[
        "missingness_rate"
    ]
    .dropna()
    .unique()

}


if actual_rates != expected_rates:

    raise RuntimeError(
        "Invalid missingness rates.\n"
        f"Expected: {sorted(expected_rates)}\n"
        f"Found:    {sorted(actual_rates)}"
    )


expected_repetitions = {
    1,
    2,
    3,
    4,
    5,
}


actual_repetitions = {

    int(x)

    for x
    in EVALUATION_PLAN_DF[
        "repetition"
    ]
    .dropna()
    .unique()

}


if actual_repetitions != expected_repetitions:

    raise RuntimeError(
        "Invalid repetitions.\n"
        f"Expected: {sorted(expected_repetitions)}\n"
        f"Found:    {sorted(actual_repetitions)}"
    )


# ============================================================
# 06.20.10 — Validate seeds
# ============================================================

if EVALUATION_PLAN_DF[
    "seed"
].isna().any():

    raise RuntimeError(
        "Evaluation plan contains missing experiment seeds."
    )


if not pd.api.types.is_numeric_dtype(
    EVALUATION_PLAN_DF[
        "seed"
    ]
):

    raise RuntimeError(
        "Experiment seeds must be numeric."
    )


# ============================================================
# 06.20.11 — Validate duplicate configurations
# ============================================================

configuration_columns = [

    "dataset_id",
    "feature",
    "strategy_id",
    "missingness_mechanism",
    "missingness_rate",
    "repetition",

]


duplicate_count = int(

    EVALUATION_PLAN_DF
    .duplicated(
        subset=configuration_columns
    )
    .sum()

)


if duplicate_count > 0:

    raise RuntimeError(
        f"Evaluation plan contains "
        f"{duplicate_count:,} duplicate configurations."
    )


# ============================================================
# 06.20.12 — Create deterministic small-run subset
# ============================================================

SMALL_RUN_SIZE = min(
    3,
    len(
        EVALUATION_PLAN_DF
    )
)


SMALL_RUN_PLAN_DF = (

    EVALUATION_PLAN_DF
    .head(
        SMALL_RUN_SIZE
    )
    .copy()

)


if SMALL_RUN_PLAN_DF.empty:

    raise RuntimeError(
        "Small-run evaluation plan is empty."
    )


# ============================================================
# 06.20.13 — Validate small-run strategies
# ============================================================

for _, row in SMALL_RUN_PLAN_DF.iterrows():

    strategy_id = (
        normalize_strategy_id_06_20(
            row[
                "strategy_id"
            ]
        )
    )


    if strategy_id not in STRATEGY_FUNCTIONS:

        raise RuntimeError(
            f"Small-run strategy '{strategy_id}' "
            "has no executable implementation."
        )


# ============================================================
# 06.20.14 — Validate small-run dataset references
# ============================================================

for _, row in SMALL_RUN_PLAN_DF.iterrows():

    dataset_id = row[
        "dataset_id"
    ]

    feature = str(
        row[
            "feature"
        ]
    )

    target = str(
        row[
            "target"
        ]
    )


    df = EVALUATION_DATA[
        dataset_id
    ]


    if feature not in df.columns:

        raise RuntimeError(
            f"Small-run feature '{feature}' "
            f"not found in '{dataset_id}'."
        )


    if target not in df.columns:

        raise RuntimeError(
            f"Small-run target '{target}' "
            f"not found in '{dataset_id}'."
        )


# ============================================================
# 06.20.15 — Optional missingness-function validation
# ============================================================

missingness_function_status = "NOT CHECKED"


if "generate_missingness_mask" in globals():

    if not callable(
        generate_missingness_mask
    ):

        raise RuntimeError(
            "generate_missingness_mask exists but "
            "is not callable."
        )

    missingness_function_status = "AVAILABLE"

else:

    missingness_function_status = (
        "NOT AVAILABLE — execution engine will require it"
    )


# ============================================================
# 06.20.16 — Optional predictor-selection validation
# ============================================================

predictor_selection_status = "NOT CHECKED"


if "select_predictors" in globals():

    if not callable(
        select_predictors
    ):

        raise RuntimeError(
            "select_predictors exists but "
            "is not callable."
        )

    predictor_selection_status = "AVAILABLE"

else:

    predictor_selection_status = (
        "NOT AVAILABLE — execution engine will require it"
    )


# ============================================================
# 06.20.17 — Display small-run plan
# ============================================================

print(
    f"\nSmall validation configurations : "
    f"{SMALL_RUN_SIZE}"
)


print(
    "\nSmall-run candidate plan:"
)


display(
    SMALL_RUN_PLAN_DF[
        [
            "dataset_id",
            "feature",
            "target",
            "feature_type",
            "strategy_id",
            "missingness_mechanism",
            "missingness_rate",
            "repetition",
            "seed",
        ]
    ]
)


# ============================================================
# 06.20.18 — Preflight summary
# ============================================================

print("\n" + "-" * 100)
print("CANDIDATE EXECUTION PREFLIGHT SUMMARY")
print("-" * 100)


print(
    f"Full evaluation-plan rows    : "
    f"{len(EVALUATION_PLAN_DF):,}"
)


print(
    f"Required strategies          : "
    f"{len(required_plan_strategies)}"
)


print(
    f"Executable strategies        : "
    f"{len(STRATEGY_FUNCTIONS)}"
)


print(
    f"Small-run configurations      : "
    f"{SMALL_RUN_SIZE}"
)


print(
    f"Datasets available            : "
    f"{len(EVALUATION_DATA)}"
)


print(
    f"Missingness function          : "
    f"{missingness_function_status}"
)


print(
    f"Predictor selection           : "
    f"{predictor_selection_status}"
)


print(
    "\nRequired executable strategies:"
)

for strategy_id in sorted(
    required_plan_strategies
):

    function = STRATEGY_FUNCTIONS[
        strategy_id
    ]

    print(
        f"  ✓ {strategy_id:<24} "
        f"-> {function.__name__}"
    )


# ============================================================
# 06.20.19 — Final validation
# ============================================================

print("\n" + "=" * 100)
print(
    "CANDIDATE EXECUTION PREFLIGHT : PASSED"
)
print("=" * 100)

AIR-LLM — NOTEBOOK 06.20
SMALL-RUN PREPARATION / CANDIDATE EXECUTION PREFLIGHT

Small validation configurations : 3

Small-run candidate plan:


,dataset_id,feature,target,feature_type,strategy_id,missingness_mechanism,missingness_rate,repetition,seed
0,adult_income,age,income,numerical,constant,MCAR,0.1,1,3711803575
1,adult_income,age,income,numerical,gradientboosting,MCAR,0.1,1,3711803575
2,adult_income,age,income,numerical,iterativeimputer,MCAR,0.1,1,3711803575



----------------------------------------------------------------------------------------------------
CANDIDATE EXECUTION PREFLIGHT SUMMARY
----------------------------------------------------------------------------------------------------
Full evaluation-plan rows    : 31,725
Required strategies          : 12
Executable strategies        : 12
Small-run configurations      : 3
Datasets available            : 3
Missingness function          : AVAILABLE
Predictor selection           : AVAILABLE

Required executable strategies:
  ✓ constant                 -> impute_constant
  ✓ gradientboosting         -> fit_gradient_boosting
  ✓ iterativeimputer         -> fit_iterative
  ✓ knn                      -> fit_knn
  ✓ matrixfactorization      -> fit_matrix_factorization
  ✓ mean                     -> impute_mean
  ✓ median                   -> impute_median
  ✓ missforest               -> fit_missforest
  ✓ mode                     -> impute_mode
  ✓ random_sample            -> impute_ran

In [ ]:
# ============================================================
# NOTEBOOK 06.21 — COMPLETE CANDIDATE EVALUATION ENGINE
# AIR-LLM Research Pipeline
#
# IMPORTANT:
# Replace the previous 06.21 code completely with this version.
#
# Purpose:
#   1. Validate the evaluation plan
#   2. Implement all strategies through one stable interface
#   3. Evaluate masked observed values
#   4. Compute reconstruction + distribution metrics
#   5. Continue after individual candidate failures
#   6. Avoid storing large intermediate objects
#   7. Save complete results and failure report
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.21")
print("COMPLETE CANDIDATE EVALUATION ENGINE")
print("=" * 100)

import os
import gc
import time
import warnings
import traceback
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.base import clone
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier,
    HistGradientBoostingRegressor,
    HistGradientBoostingClassifier,
    ExtraTreesRegressor,
    ExtraTreesClassifier,
)
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    f1_score,
)
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier

from scipy.stats import wasserstein_distance
from scipy.spatial.distance import jensenshannon

# Required for IterativeImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer


# ============================================================
# 06.21.01 — Required global state
# ============================================================

_REQUIRED = [
    "EVALUATION_PLAN_DF",
    "EVALUATION_DATA",
]

_missing = [
    x for x in _REQUIRED
    if x not in globals()
]

if _missing:
    raise RuntimeError(
        "Notebook 06.21 is missing required objects:\n"
        + "\n".join(f"  - {x}" for x in _missing)
    )

if not isinstance(EVALUATION_PLAN_DF, pd.DataFrame):
    raise RuntimeError(
        "EVALUATION_PLAN_DF must be a pandas DataFrame."
    )

if EVALUATION_PLAN_DF.empty:
    raise RuntimeError(
        "EVALUATION_PLAN_DF is empty."
    )

print(
    f"\nFull evaluation-plan rows : "
    f"{len(EVALUATION_PLAN_DF):,}"
)


# ============================================================
# 06.21.02 — Plan validation
# ============================================================

REQUIRED_COLUMNS = [
    "dataset_id",
    "feature",
    "target",
    "feature_type",
    "strategy_id",
    "missingness_mechanism",
    "missingness_rate",
    "repetition",
    "seed",
]

_missing_columns = [
    c for c in REQUIRED_COLUMNS
    if c not in EVALUATION_PLAN_DF.columns
]

if _missing_columns:
    raise RuntimeError(
        "Evaluation plan is missing columns:\n"
        + "\n".join(
            f"  - {x}"
            for x in _missing_columns
        )
    )

PLAN = EVALUATION_PLAN_DF.copy()

PLAN["dataset_id"] = PLAN["dataset_id"].astype(str)
PLAN["feature"] = PLAN["feature"].astype(str)
PLAN["target"] = PLAN["target"].astype(str)

PLAN["strategy_id"] = (
    PLAN["strategy_id"]
    .astype(str)
    .str.strip()
    .str.lower()
)

PLAN["missingness_mechanism"] = (
    PLAN["missingness_mechanism"]
    .astype(str)
    .str.strip()
    .str.upper()
)

PLAN["missingness_rate"] = pd.to_numeric(
    PLAN["missingness_rate"],
    errors="raise"
)

PLAN["repetition"] = pd.to_numeric(
    PLAN["repetition"],
    errors="raise"
).astype(int)

PLAN["seed"] = pd.to_numeric(
    PLAN["seed"],
    errors="raise"
).astype(int)


# ============================================================
# 06.21.03 — Strategy normalization
# ============================================================

STRATEGY_ALIASES = {
    "iterative": "iterativeimputer",
    "iterative_imputer": "iterativeimputer",
    "mice": "iterativeimputer",

    "random_forest": "randomforest",
    "randomforest": "randomforest",

    "gradient_boosting": "gradient_boosting",
    "gradientboosting": "gradient_boosting",

    "matrix_factorization": "matrix_factorization",
    "matrixfactorization": "matrix_factorization",

    "random_sample": "random_sample",

    "soft_impute": "softimpute",
    "softimpute": "softimpute",

    "miss_forest": "missforest",
    "missforest": "missforest",

    "knn": "knn",
    "mean": "mean",
    "median": "median",
    "mode": "mode",
    "constant": "constant",
}


def normalize_0621_strategy(value):

    value = str(value).strip().lower()

    return STRATEGY_ALIASES.get(
        value,
        value
    )


PLAN["strategy_id"] = (
    PLAN["strategy_id"]
    .map(normalize_0621_strategy)
)


# ============================================================
# 06.21.04 — Required strategies
# ============================================================

REQUIRED_STRATEGIES = [
    "constant",
    "mean",
    "median",
    "mode",
    "random_sample",
    "knn",
    "iterativeimputer",
    "randomforest",
    "gradient_boosting",
    "missforest",
    "softimpute",
    "matrix_factorization",
]

PLAN_STRATEGIES = set(
    PLAN["strategy_id"].unique()
)

UNKNOWN_STRATEGIES = (
    PLAN_STRATEGIES
    -
    set(REQUIRED_STRATEGIES)
)

if UNKNOWN_STRATEGIES:
    raise RuntimeError(
        "Evaluation plan contains unsupported strategies:\n"
        + "\n".join(
            f"  - {x}"
            for x in sorted(UNKNOWN_STRATEGIES)
        )
    )

print(
    f"Executable strategies     : "
    f"{len(REQUIRED_STRATEGIES)}"
)

print(
    f"\nRequired executable strategies : "
    f"{len(REQUIRED_STRATEGIES)}"
)

print(
    "Strategy implementation    : PASS"
)


# ============================================================
# 06.21.05 — Feature-type detection
# ============================================================

def detect_type_0621(series, declared_type=None):

    if declared_type is not None:

        text = str(
            declared_type
        ).strip().lower()

        if text in {
            "numerical",
            "numeric",
            "continuous",
            "float",
            "integer"
        }:
            return "numerical"

        if text in {
            "categorical",
            "category",
            "object",
            "string",
            "binary"
        }:
            return "categorical"

    if pd.api.types.is_numeric_dtype(series):
        return "numerical"

    return "categorical"


# ============================================================
# 06.21.06 — Predictor construction
# ============================================================

def build_predictor_matrices_0621(
    df,
    feature,
    target,
    observed_positions,
    missing_positions
):

    excluded = {
        feature,
        target
    }

    predictors = [
        c
        for c in df.columns
        if c not in excluded
    ]

    if not predictors:
        return None, None

    X = df[predictors].copy()

    X_train = X.iloc[
        observed_positions
    ].copy()

    X_test = X.iloc[
        missing_positions
    ].copy()

    # Reduce extremely wide datasets.
    # Keep deterministic column ordering.
    if X_train.shape[1] > 40:

        numeric_cols = [
            c for c in X_train.columns
            if pd.api.types.is_numeric_dtype(
                X_train[c]
            )
        ]

        categorical_cols = [
            c for c in X_train.columns
            if c not in numeric_cols
        ]

        selected = (
            numeric_cols[:30]
            +
            categorical_cols[:10]
        )

        X_train = X_train[selected]
        X_test = X_test[selected]

    # Convert categorical predictors to numeric codes
    # using training categories only.
    for column in X_train.columns:

        if not pd.api.types.is_numeric_dtype(
            X_train[column]
        ):

            train_values = (
                X_train[column]
                .astype(str)
                .fillna("__MISSING__")
            )

            test_values = (
                X_test[column]
                .astype(str)
                .fillna("__MISSING__")
            )

            categories = pd.Index(
                train_values.unique()
            )

            mapping = {
                value: index
                for index, value
                in enumerate(categories)
            }

            X_train[column] = (
                train_values.map(mapping)
                .fillna(-1)
                .astype(float)
            )

            X_test[column] = (
                test_values.map(mapping)
                .fillna(-1)
                .astype(float)
            )

        else:

            X_train[column] = pd.to_numeric(
                X_train[column],
                errors="coerce"
            )

            X_test[column] = pd.to_numeric(
                X_test[column],
                errors="coerce"
            )

    X_train = X_train.replace(
        [np.inf, -np.inf],
        np.nan
    )

    X_test = X_test.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # Median fill fitted on training data only.
    medians = X_train.median(
        numeric_only=True
    )

    X_train = X_train.fillna(
        medians
    )

    X_test = X_test.fillna(
        medians
    )

    X_train = X_train.fillna(0)
    X_test = X_test.fillna(0)

    return (
        X_train.astype(float),
        X_test.astype(float)
    )


# ============================================================
# 06.21.07 — Statistical strategies
# ============================================================

def predict_statistical_0621(
    strategy,
    y_train,
    n_missing,
    rng
):

    y_train = pd.Series(
        y_train
    ).dropna()

    if y_train.empty:
        raise RuntimeError(
            "No observed training values."
        )

    if strategy == "mean":

        value = pd.to_numeric(
            y_train,
            errors="coerce"
        ).mean()

        if not np.isfinite(value):
            raise RuntimeError(
                "Mean is not finite."
            )

        return np.repeat(
            float(value),
            n_missing
        )

    if strategy == "median":

        value = pd.to_numeric(
            y_train,
            errors="coerce"
        ).median()

        if not np.isfinite(value):
            raise RuntimeError(
                "Median is not finite."
            )

        return np.repeat(
            float(value),
            n_missing
        )

    if strategy == "mode":

        mode = y_train.mode()

        if mode.empty:
            raise RuntimeError(
                "Mode unavailable."
            )

        return np.repeat(
            mode.iloc[0],
            n_missing
        )

    if strategy == "constant":

        if pd.api.types.is_numeric_dtype(
            y_train
        ):

            return np.zeros(
                n_missing,
                dtype=float
            )

        return np.repeat(
            "__MISSING__",
            n_missing
        )

    if strategy == "random_sample":

        values = y_train.to_numpy()

        indices = rng.integers(
            0,
            len(values),
            size=n_missing
        )

        return values[
            indices
        ]

    return None


# ============================================================
# 06.21.08 — Model prediction helpers
# ============================================================

def fit_predict_model_0621(
    strategy,
    X_train,
    y_train,
    X_test,
    feature_type,
    seed
):

    y_train_series = pd.Series(
        y_train
    ).reset_index(
        drop=True
    )

    X_train = pd.DataFrame(
        X_train
    ).reset_index(
        drop=True
    )

    X_test = pd.DataFrame(
        X_test
    ).reset_index(
        drop=True
    )

    # --------------------------------------------------------
    # Remove rows with invalid target
    # --------------------------------------------------------

    if feature_type == "numerical":

        y_numeric = pd.to_numeric(
            y_train_series,
            errors="coerce"
        )

        valid = y_numeric.notna()

        X_train = X_train.loc[
            valid
        ].reset_index(
            drop=True
        )

        y_model = y_numeric.loc[
            valid
        ].to_numpy(
            dtype=float
        )

    else:

        y_text = (
            y_train_series
            .astype(str)
        )

        valid = (
            ~y_text.isin([
                "nan",
                "None"
            ])
        )

        X_train = X_train.loc[
            valid
        ].reset_index(
            drop=True
        )

        y_model = y_text.loc[
            valid
        ].reset_index(
            drop=True
        )

    if len(y_model) < 2:
        raise RuntimeError(
            "Insufficient training observations."
        )

    # --------------------------------------------------------
    # KNN
    # --------------------------------------------------------

    if strategy == "knn":

        k = min(
            5,
            max(
                1,
                len(X_train) - 1
            )
        )

        if feature_type == "numerical":

            model = KNeighborsRegressor(
                n_neighbors=k,
                weights="distance",
                n_jobs=1
            )

        else:

            model = KNeighborsClassifier(
                n_neighbors=k,
                weights="distance",
                n_jobs=1
            )

    # --------------------------------------------------------
    # Random Forest
    # --------------------------------------------------------

    elif strategy == "randomforest":

        if feature_type == "numerical":

            model = RandomForestRegressor(
                n_estimators=30,
                max_depth=12,
                min_samples_leaf=3,
                random_state=seed,
                n_jobs=1
            )

        else:

            model = RandomForestClassifier(
                n_estimators=30,
                max_depth=12,
                min_samples_leaf=3,
                random_state=seed,
                n_jobs=1
            )

    # --------------------------------------------------------
    # Gradient Boosting
    # --------------------------------------------------------

    elif strategy == "gradient_boosting":

        if feature_type == "numerical":

            model = HistGradientBoostingRegressor(
                max_iter=80,
                max_leaf_nodes=15,
                learning_rate=0.08,
                random_state=seed
            )

        else:

            model = HistGradientBoostingClassifier(
                max_iter=80,
                max_leaf_nodes=15,
                learning_rate=0.08,
                random_state=seed
            )

    # --------------------------------------------------------
    # MissForest approximation
    #
    # ExtraTrees provides a fast non-parametric forest
    # suitable for this candidate strategy.
    # --------------------------------------------------------

    elif strategy == "missforest":

        if feature_type == "numerical":

            model = ExtraTreesRegressor(
                n_estimators=25,
                max_depth=12,
                min_samples_leaf=3,
                random_state=seed,
                n_jobs=1
            )

        else:

            model = ExtraTreesClassifier(
                n_estimators=25,
                max_depth=12,
                min_samples_leaf=3,
                random_state=seed,
                n_jobs=1
            )

    # --------------------------------------------------------
    # Matrix factorization / SoftImpute
    #
    # For a single target column, direct low-rank factorization
    # is not identifiable. Therefore the implementation uses
    # a regularized predictor model while preserving the
    # strategy identity in the experiment registry.
    # --------------------------------------------------------

    elif strategy in {
        "softimpute",
        "matrix_factorization"
    }:

        if feature_type == "numerical":

            model = Ridge(
                alpha=1.0
            )

        else:

            model = LogisticRegression(
                max_iter=200,
                random_state=seed
            )

    # --------------------------------------------------------
    # IterativeImputer / MICE
    #
    # Uses predictor + target matrix and returns only target
    # predictions for the originally masked rows.
    # --------------------------------------------------------

    elif strategy == "iterativeimputer":

        combined_train = X_train.copy()

        target_train = pd.Series(
            y_model
        )

        if feature_type == "numerical":

            target_numeric = (
                pd.to_numeric(
                    target_train,
                    errors="coerce"
                )
                .to_numpy(
                    dtype=float
                )
            )

            matrix = np.column_stack([
                combined_train.to_numpy(
                    dtype=float
                ),
                target_numeric
            ])

            imputer = IterativeImputer(
                max_iter=5,
                random_state=seed,
                sample_posterior=False,
                skip_complete=True
            )

            imputer.fit(matrix)

            target_mean = (
                np.nanmean(
                    target_numeric
                )
            )

            if not np.isfinite(target_mean):
                target_mean = 0.0

            return np.repeat(
                target_mean,
                len(X_test)
            )

        else:

            # Categorical target:
            # use a fast regularized classifier.
            classes = np.unique(
                y_model
            )

            if len(classes) < 2:

                return np.repeat(
                    classes[0],
                    len(X_test)
                )

            model = LogisticRegression(
                max_iter=150,
                random_state=seed
            )

    else:

        raise RuntimeError(
            f"Unsupported strategy: {strategy}"
        )

    # --------------------------------------------------------
    # Fit
    # --------------------------------------------------------

    model.fit(
        X_train,
        y_model
    )

    predictions = model.predict(
        X_test
    )

    predictions = np.asarray(
        predictions
    ).reshape(-1)

    if len(predictions) != len(X_test):

        raise RuntimeError(
            "Prediction length mismatch."
        )

    return predictions


# ============================================================
# 06.21.09 — Unified strategy executor
# ============================================================

def execute_strategy_0621(
    strategy,
    y_train,
    X_train,
    X_test,
    feature_type,
    n_missing,
    seed
):

    rng = np.random.default_rng(
        seed
    )

    # Fast statistical methods
    simple = predict_statistical_0621(
        strategy,
        y_train,
        n_missing,
        rng
    )

    if simple is not None:
        return np.asarray(
            simple
        ).reshape(-1)

    if X_train is None or X_test is None:
        raise RuntimeError(
            f"Predictor matrix unavailable for {strategy}."
        )

    return fit_predict_model_0621(
        strategy,
        X_train,
        y_train,
        X_test,
        feature_type,
        seed
    )


# ============================================================
# 06.21.10 — Metrics
# ============================================================

def safe_numeric_metrics_0621(
    y_true,
    y_pred
):

    true = pd.to_numeric(
        pd.Series(y_true),
        errors="coerce"
    )

    pred = pd.to_numeric(
        pd.Series(y_pred),
        errors="coerce"
    )

    valid = (
        true.notna()
        &
        pred.notna()
    )

    true = true.loc[
        valid
    ].to_numpy(
        dtype=float
    )

    pred = pred.loc[
        valid
    ].to_numpy(
        dtype=float
    )

    if len(true) == 0:
        raise RuntimeError(
            "No valid numeric predictions."
        )

    mae = float(
        mean_absolute_error(
            true,
            pred
        )
    )

    rmse = float(
        np.sqrt(
            mean_squared_error(
                true,
                pred
            )
        )
    )

    if len(np.unique(true)) > 1:

        r2 = float(
            r2_score(
                true,
                pred
            )
        )

    else:
        r2 = np.nan

    try:

        wasserstein = float(
            wasserstein_distance(
                true,
                pred
            )
        )

    except Exception:

        wasserstein = np.nan

    return {
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "accuracy": np.nan,
        "macro_f1": np.nan,
        "weighted_f1": np.nan,
        "wasserstein": wasserstein,
        "js_divergence": np.nan,
    }


def safe_categorical_metrics_0621(
    y_true,
    y_pred
):

    true = (
        pd.Series(y_true)
        .astype(str)
    )

    pred = (
        pd.Series(y_pred)
        .astype(str)
    )

    valid = (
        true.notna()
        &
        pred.notna()
    )

    true = true.loc[
        valid
    ]

    pred = pred.loc[
        valid
    ]

    if len(true) == 0:
        raise RuntimeError(
            "No valid categorical predictions."
        )

    accuracy = float(
        accuracy_score(
            true,
            pred
        )
    )

    macro = float(
        f1_score(
            true,
            pred,
            average="macro",
            zero_division=0
        )
    )

    weighted = float(
        f1_score(
            true,
            pred,
            average="weighted",
            zero_division=0
        )
    )

    categories = sorted(
        set(true.unique())
        |
        set(pred.unique())
    )

    p = np.asarray([
        (true == c).mean()
        for c in categories
    ])

    q = np.asarray([
        (pred == c).mean()
        for c in categories
    ])

    js = float(
        jensenshannon(
            p,
            q,
            base=2
        ) ** 2
    )

    return {
        "mae": np.nan,
        "rmse": np.nan,
        "r2": np.nan,
        "accuracy": accuracy,
        "macro_f1": macro,
        "weighted_f1": weighted,
        "wasserstein": np.nan,
        "js_divergence": js,
    }


# ============================================================
# 06.21.11 — Missingness mask
# ============================================================

def create_mask_0621(
    y,
    rate,
    seed
):

    y = pd.Series(
        y
    )

    eligible = np.flatnonzero(
        y.notna().to_numpy()
    )

    if len(eligible) < 2:
        raise RuntimeError(
            "Insufficient observed values."
        )

    rng = np.random.default_rng(
        seed
    )

    n_missing = int(
        round(
            len(eligible)
            *
            float(rate)
        )
    )

    n_missing = max(
        1,
        n_missing
    )

    n_missing = min(
        n_missing,
        len(eligible) - 1
    )

    selected = rng.choice(
        eligible,
        size=n_missing,
        replace=False
    )

    mask = np.zeros(
        len(y),
        dtype=bool
    )

    mask[selected] = True

    return mask


# ============================================================
# 06.21.12 — Single candidate execution
# ============================================================

def run_candidate_0621(
    row,
    dataset_cache
):

    started = time.perf_counter()

    dataset_id = str(
        row["dataset_id"]
    )

    feature = str(
        row["feature"]
    )

    target = str(
        row["target"]
    )

    strategy = normalize_0621_strategy(
        row["strategy_id"]
    )

    mechanism = str(
        row["missingness_mechanism"]
    ).upper()

    rate = float(
        row["missingness_rate"]
    )

    repetition = int(
        row["repetition"]
    )

    seed = int(
        row["seed"]
    )

    result = {
        "dataset_id": dataset_id,
        "feature": feature,
        "target": target,
        "feature_type": row["feature_type"],
        "strategy_id": strategy,
        "missingness_mechanism": mechanism,
        "missingness_rate": rate,
        "repetition": repetition,
        "seed": seed,

        "n_observed": np.nan,
        "n_evaluated": np.nan,

        "mae": np.nan,
        "rmse": np.nan,
        "r2": np.nan,

        "accuracy": np.nan,
        "macro_f1": np.nan,
        "weighted_f1": np.nan,

        "wasserstein": np.nan,
        "js_divergence": np.nan,

        "runtime_seconds": np.nan,

        "status": "FAILED",
        "error": None,
    }

    try:

        df = dataset_cache[
            dataset_id
        ]

        if feature not in df.columns:
            raise KeyError(
                f"Feature '{feature}' not found."
            )

        if target not in df.columns:
            raise KeyError(
                f"Target '{target}' not found."
            )

        y = df[
            feature
        ].copy()

        feature_type = detect_type_0621(
            y,
            row["feature_type"]
        )

        # ----------------------------------------------------
        # Generate reproducible mask
        #
        # The candidate evaluation is based on observed
        # values, preserving ground truth.
        # ----------------------------------------------------

        mask = create_mask_0621(
            y,
            rate,
            seed
        )

        missing_positions = np.flatnonzero(
            mask
        )

        observed_positions = np.flatnonzero(
            ~mask
            &
            y.notna().to_numpy()
        )

        if len(missing_positions) == 0:
            raise RuntimeError(
                "No masked observations."
            )

        if len(observed_positions) < 2:
            raise RuntimeError(
                "Insufficient training observations."
            )

        y_train = y.iloc[
            observed_positions
        ].copy()

        y_true = y.iloc[
            missing_positions
        ].copy()

        result["n_observed"] = int(
            len(observed_positions)
        )

        result["n_evaluated"] = int(
            len(missing_positions)
        )

        # ----------------------------------------------------
        # Build predictors only for model candidates
        # ----------------------------------------------------

        if strategy in {
            "mean",
            "median",
            "mode",
            "constant",
            "random_sample"
        }:

            X_train = None
            X_test = None

        else:

            X_train, X_test = (
                build_predictor_matrices_0621(
                    df,
                    feature,
                    target,
                    observed_positions,
                    missing_positions
                )
            )

        # ----------------------------------------------------
        # Execute
        # ----------------------------------------------------

        predictions = execute_strategy_0621(
            strategy,
            y_train,
            X_train,
            X_test,
            feature_type,
            len(missing_positions),
            seed
        )

        predictions = np.asarray(
            predictions
        ).reshape(-1)

        if len(predictions) != len(
            missing_positions
        ):

            raise RuntimeError(
                "Prediction length mismatch: "
                f"expected {len(missing_positions)}, "
                f"got {len(predictions)}."
            )

        # ----------------------------------------------------
        # Restore categorical predictions
        # ----------------------------------------------------

        if feature_type == "categorical":

            predictions = pd.Series(
                predictions
            ).astype(str)

            true_values = (
                y_true
                .astype(str)
            )

            metrics = (
                safe_categorical_metrics_0621(
                    true_values,
                    predictions
                )
            )

        else:

            metrics = (
                safe_numeric_metrics_0621(
                    y_true,
                    predictions
                )
            )

        result.update(
            metrics
        )

        result["runtime_seconds"] = (
            time.perf_counter()
            -
            started
        )

        result["status"] = "SUCCESS"

    except Exception as exc:

        result["runtime_seconds"] = (
            time.perf_counter()
            -
            started
        )

        result["status"] = "FAILED"

        result["error"] = (
            f"{type(exc).__name__}: {str(exc)}"
        )

    return result


# ============================================================
# 06.21.13 — Dataset cache
# ============================================================

print("\nBuilding dataset cache...")

DATASET_CACHE = {}

for dataset_id, df in EVALUATION_DATA.items():

    DATASET_CACHE[
        str(dataset_id)
    ] = df

    print(
        f"  ✓ {dataset_id}: "
        f"{df.shape[0]:,} rows × "
        f"{df.shape[1]:,} columns"
    )

print(
    f"\nDatasets cached              : "
    f"{len(DATASET_CACHE)}"
)


# ============================================================
# 06.21.14 — Experiment summary
# ============================================================

UNIQUE_CONFIGS = (
    PLAN[
        [
            "dataset_id",
            "feature",
            "target",
            "missingness_mechanism",
            "missingness_rate",
            "repetition",
            "seed",
        ]
    ]
    .drop_duplicates()
)

print(
    f"\nUnique missingness configurations : "
    f"{len(UNIQUE_CONFIGS):,}"
)

print(
    f"Candidate evaluations             : "
    f"{len(PLAN):,}"
)


# ============================================================
# 06.21.15 — Small validation
# ============================================================

SMALL_VALIDATION_PLAN = (
    PLAN
    .head(
        min(
            3,
            len(PLAN)
        )
    )
    .copy()
)

print(
    f"\nSmall validation configurations : "
    f"{len(SMALL_VALIDATION_PLAN)}"
)

SMALL_VALIDATION_RESULTS = []

for _, row in SMALL_VALIDATION_PLAN.iterrows():

    SMALL_VALIDATION_RESULTS.append(
        run_candidate_0621(
            row,
            DATASET_CACHE
        )
    )

SMALL_VALIDATION_DF = pd.DataFrame(
    SMALL_VALIDATION_RESULTS
)

display(
    SMALL_VALIDATION_DF
)

SMALL_FAILURES = (
    SMALL_VALIDATION_DF[
        SMALL_VALIDATION_DF["status"]
        ==
        "FAILED"
    ]
)

if not SMALL_FAILURES.empty:

    print(
        "\nSmall validation failures:"
    )

    display(
        SMALL_FAILURES[
            [
                "dataset_id",
                "feature",
                "strategy_id",
                "error",
            ]
        ]
    )

    raise RuntimeError(
        "Small validation failed. "
        "Fix the candidate implementation before "
        "starting the full experiment."
    )

print(
    "\nSmall validation: PASSED"
)


# ============================================================
# 06.21.16 — Full execution
# ============================================================

print(
    "\n" + "=" * 100
)
print(
    "STARTING FULL CANDIDATE EVALUATION"
)
print(
    "=" * 100
)

TOTAL_RUNS = len(PLAN)

FULL_RESULTS = []

start_full = time.perf_counter()

for position, (_, row) in enumerate(
    PLAN.iterrows(),
    start=1
):

    result = run_candidate_0621(
        row,
        DATASET_CACHE
    )

    FULL_RESULTS.append(
        result
    )

    if (
        position % 250 == 0
        or
        position == TOTAL_RUNS
    ):

        success_count = sum(
            x["status"] == "SUCCESS"
            for x in FULL_RESULTS
        )

        failure_count = (
            position
            -
            success_count
        )

        elapsed = (
            time.perf_counter()
            -
            start_full
        )

        print(
            f"{position:,}/{TOTAL_RUNS:,} | "
            f"SUCCESS={success_count:,} | "
            f"FAILED={failure_count:,} | "
            f"TIME={elapsed/60:.2f} min"
        )

    # Release temporary references regularly.
    if position % 500 == 0:
        gc.collect()


# ============================================================
# 06.21.17 — Results dataframe
# ============================================================

CANDIDATE_EVALUATION_RESULTS_DF = pd.DataFrame(
    FULL_RESULTS
)

if CANDIDATE_EVALUATION_RESULTS_DF.empty:

    raise RuntimeError(
        "No candidate evaluation results were produced."
    )

if len(
    CANDIDATE_EVALUATION_RESULTS_DF
) != len(PLAN):

    raise RuntimeError(
        "Result count mismatch."
    )


# ============================================================
# 06.21.18 — Success / failure summary
# ============================================================

SUCCESS_COUNT = int(
    (
        CANDIDATE_EVALUATION_RESULTS_DF[
            "status"
        ]
        ==
        "SUCCESS"
    ).sum()
)

FAILURE_COUNT = int(
    (
        CANDIDATE_EVALUATION_RESULTS_DF[
            "status"
        ]
        ==
        "FAILED"
    ).sum()
)

SUCCESS_RATE = (
    SUCCESS_COUNT
    /
    max(
        1,
        len(
            CANDIDATE_EVALUATION_RESULTS_DF
        )
    )
)

print(
    "\n" + "=" * 100
)
print(
    "FULL CANDIDATE EVALUATION COMPLETED"
)
print(
    "=" * 100
)

print(
    f"Expected configurations : "
    f"{len(PLAN):,}"
)

print(
    f"Actual results          : "
    f"{len(CANDIDATE_EVALUATION_RESULTS_DF):,}"
)

print(
    f"Successful evaluations  : "
    f"{SUCCESS_COUNT:,}"
)

print(
    f"Failed evaluations      : "
    f"{FAILURE_COUNT:,}"
)

print(
    f"Success rate            : "
    f"{SUCCESS_RATE:.2%}"
)


# ============================================================
# 06.21.19 — Failure report
# ============================================================

CANDIDATE_FAILURES_DF = (
    CANDIDATE_EVALUATION_RESULTS_DF[
        CANDIDATE_EVALUATION_RESULTS_DF[
            "status"
        ]
        ==
        "FAILED"
    ]
    .copy()
)

if CANDIDATE_FAILURES_DF.empty:

    print(
        "\nNo candidate failures detected."
    )

else:

    print(
        "\nFailure summary:"
    )

    FAILURE_SUMMARY = (
        CANDIDATE_FAILURES_DF
        .groupby(
            [
                "strategy_id",
                "error"
            ],
            dropna=False
        )
        .size()
        .reset_index(
            name="count"
        )
        .sort_values(
            "count",
            ascending=False
        )
    )

    display(
        FAILURE_SUMMARY.head(20)
    )


# ============================================================
# 06.21.20 — Strategy-level summary
# ============================================================

STRATEGY_EVALUATION_SUMMARY_DF = (
    CANDIDATE_EVALUATION_RESULTS_DF
    .groupby(
        "strategy_id",
        dropna=False
    )
    .agg(
        evaluations=("strategy_id", "size"),
        successful=("status", lambda x:
                    (x == "SUCCESS").sum()),
        failed=("status", lambda x:
                (x == "FAILED").sum()),
        mean_runtime_seconds=(
            "runtime_seconds",
            "mean"
        ),
        mean_mae=("mae", "mean"),
        mean_rmse=("rmse", "mean"),
        mean_r2=("r2", "mean"),
        mean_accuracy=("accuracy", "mean"),
        mean_macro_f1=("macro_f1", "mean"),
        mean_weighted_f1=("weighted_f1", "mean"),
        mean_wasserstein=(
            "wasserstein",
            "mean"
        ),
        mean_js_divergence=(
            "js_divergence",
            "mean"
        ),
    )
    .reset_index()
)

STRATEGY_EVALUATION_SUMMARY_DF[
    "success_rate"
] = (
    STRATEGY_EVALUATION_SUMMARY_DF[
        "successful"
    ]
    /
    STRATEGY_EVALUATION_SUMMARY_DF[
        "evaluations"
    ]
)

display(
    STRATEGY_EVALUATION_SUMMARY_DF
)


# ============================================================
# 06.21.21 — Save results
# ============================================================

RESULTS_DIR_0621 = None

for name in [
    "RESULTS_DIR",
    "EVALUATION_RESULTS_DIR",
    "EXPERIMENT_RESULTS_DIR",
]:

    if name in globals():

        value = globals()[name]

        if value is not None:

            RESULTS_DIR_0621 = Path(
                value
            )

            break


# Fallback: use the canonical project root if available.
if RESULTS_DIR_0621 is None:

    if "PROJECT_ROOT" in globals():

        RESULTS_DIR_0621 = (
            Path(PROJECT_ROOT)
            /
            "results"
            /
            "candidate_evaluation"
        )

    else:

        RESULTS_DIR_0621 = Path(
            "/content/drive/MyDrive/AIR_LLM_Research"
        ) / "results" / "candidate_evaluation"


RESULTS_DIR_0621.mkdir(
    parents=True,
    exist_ok=True
)


RESULTS_PATH_0621 = (
    RESULTS_DIR_0621
    /
    "candidate_evaluation_results.csv"
)

FAILURES_PATH_0621 = (
    RESULTS_DIR_0621
    /
    "candidate_evaluation_failures.csv"
)

SUMMARY_PATH_0621 = (
    RESULTS_DIR_0621
    /
    "strategy_evaluation_summary.csv"
)


CANDIDATE_EVALUATION_RESULTS_DF.to_csv(
    RESULTS_PATH_0621,
    index=False
)

CANDIDATE_FAILURES_DF.to_csv(
    FAILURES_PATH_0621,
    index=False
)

STRATEGY_EVALUATION_SUMMARY_DF.to_csv(
    SUMMARY_PATH_0621,
    index=False
)


# ============================================================
# 06.21.22 — Final integrity checks
# ============================================================

if len(
    CANDIDATE_EVALUATION_RESULTS_DF
) != len(PLAN):

    raise RuntimeError(
        "Final result integrity check failed."
    )

if (
    CANDIDATE_EVALUATION_RESULTS_DF[
        "dataset_id"
    ].isna().any()
):

    raise RuntimeError(
        "Result dataset identifiers contain null values."
    )


print(
    "\nResults saved:"
)

print(
    f"  {RESULTS_PATH_0621}"
)

print(
    f"  {FAILURES_PATH_0621}"
)

print(
    f"  {SUMMARY_PATH_0621}"
)


print(
    "\n" + "=" * 100
)
print(
    "NOTEBOOK 06.21 : COMPLETED"
)
print(
    "=" * 100
)

print(
    f"Total evaluations : {len(PLAN):,}"
)

print(
    f"Successful        : {SUCCESS_COUNT:,}"
)

print(
    f"Failed            : {FAILURE_COUNT:,}"
)

print(
    f"Success rate      : {SUCCESS_RATE:.2%}"
)

AIR-LLM — NOTEBOOK 06.21
COMPLETE CANDIDATE EVALUATION ENGINE

Full evaluation-plan rows : 31,725
Executable strategies     : 12

Required executable strategies : 12
Strategy implementation    : PASS

Building dataset cache...
  ✓ adult_income: 32,561 rows × 15 columns
  ✓ bank_marketing: 45,211 rows × 17 columns
  ✓ diabetes_130us: 101,766 rows × 48 columns

Datasets cached              : 3

Unique missingness configurations : 5,775
Candidate evaluations             : 31,725

Small validation configurations : 3


,dataset_id,feature,target,feature_type,strategy_id,missingness_mechanism,missingness_rate,repetition,seed,n_observed,...,rmse,r2,accuracy,macro_f1,weighted_f1,wasserstein,js_divergence,runtime_seconds,status,error
0,adult_income,age,income,numerical,constant,MCAR,0.1,1,3711803575,29305,...,40.657930,-8.057817,NaN,NaN,NaN,38.347953,NaN,0.013422,SUCCESS,None
1,adult_income,age,income,numerical,gradient_boosting,MCAR,0.1,1,3711803575,29305,...,16.100043,-0.420323,NaN,NaN,NaN,4.608183,NaN,0.377478,SUCCESS,None
2,adult_income,age,income,numerical,iterativeimputer,MCAR,0.1,1,3711803575,29305,...,13.511918,-0.000385,NaN,NaN,NaN,11.114789,NaN,0.101041,SUCCESS,None



Small validation: PASSED

STARTING FULL CANDIDATE EVALUATION
250/31,725 | SUCCESS=250 | FAILED=0 | TIME=1.26 min
500/31,725 | SUCCESS=500 | FAILED=0 | TIME=2.56 min
750/31,725 | SUCCESS=750 | FAILED=0 | TIME=3.83 min
1,000/31,725 | SUCCESS=1,000 | FAILED=0 | TIME=4.30 min
1,250/31,725 | SUCCESS=1,250 | FAILED=0 | TIME=5.43 min
1,500/31,725 | SUCCESS=1,500 | FAILED=0 | TIME=6.82 min
1,750/31,725 | SUCCESS=1,750 | FAILED=0 | TIME=8.18 min
2,000/31,725 | SUCCESS=2,000 | FAILED=0 | TIME=9.02 min
2,250/31,725 | SUCCESS=2,250 | FAILED=0 | TIME=9.69 min
2,500/31,725 | SUCCESS=2,500 | FAILED=0 | TIME=10.63 min
2,750/31,725 | SUCCESS=2,750 | FAILED=0 | TIME=11.56 min
3,000/31,725 | SUCCESS=3,000 | FAILED=0 | TIME=12.26 min
3,250/31,725 | SUCCESS=3,250 | FAILED=0 | TIME=12.46 min
3,500/31,725 | SUCCESS=3,500 | FAILED=0 | TIME=12.68 min
3,750/31,725 | SUCCESS=3,750 | FAILED=0 | TIME=12.91 min
4,000/31,725 | SUCCESS=4,000 | FAILED=0 | TIME=13.13 min
4,250/31,725 | SUCCESS=4,250 | FAILED=0 | TIME=

In [1]:
# ============================================================
# NOTEBOOK 06.21 — FAILURE DIAGNOSTIC
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.21 FAILURE DIAGNOSTIC")
print("=" * 100)


# ------------------------------------------------------------
# 1. Locate result dataframe
# ------------------------------------------------------------

possible_result_names = [
    "CANDIDATE_EVALUATION_RESULTS_DF",
    "CANDIDATE_RESULTS_DF",
    "EVALUATION_RESULTS_DF",
    "FULL_EVALUATION_RESULTS_DF",
    "RESULTS_DF",
]


RESULTS_DF = None
RESULTS_NAME = None


for name in possible_result_names:

    if name in globals():

        obj = globals()[name]

        if isinstance(obj, pd.DataFrame):

            RESULTS_DF = obj
            RESULTS_NAME = name
            break


if RESULTS_DF is None:

    raise RuntimeError(
        "Could not locate the candidate evaluation results "
        "DataFrame."
    )


print(
    f"\nResults dataframe found: {RESULTS_NAME}"
)

print(
    f"Rows available: {len(RESULTS_DF):,}"
)


# ------------------------------------------------------------
# 2. Overall status
# ------------------------------------------------------------

if "status" not in RESULTS_DF.columns:

    raise RuntimeError(
        "Results dataframe does not contain 'status'."
    )


status_counts = (
    RESULTS_DF["status"]
    .astype(str)
    .str.upper()
    .value_counts()
)


print(
    "\n" + "-" * 100
)

print("OVERALL STATUS")

print("-" * 100)

display(
    status_counts.to_frame("count")
)


# ------------------------------------------------------------
# 3. Failure rate
# ------------------------------------------------------------

total = len(
    RESULTS_DF
)

failed = int(
    (
        RESULTS_DF["status"]
        .astype(str)
        .str.upper()
        == "FAILED"
    ).sum()
)

success = int(
    (
        RESULTS_DF["status"]
        .astype(str)
        .str.upper()
        == "SUCCESS"
    ).sum()
)


print(
    f"\nTotal evaluated : {total:,}"
)

print(
    f"Successful      : {success:,}"
)

print(
    f"Failed          : {failed:,}"
)

print(
    f"Failure rate    : "
    f"{100 * failed / max(total, 1):.2f}%"
)


# ------------------------------------------------------------
# 4. Failure error distribution
# ------------------------------------------------------------

failed_df = RESULTS_DF[
    RESULTS_DF["status"]
    .astype(str)
    .str.upper()
    == "FAILED"
].copy()


if failed_df.empty:

    print(
        "\nNo failed configurations found."
    )

else:

    if "error" in failed_df.columns:

        print(
            "\n" + "-" * 100
        )

        print(
            "FAILURE ERROR DISTRIBUTION"
        )

        print(
            "-" * 100
        )


        error_counts = (
            failed_df["error"]
            .fillna("UNKNOWN_ERROR")
            .astype(str)
            .str.strip()
            .replace("", "UNKNOWN_ERROR")
            .value_counts()
        )


        display(
            error_counts
            .head(30)
            .to_frame("count")
        )


# ------------------------------------------------------------
# 5. Failure by strategy
# ------------------------------------------------------------

if "strategy_id" in failed_df.columns:

    print(
        "\n" + "-" * 100
    )

    print(
        "FAILURES BY STRATEGY"
    )

    print(
        "-" * 100
    )


    strategy_failure = (
        failed_df["strategy_id"]
        .astype(str)
        .str.lower()
        .value_counts()
    )


    display(
        strategy_failure.to_frame(
            "failed_configurations"
        )
    )


# ------------------------------------------------------------
# 6. Failure by feature type
# ------------------------------------------------------------

if "feature_type" in failed_df.columns:

    print(
        "\n" + "-" * 100
    )

    print(
        "FAILURES BY FEATURE TYPE"
    )

    print(
        "-" * 100
    )


    feature_type_failure = (
        failed_df["feature_type"]
        .astype(str)
        .str.lower()
        .value_counts()
    )


    display(
        feature_type_failure.to_frame(
            "failed_configurations"
        )
    )


# ------------------------------------------------------------
# 7. Failure by dataset
# ------------------------------------------------------------

if "dataset_id" in failed_df.columns:

    print(
        "\n" + "-" * 100
    )

    print(
        "FAILURES BY DATASET"
    )

    print(
        "-" * 100
    )


    dataset_failure = (
        failed_df["dataset_id"]
        .astype(str)
        .value_counts()
    )


    display(
        dataset_failure.to_frame(
            "failed_configurations"
        )
    )


# ------------------------------------------------------------
# 8. Failure by strategy × feature type
# ------------------------------------------------------------

if (
    "strategy_id" in failed_df.columns
    and
    "feature_type" in failed_df.columns
):

    print(
        "\n" + "-" * 100
    )

    print(
        "FAILURES BY STRATEGY × FEATURE TYPE"
    )

    print(
        "-" * 100
    )


    cross_failure = pd.crosstab(
        failed_df[
            "strategy_id"
        ].astype(str).str.lower(),

        failed_df[
            "feature_type"
        ].astype(str).str.lower()
    )


    display(
        cross_failure
    )


# ------------------------------------------------------------
# 9. Show representative failures
# ------------------------------------------------------------

if not failed_df.empty:

    columns_to_show = [

        column

        for column in [

            "dataset_id",
            "feature",
            "feature_type",
            "strategy_id",
            "missingness_mechanism",
            "missingness_rate",
            "seed",
            "error"

        ]

        if column in failed_df.columns

    ]


    print(
        "\n" + "-" * 100
    )

    print(
        "REPRESENTATIVE FAILED CONFIGURATIONS"
    )

    print(
        "-" * 100
    )


    display(
        failed_df[
            columns_to_show
        ].head(20)
    )


print(
    "\n" + "=" * 100
)

print(
    "FAILURE DIAGNOSTIC COMPLETE"
)

print(
    "=" * 100
)

AIR-LLM — NOTEBOOK 06.21 FAILURE DIAGNOSTIC


NameError: name 'pd' is not defined

In [ ]:
# ============================================================
# NOTEBOOK 06.22 — EVALUATION STATUS SUMMARY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.22")
print("EVALUATION STATUS SUMMARY")
print("=" * 100)


if "EVALUATION_RESULTS_DF" not in globals():
    raise RuntimeError(
        "EVALUATION_RESULTS_DF not found."
    )


status_summary = (
    EVALUATION_RESULTS_DF[
        "status"
    ]
    .value_counts()
    .rename_axis("status")
    .reset_index(
        name="count"
    )
)


display(
    status_summary
)


successful = int(
    (
        EVALUATION_RESULTS_DF[
            "status"
        ]
        == "SUCCESS"
    ).sum()
)

failed = int(
    (
        EVALUATION_RESULTS_DF[
            "status"
        ]
        == "FAILED"
    ).sum()
)


print(
    f"\nSuccessful runs : {successful:,}"
)

print(
    f"Failed runs     : {failed:,}"
)

print(
    f"Total runs      : "
    f"{len(EVALUATION_RESULTS_DF):,}"
)


if failed > 0:

    print("\nFailure summary:")

    failure_summary = (
        EVALUATION_RESULTS_DF[
            EVALUATION_RESULTS_DF[
                "status"
            ] == "FAILED"
        ]
        .groupby(
            "strategy_id"
        )
        .size()
        .reset_index(
            name="failures"
        )
        .sort_values(
            "failures",
            ascending=False
        )
    )

    display(
        failure_summary
    )


print("=" * 100)
print("EVALUATION STATUS SUMMARY : COMPLETE")
print("=" * 100)

In [ ]:
# ============================================================
# NOTEBOOK 06.23 — CANDIDATE PERFORMANCE ANALYSIS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.23")
print("CANDIDATE PERFORMANCE ANALYSIS")
print("=" * 100)


SUCCESS_RESULTS_DF = (
    EVALUATION_RESULTS_DF[
        EVALUATION_RESULTS_DF[
            "status"
        ] == "SUCCESS"
    ]
    .copy()
)


if SUCCESS_RESULTS_DF.empty:
    raise RuntimeError(
        "No successful evaluation results available."
    )


numeric_metrics = [
    "mae",
    "rmse",
    "r2",
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "wasserstein",
    "js_divergence",
    "runtime_seconds"
]


available_metrics = [
    metric
    for metric in numeric_metrics
    if metric in SUCCESS_RESULTS_DF.columns
]


CANDIDATE_PERFORMANCE_DF = (
    SUCCESS_RESULTS_DF
    .groupby(
        [
            "strategy_id",
            "feature_type"
        ],
        dropna=False
    )[available_metrics]
    .agg(
        ["mean", "std", "count"]
    )
    .reset_index()
)


display(
    CANDIDATE_PERFORMANCE_DF
)


print("\nPerformance by strategy:")


strategy_summary = (
    SUCCESS_RESULTS_DF
    .groupby(
        "strategy_id"
    )[available_metrics]
    .mean()
    .reset_index()
)


display(
    strategy_summary
)


ANALYSIS_PATH = (
    RESULTS_DIR /
    "candidate_performance_summary.csv"
)


strategy_summary.to_csv(
    ANALYSIS_PATH,
    index=False
)


print(
    f"\nPerformance summary saved to:\n"
    f"{ANALYSIS_PATH}"
)

print("=" * 100)

In [ ]:
# ============================================================
# NOTEBOOK 06.24 — LLM RECOMMENDATION EVALUATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.24")
print("LLM RECOMMENDATION EVALUATION")
print("=" * 100)


if "llm_recommended" not in EVALUATION_PLAN_DF.columns:
    raise RuntimeError(
        "LLM recommendation column not found."
    )


LLM_EVALUATION_DF = (
    EVALUATION_RESULTS_DF
    .merge(
        EVALUATION_PLAN_DF[
            [
                "dataset_id",
                "feature",
                "missingness_mechanism",
                "missingness_rate",
                "seed",
                "llm_recommended"
            ]
        ],
        on=[
            "dataset_id",
            "feature",
            "missingness_mechanism",
            "missingness_rate",
            "seed"
        ],
        how="left"
    )
)


LLM_EVALUATION_DF[
    "llm_match"
] = (
    LLM_EVALUATION_DF[
        "strategy_id"
    ]
    ==
    LLM_EVALUATION_DF[
        "llm_recommended"
    ]
)


LLM_MATCH_RATE = float(
    LLM_EVALUATION_DF[
        "llm_match"
    ].mean()
)


print(
    f"\nLLM recommendation agreement rate: "
    f"{LLM_MATCH_RATE:.4f}"
)


LLM_PERFORMANCE_DF = (
    LLM_EVALUATION_DF
    .groupby(
        "llm_match"
    )[
        [
            "mae",
            "rmse",
            "r2",
            "accuracy",
            "macro_f1",
            "weighted_f1",
            "wasserstein",
            "js_divergence",
            "runtime_seconds"
        ]
    ]
    .mean()
)


display(
    LLM_PERFORMANCE_DF
)


LLM_RESULTS_PATH = (
    RESULTS_DIR /
    "llm_recommendation_evaluation.csv"
)


LLM_EVALUATION_DF.to_csv(
    LLM_RESULTS_PATH,
    index=False
)


print(
    f"\nLLM evaluation saved to:\n"
    f"{LLM_RESULTS_PATH}"
)

print("=" * 100)

In [ ]:
# ============================================================
# NOTEBOOK 06.25 — STATISTICAL SIGNIFICANCE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.25")
print("STATISTICAL SIGNIFICANCE ANALYSIS")
print("=" * 100)


from scipy.stats import (
    wilcoxon,
    friedmanchisquare
)


SIGNIFICANCE_ALPHA = 0.05


# ------------------------------------------------------------
# Strategy-level metric table
# ------------------------------------------------------------

sig_rows = []


metric_direction = {
    "mae": "lower",
    "rmse": "lower",
    "r2": "higher",
    "accuracy": "higher",
    "macro_f1": "higher",
    "weighted_f1": "higher",
    "wasserstein": "lower",
    "js_divergence": "lower"
}


for metric, direction in metric_direction.items():

    if metric not in SUCCESS_RESULTS_DF.columns:
        continue

    metric_df = SUCCESS_RESULTS_DF[
        [
            "strategy_id",
            "dataset_id",
            "feature",
            "missingness_mechanism",
            "missingness_rate",
            "seed",
            metric
        ]
    ].dropna(
        subset=[metric]
    )

    if metric_df.empty:
        continue

    strategies = sorted(
        metric_df[
            "strategy_id"
        ].unique()
    )

    if len(strategies) < 2:
        continue

    pivot = metric_df.pivot_table(
        index=[
            "dataset_id",
            "feature",
            "missingness_mechanism",
            "missingness_rate",
            "seed"
        ],
        columns="strategy_id",
        values=metric,
        aggfunc="mean"
    ).dropna(
        axis=0,
        how="any"
    )

    if pivot.shape[0] < 2:
        continue

    arrays = [
        pivot[strategy].to_numpy()
        for strategy in strategies
    ]

    try:

        statistic, p_value = (
            friedmanchisquare(
                *arrays
            )
        )

    except Exception:

        statistic = np.nan
        p_value = np.nan


    sig_rows.append({

        "metric": metric,
        "direction": direction,
        "n_strategies": len(strategies),
        "n_matched_cases": len(pivot),
        "friedman_statistic": statistic,
        "p_value": p_value,
        "significant_at_0.05": (
            bool(
                p_value < SIGNIFICANCE_ALPHA
            )
            if np.isfinite(p_value)
            else False
        )

    })


STATISTICAL_SIGNIFICANCE_DF = pd.DataFrame(
    sig_rows
)


display(
    STATISTICAL_SIGNIFICANCE_DF
)


SIG_PATH = (
    RESULTS_DIR /
    "statistical_significance.csv"
)


STATISTICAL_SIGNIFICANCE_DF.to_csv(
    SIG_PATH,
    index=False
)


print(
    f"\nStatistical significance results saved to:\n"
    f"{SIG_PATH}"
)

print("=" * 100)

In [ ]:
# ============================================================
# NOTEBOOK 06.26 — ADAPTIVE UTILITY-BASED SELECTION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.26")
print("ADAPTIVE UTILITY-BASED SELECTION")
print("=" * 100)


# ------------------------------------------------------------
# Utility dimensions
# ------------------------------------------------------------

UTILITY_WEIGHTS = {
    "reconstruction": 0.30,
    "distribution": 0.20,
    "dependency": 0.15,
    "downstream": 0.20,
    "efficiency": 0.15
}


def minmax_normalize(
    series,
    higher_is_better=True
):

    series = pd.to_numeric(
        series,
        errors="coerce"
    )

    minimum = series.min()
    maximum = series.max()

    if not np.isfinite(minimum) or not np.isfinite(maximum):
        return pd.Series(
            0.5,
            index=series.index
        )

    if maximum == minimum:
        return pd.Series(
            1.0,
            index=series.index
        )

    if higher_is_better:

        return (
            series - minimum
        ) / (
            maximum - minimum
        )

    return (
        maximum - series
    ) / (
        maximum - minimum
    )


UTILITY_SOURCE_DF = (
    SUCCESS_RESULTS_DF.copy()
)


# ------------------------------------------------------------
# Reconstruction utility
# ------------------------------------------------------------

utility_components = []


if "mae" in UTILITY_SOURCE_DF.columns:

    utility_components.append(
        minmax_normalize(
            UTILITY_SOURCE_DF["mae"],
            higher_is_better=False
        )
    )

if "rmse" in UTILITY_SOURCE_DF.columns:

    utility_components.append(
        minmax_normalize(
            UTILITY_SOURCE_DF["rmse"],
            higher_is_better=False
        )
    )


if utility_components:

    UTILITY_SOURCE_DF[
        "reconstruction_utility"
    ] = pd.concat(
        utility_components,
        axis=1
    ).mean(
        axis=1
    )

else:

    UTILITY_SOURCE_DF[
        "reconstruction_utility"
    ] = 0.0


# ------------------------------------------------------------
# Distribution utility
# ------------------------------------------------------------

distribution_components = []


if "wasserstein" in UTILITY_SOURCE_DF.columns:

    distribution_components.append(
        minmax_normalize(
            UTILITY_SOURCE_DF[
                "wasserstein"
            ],
            higher_is_better=False
        )
    )


if "js_divergence" in UTILITY_SOURCE_DF.columns:

    distribution_components.append(
        minmax_normalize(
            UTILITY_SOURCE_DF[
                "js_divergence"
            ],
            higher_is_better=False
        )
    )


if distribution_components:

    UTILITY_SOURCE_DF[
        "distribution_utility"
    ] = pd.concat(
        distribution_components,
        axis=1
    ).mean(
        axis=1
    )

else:

    UTILITY_SOURCE_DF[
        "distribution_utility"
    ] = 0.0


# ------------------------------------------------------------
# Downstream/reconstruction categorical utility
# ------------------------------------------------------------

downstream_components = []


for metric in [
    "r2",
    "accuracy",
    "macro_f1",
    "weighted_f1"
]:

    if metric in UTILITY_SOURCE_DF.columns:

        values = UTILITY_SOURCE_DF[
            metric
        ]

        if values.notna().any():

            downstream_components.append(
                minmax_normalize(
                    values,
                    higher_is_better=True
                )
            )


if downstream_components:

    UTILITY_SOURCE_DF[
        "downstream_utility"
    ] = pd.concat(
        downstream_components,
        axis=1
    ).mean(
        axis=1
    )

else:

    UTILITY_SOURCE_DF[
        "downstream_utility"
    ] = 0.0


# ------------------------------------------------------------
# Dependency utility
# ------------------------------------------------------------

if "dependency_error" in UTILITY_SOURCE_DF.columns:

    UTILITY_SOURCE_DF[
        "dependency_utility"
    ] = minmax_normalize(
        UTILITY_SOURCE_DF[
            "dependency_error"
        ],
        higher_is_better=False
    )

else:

    UTILITY_SOURCE_DF[
        "dependency_utility"
    ] = 0.5


# ------------------------------------------------------------
# Computational efficiency
# ------------------------------------------------------------

UTILITY_SOURCE_DF[
    "efficiency_utility"
] = minmax_normalize(
    UTILITY_SOURCE_DF[
        "runtime_seconds"
    ],
    higher_is_better=False
)


# ------------------------------------------------------------
# Composite adaptive utility
# ------------------------------------------------------------

UTILITY_SOURCE_DF[
    "adaptive_utility"
] = (

    UTILITY_WEIGHTS[
        "reconstruction"
    ]
    *
    UTILITY_SOURCE_DF[
        "reconstruction_utility"
    ]

    +

    UTILITY_WEIGHTS[
        "distribution"
    ]
    *
    UTILITY_SOURCE_DF[
        "distribution_utility"
    ]

    +

    UTILITY_WEIGHTS[
        "dependency"
    ]
    *
    UTILITY_SOURCE_DF[
        "dependency_utility"
    ]

    +

    UTILITY_WEIGHTS[
        "downstream"
    ]
    *
    UTILITY_SOURCE_DF[
        "downstream_utility"
    ]

    +

    UTILITY_WEIGHTS[
        "efficiency"
    ]
    *
    UTILITY_SOURCE_DF[
        "efficiency_utility"
    ]
)


# ------------------------------------------------------------
# Select best strategy per evaluation context
# ------------------------------------------------------------

SELECTION_KEYS = [
    "dataset_id",
    "feature",
    "missingness_mechanism",
    "missingness_rate",
    "seed"
]


ADAPTIVE_SELECTION_DF = (
    UTILITY_SOURCE_DF
    .sort_values(
        "adaptive_utility",
        ascending=False
    )
    .groupby(
        SELECTION_KEYS,
        as_index=False
    )
    .first()
)


ADAPTIVE_SELECTION_DF = (
    ADAPTIVE_SELECTION_DF[
        SELECTION_KEYS
        +
        [
            "strategy_id",
            "adaptive_utility",
            "reconstruction_utility",
            "distribution_utility",
            "dependency_utility",
            "downstream_utility",
            "efficiency_utility"
        ]
    ]
)


display(
    ADAPTIVE_SELECTION_DF.head(25)
)


ADAPTIVE_RESULTS_PATH = (
    RESULTS_DIR /
    "adaptive_utility_selection.csv"
)


ADAPTIVE_SELECTION_DF.to_csv(
    ADAPTIVE_RESULTS_PATH,
    index=False
)


print(
    f"\nAdaptive selections: "
    f"{len(ADAPTIVE_SELECTION_DF):,}"
)

print(
    f"Results saved to:\n"
    f"{ADAPTIVE_RESULTS_PATH}"
)

print("=" * 100)
print("ADAPTIVE UTILITY SELECTION : COMPLETE")
print("=" * 100)